<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/GES_Stage6C_Cell_6C_4A0_Final_Figures_Error_Review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==================================================================================================
# COLAB NOTEBOOK FILE NAME:
# GES_Stage6C_Cell_6C_4A0_Final_Figures_Error_Review.ipynb
#
# STAGE 6C STEP 4A — CELL 6C-4A0
# STANDALONE FINAL FIGURES AND FALSE-STABLE / FALSE-UNSTABLE ERROR REVIEW
#
# This cell is designed for a NEW Google Colab notebook:
#   1. It mounts Google Drive.
#   2. It freshly verifies the immutable Stage 6B evaluable cohort and SHA-256 sidecar.
#   3. It creates the prespecified temporal-validation figures.
#   4. It creates complete and priority-review false-stable / false-unstable error tables.
#   5. It writes versioned staging artifacts, SHA-256 sidecars, a manifest, and fresh readback QC.
#
# Scientific boundary:
#   - The frozen Stage 6B cohort, outcomes, scores, thresholds, row order, genes, and policies are
#     never modified.
#   - The inherited Stage 4C threshold is used unchanged:
#         predicted instability = full_ges_instability_risk_t0 > 0.50
#         equivalently, full_ges_p_stable_t0 < 0.50
#   - No threshold optimization, recalibration, model fitting, score fitting, or outcome revision occurs.
#   - Exact-rank enrichment and decile memberships use descending frozen risk with t0_row_order as
#     the deterministic tie-break. Existing score-tie sensitivity results remain separate.
#   - These are versioned staging outputs for the subsequent final Stage 6C package freeze.
# ==================================================================================================

# --------------------------------------------------------------------------------------------------
# 0. MOUNT GOOGLE DRIVE AND RECORD THE NOTEBOOK FILE NAME
# --------------------------------------------------------------------------------------------------

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

NOTEBOOK_FILENAME = "GES_Stage6C_Cell_6C_4A0_Final_Figures_Error_Review.ipynb"
print(f"Use this Colab notebook file name: {NOTEBOOK_FILENAME}")


# --------------------------------------------------------------------------------------------------
# 1. IMPORTS
# --------------------------------------------------------------------------------------------------

from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import math
import platform
import re
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq
import sklearn
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
    roc_auc_score,
)


# --------------------------------------------------------------------------------------------------
# 2. FROZEN INPUT AND VERSIONED OUTPUT PATHS
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.exists():
    raise FileNotFoundError("Google Drive is not mounted at /content/drive/MyDrive.")

PROJECT_DIR = DRIVE_ROOT / "GES_RAG_Temporal_Study"
STAGE6_DIR = PROJECT_DIR / "data_processed" / "stage6_temporal_validation"

EVALUABLE_PARQUET = (
    STAGE6_DIR / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)
EVALUABLE_SIDECAR = Path(str(EVALUABLE_PARQUET) + ".sha256")

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
)
EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151

OUTPUT_VERSION = "v1"
FIGURE_DIR = (
    PROJECT_DIR
    / "outputs"
    / "figures"
    / "stage6_temporal_validation"
    / f"stage6c_4a0_final_figures_error_review_{OUTPUT_VERSION}"
)
TABLE_DIR = (
    PROJECT_DIR
    / "outputs"
    / "tables"
    / "stage6_temporal_validation"
    / f"stage6c_4a0_final_figures_error_review_{OUTPUT_VERSION}"
)
QC_DIR = (
    PROJECT_DIR
    / "outputs"
    / "quality_checks"
    / "stage6_temporal_validation"
    / f"stage6c_4a0_final_figures_error_review_{OUTPUT_VERSION}"
)
CONFIG_DIR = (
    PROJECT_DIR
    / "configs"
    / "stage6_temporal_validation"
    / f"stage6c_4a0_final_figures_error_review_{OUTPUT_VERSION}"
)

for directory in [FIGURE_DIR, TABLE_DIR, QC_DIR, CONFIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = CONFIG_DIR / "stage6c_4a0_figures_error_review_manifest_v1.json"
QC_PATH = QC_DIR / "stage6c_4a0_figures_error_review_qc_v1.json"

RCV_COLUMN = "rcv_accession"
ROW_ORDER_COLUMN = "t0_row_order"
GENE_COLUMN = "target_gene"
OUTCOME_COLUMN = "primary_future_instability"
FULL_RISK_COLUMN = "full_ges_instability_risk_t0"
FULL_P_STABLE_COLUMN = "full_ges_p_stable_t0"

FROZEN_THRESHOLD = 0.50
N_ERROR_REVIEW_PER_TYPE = 50
RISK_FRACTIONS = [0.05, 0.10, 0.20]
N_DECILES = 10

SCORES = OrderedDict(
    [
        (
            "full_ges",
            {
                "column": "full_ges_instability_risk_t0",
                "display": "Full GES",
                "calibration_type": "continuous",
            },
        ),
        (
            "no_star_ges",
            {
                "column": "no_star_ges_instability_risk_t0",
                "display": "No-star GES",
                "calibration_type": "continuous",
            },
        ),
        (
            "review_stars",
            {
                "column": "review_stars_instability_risk",
                "display": "Review stars",
                "calibration_type": "exact",
            },
        ),
        (
            "combined_metadata",
            {
                "column": "combined_metadata_instability_risk",
                "display": "Combined metadata",
                "calibration_type": "continuous",
            },
        ),
        (
            "conflict",
            {
                "column": "conflict_instability_risk",
                "display": "Conflict",
                "calibration_type": "exact",
            },
        ),
        (
            "recency",
            {
                "column": "recency_instability_risk",
                "display": "Recency",
                "calibration_type": "continuous",
            },
        ),
        (
            "submitter",
            {
                "column": "submitter_instability_risk",
                "display": "Submitter support",
                "calibration_type": "continuous",
            },
        ),
        (
            "entropy",
            {
                "column": "entropy_instability_risk",
                "display": "Classification entropy",
                "calibration_type": "continuous",
            },
        ),
        (
            "additive",
            {
                "column": "additive_instability_risk",
                "display": "Additive risk",
                "calibration_type": "exact",
            },
        ),
    ]
)

PRINCIPAL_MODELS = [
    "full_ges",
    "no_star_ges",
    "review_stars",
    "combined_metadata",
]

EVENT_COMPONENT_COLUMNS = [
    "event_material_clinical_group_change",
    "event_new_unresolved_conflict_at_t1",
    "event_prior_conflict_resolved_to_material_group",
]


# --------------------------------------------------------------------------------------------------
# 3. GENERAL HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding="utf-8").strip()
    matches = re.findall(r"\b[a-fA-F0-9]{64}\b", text)
    if not matches:
        raise ValueError(f"No SHA-256 value found in sidecar: {path}")
    return matches[0].lower()


def write_sha256_sidecar(path: Path) -> Path:
    digest = sha256_file(path)
    sidecar = Path(str(path) + ".sha256")
    sidecar.write_text(f"{digest}  {path.name}\n", encoding="utf-8")
    return sidecar


def save_csv(frame: pd.DataFrame, path: Path) -> Path:
    frame.to_csv(path, index=False, lineterminator="\n", float_format="%.12g")
    return path


def save_parquet(frame: pd.DataFrame, path: Path) -> Path:
    frame.to_parquet(
        path,
        index=False,
        engine="pyarrow",
        compression="snappy",
    )
    return path


def save_png(fig, path: Path) -> Path:
    fig.savefig(
        path,
        dpi=300,
        bbox_inches="tight",
        metadata={
            "Software": "Matplotlib",
            "Title": path.stem,
        },
    )
    plt.close(fig)
    return path


def ordered_unique(values):
    output = []
    seen = set()
    for value in values:
        if value not in seen:
            output.append(value)
            seen.add(value)
    return output


def validate_binary(series: pd.Series, label: str) -> pd.Series:
    numeric = pd.to_numeric(series, errors="raise")
    unique = set(numeric.dropna().unique().tolist())
    if not unique.issubset({0, 1, 0.0, 1.0, False, True}):
        raise AssertionError(f"{label} is not binary: {sorted(unique)}")
    if numeric.isna().any():
        raise AssertionError(f"{label} contains missing values.")
    return numeric.astype(np.int8)


def make_reliability_table(
    score: np.ndarray,
    outcome: np.ndarray,
    model_key: str,
    model_display: str,
    grouping: str,
    n_bins: int = 10,
) -> pd.DataFrame:
    score = np.asarray(score, dtype=float)
    outcome = np.asarray(outcome, dtype=np.int8)

    if grouping == "exact":
        unique_scores = np.sort(np.unique(score))
        mapping = {value: index + 1 for index, value in enumerate(unique_scores)}
        group_id = pd.Series(score).map(mapping).astype(int)
        grouping_method = "exact_score_value_groups"
    else:
        # Tie-preserving equal-frequency quantile cutpoints, matching the locked calibration design.
        group_id = pd.qcut(
            pd.Series(score),
            q=n_bins,
            labels=False,
            duplicates="drop",
        )
        if group_id.isna().any():
            raise AssertionError(f"Reliability grouping failed for {model_key}.")
        group_id = group_id.astype(int) + 1
        grouping_method = "equal_frequency_quantile_bins"

    temp = pd.DataFrame(
        {
            "group_id": group_id.to_numpy(),
            "score": score,
            "outcome": outcome,
        }
    )

    rows = []
    for group_value, group in temp.groupby("group_id", sort=True, observed=True):
        rows.append(
            {
                "model_key": model_key,
                "model": model_display,
                "grouping_method": grouping_method,
                "group_id": int(group_value),
                "records": int(len(group)),
                "events": int(group["outcome"].sum()),
                "score_min": float(group["score"].min()),
                "score_max": float(group["score"].max()),
                "mean_predicted_risk": float(group["score"].mean()),
                "observed_event_rate": float(group["outcome"].mean()),
                "calibration_gap_observed_minus_predicted": float(
                    group["outcome"].mean() - group["score"].mean()
                ),
            }
        )

    result = pd.DataFrame(rows)
    if int(result["records"].sum()) != len(score):
        raise AssertionError(f"Reliability rows do not reconcile for {model_key}.")
    return result


def exact_rank_order_descending_risk(
    risk: np.ndarray,
    row_order: np.ndarray,
) -> np.ndarray:
    # Primary key: descending frozen risk. Deterministic tie-break: ascending frozen row order.
    return np.lexsort((row_order, -risk))


def create_error_table(
    source: pd.DataFrame,
    mask: np.ndarray,
    error_type: str,
    sort_ascending_risk: bool,
) -> pd.DataFrame:
    table = source.loc[mask].copy()
    table.insert(0, "error_type", error_type)
    table.insert(
        1,
        "frozen_threshold_rule",
        "predicted_instability_if_full_ges_risk_gt_0.50",
    )
    table.insert(
        2,
        "full_ges_predicted_instability",
        table[FULL_RISK_COLUMN].gt(FROZEN_THRESHOLD),
    )
    table.insert(
        3,
        "full_ges_risk_margin_from_0_50",
        table[FULL_RISK_COLUMN] - FROZEN_THRESHOLD,
    )
    table.insert(
        4,
        "full_ges_p_stable_rederived",
        1.0 - table[FULL_RISK_COLUMN],
    )

    table = table.sort_values(
        [FULL_RISK_COLUMN, ROW_ORDER_COLUMN],
        ascending=[sort_ascending_risk, True],
        kind="mergesort",
    ).reset_index(drop=True)

    table.insert(0, "error_review_rank", np.arange(1, len(table) + 1, dtype=np.int64))
    return table


# --------------------------------------------------------------------------------------------------
# 4. CRYPTOGRAPHIC AND STRUCTURAL PREFLIGHT
# --------------------------------------------------------------------------------------------------

for required_path in [EVALUABLE_PARQUET, EVALUABLE_SIDECAR]:
    if not required_path.exists():
        raise FileNotFoundError(f"Missing frozen Stage 6B artifact:\n{required_path}")

observed_evaluable_sha256 = sha256_file(EVALUABLE_PARQUET)
if observed_evaluable_sha256 != EXPECTED_EVALUABLE_SHA256:
    raise AssertionError(
        "Stage 6B evaluable SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_EVALUABLE_SHA256}\n"
        f"Observed: {observed_evaluable_sha256}"
    )

if read_sidecar_hash(EVALUABLE_SIDECAR) != observed_evaluable_sha256:
    raise AssertionError("Stage 6B evaluable SHA-256 sidecar mismatch.")

parquet_file = pq.ParquetFile(EVALUABLE_PARQUET)
if (
    parquet_file.metadata.num_rows,
    parquet_file.metadata.num_columns,
) != (
    EXPECTED_ROWS,
    EXPECTED_COLUMNS,
):
    raise AssertionError(
        "Unexpected Stage 6B dimensions: "
        f"{parquet_file.metadata.num_rows:,} × {parquet_file.metadata.num_columns}"
    )

required_columns = [
    RCV_COLUMN,
    ROW_ORDER_COLUMN,
    GENE_COLUMN,
    OUTCOME_COLUMN,
    FULL_P_STABLE_COLUMN,
] + [spec["column"] for spec in SCORES.values()]

missing_required = [
    column for column in required_columns if column not in parquet_file.schema_arrow.names
]
if missing_required:
    raise KeyError(
        "Missing required Stage 6B columns:\n" + "\n".join(missing_required)
    )

# Load all 79 frozen columns so the error tables preserve the complete available record context.
cohort = pd.read_parquet(EVALUABLE_PARQUET).copy()

if len(cohort) != EXPECTED_ROWS or len(cohort.columns) != EXPECTED_COLUMNS:
    raise AssertionError("Loaded Stage 6B dataframe dimensions are incorrect.")

cohort[RCV_COLUMN] = cohort[RCV_COLUMN].astype(str).str.strip().str.upper()
cohort[ROW_ORDER_COLUMN] = pd.to_numeric(
    cohort[ROW_ORDER_COLUMN], errors="raise"
).astype(np.int64)
cohort[OUTCOME_COLUMN] = validate_binary(
    cohort[OUTCOME_COLUMN],
    OUTCOME_COLUMN,
)

if cohort[RCV_COLUMN].nunique(dropna=False) != EXPECTED_ROWS:
    raise AssertionError("Stage 6B RCV keys are not unique.")

if cohort[ROW_ORDER_COLUMN].nunique(dropna=False) != EXPECTED_ROWS:
    raise AssertionError("Stage 6B t0_row_order values are not unique.")

if not np.all(np.diff(cohort[ROW_ORDER_COLUMN].to_numpy()) > 0):
    raise AssertionError("Stage 6B rows are not in strictly increasing frozen row order.")

observed_events = int(cohort[OUTCOME_COLUMN].sum())
observed_negatives = int((cohort[OUTCOME_COLUMN] == 0).sum())
if (observed_events, observed_negatives) != (
    EXPECTED_EVENTS,
    EXPECTED_NEGATIVES,
):
    raise AssertionError(
        "Stage 6B outcome accounting mismatch: "
        f"{observed_events:,} events / {observed_negatives:,} negatives."
    )

for model_key, specification in SCORES.items():
    column = specification["column"]
    cohort[column] = pd.to_numeric(cohort[column], errors="raise").astype(float)
    values = cohort[column].to_numpy()
    if not np.isfinite(values).all():
        raise AssertionError(f"{column} contains nonfinite values.")
    if ((values < 0.0) | (values > 1.0)).any():
        raise AssertionError(f"{column} contains values outside [0,1].")
    if len(np.unique(values)) < 2:
        raise AssertionError(f"{column} lacks score variation.")

full_risk = cohort[FULL_RISK_COLUMN].to_numpy(dtype=float)
full_p_stable = cohort[FULL_P_STABLE_COLUMN].to_numpy(dtype=float)
if not np.allclose(full_p_stable, 1.0 - full_risk, rtol=0.0, atol=1e-12):
    raise AssertionError("Frozen full GES P(stable) and instability risk are inconsistent.")

outcome = cohort[OUTCOME_COLUMN].to_numpy(dtype=np.int8)
row_order = cohort[ROW_ORDER_COLUMN].to_numpy(dtype=np.int64)
prevalence = float(outcome.mean())


# --------------------------------------------------------------------------------------------------
# 5. LOCKED MODEL POINT-ESTIMATE TABLE
# --------------------------------------------------------------------------------------------------

model_summary_rows = []
for model_key, specification in SCORES.items():
    score = cohort[specification["column"]].to_numpy(dtype=float)
    model_summary_rows.append(
        {
            "model_key": model_key,
            "model": specification["display"],
            "score_column": specification["column"],
            "rows": int(len(cohort)),
            "events": observed_events,
            "negatives": observed_negatives,
            "prevalence": prevalence,
            "unique_score_values": int(len(np.unique(score))),
            "mean_score": float(score.mean()),
            "point_auprc": float(average_precision_score(outcome, score)),
            "auprc_minus_prevalence": float(
                average_precision_score(outcome, score) - prevalence
            ),
            "auprc_lift_over_prevalence": float(
                average_precision_score(outcome, score) / prevalence
            ),
            "point_auroc": float(roc_auc_score(outcome, score)),
            "point_brier": float(brier_score_loss(outcome, score)),
        }
    )

model_summary = pd.DataFrame(model_summary_rows)


# --------------------------------------------------------------------------------------------------
# 6. FIGURE 1 — PRECISION-RECALL CURVES
# --------------------------------------------------------------------------------------------------

fig = plt.figure(figsize=(8.5, 6.5))
ax = fig.add_subplot(111)

for model_key in PRINCIPAL_MODELS:
    specification = SCORES[model_key]
    score = cohort[specification["column"]].to_numpy(dtype=float)
    precision, recall, _ = precision_recall_curve(outcome, score)
    auprc = average_precision_score(outcome, score)
    ax.plot(
        recall,
        precision,
        linewidth=2,
        label=f"{specification['display']} (AUPRC={auprc:.4f})",
    )

ax.axhline(
    prevalence,
    linestyle="--",
    linewidth=1.5,
    label=f"Prevalence ({prevalence:.4f})",
)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Locked T0→T1 Precision–Recall Curves")
ax.set_xlim(0.0, 1.0)
ax.set_ylim(0.0, 1.0)
ax.legend(loc="best", frameon=False)
ax.grid(True, alpha=0.25)

FIGURE_1 = FIGURE_DIR / "stage6c_figure1_precision_recall_principal_models_v1.png"
save_png(fig, FIGURE_1)


# --------------------------------------------------------------------------------------------------
# 7. FIGURE 2 — CALIBRATION / RELIABILITY
# --------------------------------------------------------------------------------------------------

reliability_tables = []
for model_key in PRINCIPAL_MODELS:
    specification = SCORES[model_key]
    reliability_tables.append(
        make_reliability_table(
            score=cohort[specification["column"]].to_numpy(dtype=float),
            outcome=outcome,
            model_key=model_key,
            model_display=specification["display"],
            grouping=specification["calibration_type"],
            n_bins=10,
        )
    )

reliability_table = pd.concat(reliability_tables, ignore_index=True)

fig = plt.figure(figsize=(8.5, 6.5))
ax = fig.add_subplot(111)
ax.plot([0.0, 1.0], [0.0, 1.0], linestyle="--", linewidth=1.5, label="Ideal")

for model_key in PRINCIPAL_MODELS:
    subset = reliability_table.loc[
        reliability_table["model_key"].eq(model_key)
    ].sort_values("mean_predicted_risk")
    ax.plot(
        subset["mean_predicted_risk"],
        subset["observed_event_rate"],
        marker="o",
        linewidth=1.8,
        label=SCORES[model_key]["display"],
    )

ax.set_xlabel("Mean predicted instability risk")
ax.set_ylabel("Observed future-instability rate")
ax.set_title("Locked Reliability Plot")
ax.set_xlim(0.0, 1.0)
ax.set_ylim(0.0, 1.0)
ax.legend(loc="best", frameon=False)
ax.grid(True, alpha=0.25)

FIGURE_2 = FIGURE_DIR / "stage6c_figure2_calibration_reliability_v1.png"
save_png(fig, FIGURE_2)


# --------------------------------------------------------------------------------------------------
# 8. FIGURE 3 — EXACT-RANK TOP-RISK ENRICHMENT
# --------------------------------------------------------------------------------------------------

descending_order = exact_rank_order_descending_risk(full_risk, row_order)
enrichment_rows = []

for fraction in RISK_FRACTIONS:
    selected_count = int(math.ceil(len(cohort) * fraction))
    selected_positions = descending_order[:selected_count]

    selected_mask = np.zeros(len(cohort), dtype=bool)
    selected_mask[selected_positions] = True
    remainder_mask = ~selected_mask

    selected_events = int(outcome[selected_mask].sum())
    remainder_events = int(outcome[remainder_mask].sum())
    selected_rate = float(outcome[selected_mask].mean())
    remainder_rate = float(outcome[remainder_mask].mean())

    cutoff_score = float(full_risk[descending_order[selected_count - 1]])
    boundary_tie_total = int(np.count_nonzero(full_risk == cutoff_score))
    boundary_tie_selected = int(
        np.count_nonzero(full_risk[selected_mask] == cutoff_score)
    )

    enrichment_rows.append(
        {
            "risk_fraction": fraction,
            "risk_fraction_label": f"Top {int(round(fraction * 100))}%",
            "membership_policy": (
                "descending_frozen_full_ges_risk_then_ascending_t0_row_order"
            ),
            "selected_records": selected_count,
            "selected_events": selected_events,
            "selected_event_rate": selected_rate,
            "remainder_records": int(remainder_mask.sum()),
            "remainder_events": remainder_events,
            "remainder_event_rate": remainder_rate,
            "risk_ratio_vs_remainder": (
                selected_rate / remainder_rate if remainder_rate > 0 else np.nan
            ),
            "enrichment_over_global_prevalence": (
                selected_rate / prevalence if prevalence > 0 else np.nan
            ),
            "event_capture_fraction": selected_events / observed_events,
            "cutoff_full_ges_risk": cutoff_score,
            "boundary_tie_total_records": boundary_tie_total,
            "boundary_tie_selected_records": boundary_tie_selected,
            "cutoff_tie_sensitivity_reference": (
                "Existing Stage 6C score-tie sensitivity remains authoritative."
            ),
        }
    )

enrichment_table = pd.DataFrame(enrichment_rows)

x = np.arange(len(enrichment_table))
width = 0.36

fig = plt.figure(figsize=(8.5, 6.5))
ax = fig.add_subplot(111)
ax.bar(
    x - width / 2,
    enrichment_table["selected_event_rate"],
    width,
    label="Selected top-risk subset",
)
ax.bar(
    x + width / 2,
    enrichment_table["remainder_event_rate"],
    width,
    label="Remaining records",
)
ax.axhline(
    prevalence,
    linestyle="--",
    linewidth=1.5,
    label=f"Global prevalence ({prevalence:.4f})",
)
ax.set_xticks(x)
ax.set_xticklabels(enrichment_table["risk_fraction_label"])
ax.set_xlabel("Exact-rank Full GES risk subset")
ax.set_ylabel("Observed future-instability rate")
ax.set_title("Future Instability in Prespecified Top-Risk Fractions")
ax.legend(loc="best", frameon=False)
ax.grid(True, axis="y", alpha=0.25)

FIGURE_3 = FIGURE_DIR / "stage6c_figure3_top_risk_enrichment_v1.png"
save_png(fig, FIGURE_3)


# --------------------------------------------------------------------------------------------------
# 9. FIGURE 4 — FUTURE-INSTABILITY RATE BY EXACT-RANK P(STABLE) DECILE
# --------------------------------------------------------------------------------------------------

decile_rows = []
decile_position_groups = np.array_split(descending_order, N_DECILES)

for decile_index, positions in enumerate(decile_position_groups, start=1):
    decile_risk = full_risk[positions]
    decile_outcome = outcome[positions]

    decile_rows.append(
        {
            "risk_decile": decile_index,
            "risk_decile_label": f"D{decile_index}",
            "interpretation": (
                "D1=lowest P(stable)/highest frozen instability risk; "
                "D10=highest P(stable)/lowest risk"
            ),
            "membership_policy": (
                "descending_frozen_full_ges_risk_then_ascending_t0_row_order"
            ),
            "records": int(len(positions)),
            "events": int(decile_outcome.sum()),
            "event_rate": float(decile_outcome.mean()),
            "full_ges_risk_min": float(decile_risk.min()),
            "full_ges_risk_max": float(decile_risk.max()),
            "mean_full_ges_risk": float(decile_risk.mean()),
            "mean_p_stable": float((1.0 - decile_risk).mean()),
        }
    )

decile_table = pd.DataFrame(decile_rows)

fig = plt.figure(figsize=(9.0, 6.5))
ax = fig.add_subplot(111)
ax.plot(
    decile_table["risk_decile"],
    decile_table["event_rate"],
    marker="o",
    linewidth=2,
)
ax.axhline(
    prevalence,
    linestyle="--",
    linewidth=1.5,
    label=f"Global prevalence ({prevalence:.4f})",
)
ax.set_xticks(decile_table["risk_decile"])
ax.set_xticklabels(decile_table["risk_decile_label"])
ax.set_xlabel("Full GES exact-rank risk decile")
ax.set_ylabel("Observed future-instability rate")
ax.set_title("Future Instability by Frozen Full GES Risk Decile")
ax.legend(loc="best", frameon=False)
ax.grid(True, alpha=0.25)

FIGURE_4 = FIGURE_DIR / "stage6c_figure4_future_instability_by_risk_decile_v1.png"
save_png(fig, FIGURE_4)


# --------------------------------------------------------------------------------------------------
# 10. FIGURE 5 — FULL GES RISK DISTRIBUTION BY OBSERVED OUTCOME
# --------------------------------------------------------------------------------------------------

fig = plt.figure(figsize=(8.5, 6.5))
ax = fig.add_subplot(111)

for outcome_value, label in [
    (0, "Observed stable/no future-instability event"),
    (1, "Observed future-instability event"),
]:
    values = np.sort(full_risk[outcome == outcome_value])
    cumulative = np.arange(1, len(values) + 1, dtype=float) / len(values)
    ax.step(values, cumulative, where="post", linewidth=2, label=label)

ax.set_xlabel("Frozen Full GES instability risk")
ax.set_ylabel("Empirical cumulative fraction")
ax.set_title("Full GES Risk Distribution by T1 Outcome")
ax.set_xlim(0.0, 1.0)
ax.set_ylim(0.0, 1.0)
ax.legend(loc="best", frameon=False)
ax.grid(True, alpha=0.25)

FIGURE_5 = FIGURE_DIR / "stage6c_figure5_full_ges_risk_distribution_ecdf_v1.png"
save_png(fig, FIGURE_5)


# --------------------------------------------------------------------------------------------------
# 11. FALSE-STABLE / FALSE-UNSTABLE ERROR REVIEW
# --------------------------------------------------------------------------------------------------

predicted_instability = full_risk > FROZEN_THRESHOLD

true_positive_mask = (outcome == 1) & predicted_instability
false_stable_mask = (outcome == 1) & (~predicted_instability)
false_unstable_mask = (outcome == 0) & predicted_instability
true_negative_mask = (outcome == 0) & (~predicted_instability)

tp = int(true_positive_mask.sum())
fn = int(false_stable_mask.sum())
fp = int(false_unstable_mask.sum())
tn = int(true_negative_mask.sum())

if tp + fn != observed_events:
    raise AssertionError("Event-side confusion accounting failed.")
if fp + tn != observed_negatives:
    raise AssertionError("Negative-side confusion accounting failed.")

threshold_summary = pd.DataFrame(
    [
        {
            "model": "Full GES",
            "threshold_rule": (
                "predicted_instability_if_full_ges_instability_risk_t0_gt_0.50"
            ),
            "threshold": FROZEN_THRESHOLD,
            "rows": len(cohort),
            "events": observed_events,
            "negatives": observed_negatives,
            "predicted_instability": int(predicted_instability.sum()),
            "predicted_stable": int((~predicted_instability).sum()),
            "true_positive": tp,
            "false_stable_false_negative": fn,
            "false_unstable_false_positive": fp,
            "true_negative": tn,
            "sensitivity": tp / observed_events,
            "specificity": tn / observed_negatives,
            "positive_predictive_value": tp / (tp + fp) if (tp + fp) else np.nan,
            "negative_predictive_value": tn / (tn + fn) if (tn + fn) else np.nan,
            "balanced_accuracy": 0.5
            * ((tp / observed_events) + (tn / observed_negatives)),
            "threshold_optimized_on_t1": False,
            "recalibration_performed": False,
        }
    ]
)

false_stable_all = create_error_table(
    source=cohort,
    mask=false_stable_mask,
    error_type="FALSE_STABLE_OBSERVED_EVENT_PREDICTED_STABLE",
    sort_ascending_risk=True,
)

false_unstable_all = create_error_table(
    source=cohort,
    mask=false_unstable_mask,
    error_type="FALSE_UNSTABLE_OBSERVED_NO_EVENT_PREDICTED_UNSTABLE",
    sort_ascending_risk=False,
)

false_stable_priority = false_stable_all.head(N_ERROR_REVIEW_PER_TYPE).copy()
false_unstable_priority = false_unstable_all.head(N_ERROR_REVIEW_PER_TYPE).copy()

# Compact human-review columns, while complete 79-column context remains in the full Parquet files.
preferred_review_columns = [
    "error_review_rank",
    "error_type",
    "frozen_threshold_rule",
    RCV_COLUMN,
    "linked_t1_rcv_accession",
    ROW_ORDER_COLUMN,
    GENE_COLUMN,
    "variation_id",
    "vcv_accession",
    FULL_P_STABLE_COLUMN,
    FULL_RISK_COLUMN,
    "full_ges_p_stable_rederived",
    "full_ges_risk_margin_from_0_50",
    "no_star_ges_instability_risk_t0",
    "review_stars_instability_risk",
    "combined_metadata_instability_risk",
    "t0_aggregate_review_stars",
    "t1_aggregate_review_stars",
    "t0_aggregate_review_status",
    "t1_aggregate_review_status",
    "t0_aggregate_clinical_significance",
    "t1_aggregate_clinical_significance",
    "t0_canonical_clinical_group",
    "t1_canonical_clinical_group",
    "t0_aggregate_conflict_flag",
    "t1_aggregate_conflict_flag",
    OUTCOME_COLUMN,
    "primary_event_component_count",
    "primary_event_component_combination",
] + EVENT_COMPONENT_COLUMNS + [
    "classification_group_direction",
    "conflict_transition",
    "review_star_change",
    "review_status_change",
    "condition_name",
    "condition_identifiers_json",
]

available_review_columns = [
    column
    for column in ordered_unique(preferred_review_columns)
    if column in false_stable_priority.columns
]

# Always retain the required core columns even if optional field names differ.
required_review_core = [
    "error_review_rank",
    "error_type",
    RCV_COLUMN,
    ROW_ORDER_COLUMN,
    GENE_COLUMN,
    FULL_P_STABLE_COLUMN,
    FULL_RISK_COLUMN,
    OUTCOME_COLUMN,
]
for column in required_review_core:
    if column not in available_review_columns:
        available_review_columns.append(column)

false_stable_priority_compact = false_stable_priority[available_review_columns].copy()
false_unstable_priority_compact = false_unstable_priority[available_review_columns].copy()

# Error counts by gene.
gene_error_rows = []
for gene, group in cohort.groupby(GENE_COLUMN, dropna=False, sort=True):
    group_index = group.index.to_numpy()
    group_outcome = outcome[group_index]
    group_prediction = predicted_instability[group_index]

    group_tp = int(((group_outcome == 1) & group_prediction).sum())
    group_fn = int(((group_outcome == 1) & (~group_prediction)).sum())
    group_fp = int(((group_outcome == 0) & group_prediction).sum())
    group_tn = int(((group_outcome == 0) & (~group_prediction)).sum())

    gene_error_rows.append(
        {
            "target_gene": gene,
            "rows": int(len(group)),
            "events": int((group_outcome == 1).sum()),
            "negatives": int((group_outcome == 0).sum()),
            "true_positive": group_tp,
            "false_stable": group_fn,
            "false_unstable": group_fp,
            "true_negative": group_tn,
            "false_stable_fraction_of_events": (
                group_fn / int((group_outcome == 1).sum())
                if int((group_outcome == 1).sum()) > 0
                else np.nan
            ),
            "false_unstable_fraction_of_negatives": (
                group_fp / int((group_outcome == 0).sum())
                if int((group_outcome == 0).sum()) > 0
                else np.nan
            ),
        }
    )

error_by_gene = pd.DataFrame(gene_error_rows)

# Error counts by frozen review-star category, when available.
if "t0_aggregate_review_stars" in cohort.columns:
    review_star_error_rows = []
    for stars, group in cohort.groupby(
        "t0_aggregate_review_stars",
        dropna=False,
        sort=True,
    ):
        group_index = group.index.to_numpy()
        group_outcome = outcome[group_index]
        group_prediction = predicted_instability[group_index]

        review_star_error_rows.append(
            {
                "t0_aggregate_review_stars": stars,
                "rows": int(len(group)),
                "events": int((group_outcome == 1).sum()),
                "negatives": int((group_outcome == 0).sum()),
                "false_stable": int(
                    ((group_outcome == 1) & (~group_prediction)).sum()
                ),
                "false_unstable": int(
                    ((group_outcome == 0) & group_prediction).sum()
                ),
            }
        )
    error_by_review_stars = pd.DataFrame(review_star_error_rows)
else:
    error_by_review_stars = pd.DataFrame(
        [
            {
                "status": (
                    "t0_aggregate_review_stars was not present; "
                    "no review-star error summary was created."
                )
            }
        ]
    )

# Which primary event components dominate the false-stable records?
false_stable_component_rows = []
for component_column in EVENT_COMPONENT_COLUMNS:
    if component_column in cohort.columns:
        component = validate_binary(cohort[component_column], component_column).to_numpy()
        false_stable_component_rows.append(
            {
                "event_component_column": component_column,
                "component_events_in_full_cohort": int(component.sum()),
                "component_events_among_false_stable": int(
                    component[false_stable_mask].sum()
                ),
                "fraction_of_component_events_false_stable": (
                    float(component[false_stable_mask].sum() / component.sum())
                    if component.sum() > 0
                    else np.nan
                ),
                "fraction_of_false_stable_records_with_component": (
                    float(component[false_stable_mask].mean())
                    if false_stable_mask.sum() > 0
                    else np.nan
                ),
            }
        )

false_stable_component_summary = pd.DataFrame(false_stable_component_rows)


# --------------------------------------------------------------------------------------------------
# 12. FIGURE 6 — FALSE-STABLE / FALSE-UNSTABLE COUNTS BY GENE
# --------------------------------------------------------------------------------------------------

x = np.arange(len(error_by_gene))
width = 0.36

fig = plt.figure(figsize=(9.0, 6.5))
ax = fig.add_subplot(111)
ax.bar(
    x - width / 2,
    error_by_gene["false_stable"],
    width,
    label="False-stable",
)
ax.bar(
    x + width / 2,
    error_by_gene["false_unstable"],
    width,
    label="False-unstable",
)
ax.set_xticks(x)
ax.set_xticklabels(error_by_gene["target_gene"].astype(str))
ax.set_xlabel("Target gene")
ax.set_ylabel("Record count")
ax.set_title("Inherited 0.50-Threshold Errors by Gene")
ax.legend(loc="best", frameon=False)
ax.grid(True, axis="y", alpha=0.25)

FIGURE_6 = FIGURE_DIR / "stage6c_figure6_threshold_errors_by_gene_v1.png"
save_png(fig, FIGURE_6)


# --------------------------------------------------------------------------------------------------
# 13. WRITE VERSIONED TABLES
# --------------------------------------------------------------------------------------------------

output_files = []

table_specs = [
    (
        model_summary,
        TABLE_DIR / "stage6c_model_point_estimates_for_figures_v1.csv",
        "csv",
    ),
    (
        reliability_table,
        TABLE_DIR / "stage6c_reliability_table_for_figure_v1.csv",
        "csv",
    ),
    (
        enrichment_table,
        TABLE_DIR / "stage6c_exact_rank_top_risk_enrichment_for_figure_v1.csv",
        "csv",
    ),
    (
        decile_table,
        TABLE_DIR / "stage6c_exact_rank_risk_decile_event_rates_v1.csv",
        "csv",
    ),
    (
        threshold_summary,
        TABLE_DIR / "stage6c_full_ges_frozen_threshold_error_summary_v1.csv",
        "csv",
    ),
    (
        error_by_gene,
        TABLE_DIR / "stage6c_full_ges_errors_by_gene_v1.csv",
        "csv",
    ),
    (
        error_by_review_stars,
        TABLE_DIR / "stage6c_full_ges_errors_by_review_stars_v1.csv",
        "csv",
    ),
    (
        false_stable_component_summary,
        TABLE_DIR / "stage6c_false_stable_event_component_summary_v1.csv",
        "csv",
    ),
    (
        false_stable_priority_compact,
        TABLE_DIR / "stage6c_false_stable_priority_review_top50_v1.csv",
        "csv",
    ),
    (
        false_unstable_priority_compact,
        TABLE_DIR / "stage6c_false_unstable_priority_review_top50_v1.csv",
        "csv",
    ),
    (
        false_stable_all,
        TABLE_DIR / "stage6c_false_stable_complete_context_v1.parquet",
        "parquet",
    ),
    (
        false_unstable_all,
        TABLE_DIR / "stage6c_false_unstable_complete_context_v1.parquet",
        "parquet",
    ),
]

for frame, path, file_type in table_specs:
    if file_type == "csv":
        output_files.append(save_csv(frame, path))
    elif file_type == "parquet":
        output_files.append(save_parquet(frame, path))
    else:
        raise ValueError(f"Unsupported output type: {file_type}")

output_files.extend(
    [
        FIGURE_1,
        FIGURE_2,
        FIGURE_3,
        FIGURE_4,
        FIGURE_5,
        FIGURE_6,
    ]
)


# --------------------------------------------------------------------------------------------------
# 14. WRITE SHA-256 SIDECARS, MANIFEST, AND QC
# --------------------------------------------------------------------------------------------------

artifact_entries = []
for path in output_files:
    sidecar = write_sha256_sidecar(path)
    artifact_entries.append(
        {
            "path": str(path),
            "relative_to_project": str(path.relative_to(PROJECT_DIR)),
            "file_name": path.name,
            "bytes": int(path.stat().st_size),
            "sha256": sha256_file(path),
            "sidecar_path": str(sidecar),
            "sidecar_sha256": sha256_file(sidecar),
        }
    )

manifest = {
    "schema_version": "1.0",
    "cell": "6C-4A0",
    "stage": "Stage 6C Step 4A",
    "notebook_filename": NOTEBOOK_FILENAME,
    "output_version": OUTPUT_VERSION,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source": {
        "stage6b_evaluable_path": str(EVALUABLE_PARQUET),
        "stage6b_evaluable_sha256": observed_evaluable_sha256,
        "stage6b_evaluable_rows": EXPECTED_ROWS,
        "stage6b_evaluable_columns": EXPECTED_COLUMNS,
        "events": observed_events,
        "negatives": observed_negatives,
        "prevalence": prevalence,
    },
    "methods": {
        "score_direction": "higher_score_equals_greater_future_instability_risk",
        "principal_models": PRINCIPAL_MODELS,
        "reliability": (
            "equal-frequency quantile bins for continuous/high-cardinality scores; "
            "exact score-value groups for low-cardinality scores"
        ),
        "enrichment_membership": (
            "descending frozen full-GES risk with ascending frozen t0_row_order tie-break"
        ),
        "decile_membership": (
            "ten exact-rank groups from descending frozen full-GES risk with "
            "ascending frozen t0_row_order tie-break"
        ),
        "frozen_threshold": FROZEN_THRESHOLD,
        "threshold_rule": (
            "predicted instability when full_ges_instability_risk_t0 > 0.50"
        ),
        "false_stable_definition": (
            "observed primary_future_instability=1 and predicted instability=False"
        ),
        "false_unstable_definition": (
            "observed primary_future_instability=0 and predicted instability=True"
        ),
        "priority_review_records_per_error_type": N_ERROR_REVIEW_PER_TYPE,
        "threshold_optimized": False,
        "recalibration_performed": False,
        "model_fitted": False,
        "outcome_modified": False,
        "frozen_input_modified": False,
    },
    "error_accounting": {
        "true_positive": tp,
        "false_stable": fn,
        "false_unstable": fp,
        "true_negative": tn,
    },
    "artifacts": artifact_entries,
    "status": "STAGING_ARTIFACTS_FOR_FINAL_STAGE6C_FREEZE",
}

MANIFEST_PATH.write_text(
    json.dumps(manifest, indent=2, sort_keys=True, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
manifest_sidecar = write_sha256_sidecar(MANIFEST_PATH)

qc_checks = OrderedDict(
    [
        ("stage6b_file_hash_matches_expected", True),
        ("stage6b_sidecar_matches_file", True),
        ("stage6b_dimensions_match", True),
        ("stage6b_keys_unique", True),
        ("stage6b_row_order_unique_and_increasing", True),
        ("stage6b_outcome_accounting_matches", True),
        ("all_nine_scores_finite_and_within_0_1", True),
        ("full_p_stable_equals_one_minus_risk", True),
        ("model_summary_created", len(model_summary) == len(SCORES)),
        (
            "reliability_rows_reconcile",
            int(reliability_table["records"].sum())
            == EXPECTED_ROWS * len(PRINCIPAL_MODELS),
        ),
        ("enrichment_fractions_created", len(enrichment_table) == 3),
        ("ten_deciles_created", len(decile_table) == 10),
        (
            "decile_rows_reconcile",
            int(decile_table["records"].sum()) == EXPECTED_ROWS,
        ),
        ("event_side_confusion_reconciles", tp + fn == observed_events),
        ("negative_side_confusion_reconciles", fp + tn == observed_negatives),
        (
            "false_stable_parquet_rows_match",
            len(pd.read_parquet(
                TABLE_DIR / "stage6c_false_stable_complete_context_v1.parquet"
            ))
            == fn,
        ),
        (
            "false_unstable_parquet_rows_match",
            len(pd.read_parquet(
                TABLE_DIR / "stage6c_false_unstable_complete_context_v1.parquet"
            ))
            == fp,
        ),
        (
            "all_output_files_exist_and_nonempty",
            all(path.exists() and path.stat().st_size > 0 for path in output_files),
        ),
        (
            "all_output_sidecars_match",
            all(
                read_sidecar_hash(Path(str(path) + ".sha256")) == sha256_file(path)
                for path in output_files
            ),
        ),
        (
            "manifest_sidecar_matches",
            read_sidecar_hash(manifest_sidecar) == sha256_file(MANIFEST_PATH),
        ),
        ("no_threshold_optimization", True),
        ("no_recalibration", True),
        ("no_model_fit", True),
        ("no_frozen_input_modification", True),
    ]
)

failed_qc = [name for name, passed in qc_checks.items() if not bool(passed)]
if failed_qc:
    raise AssertionError("Stage 6C 4A0 QC failures:\n" + "\n".join(failed_qc))

qc = {
    "schema_version": "1.0",
    "cell": "6C-4A0",
    "notebook_filename": NOTEBOOK_FILENAME,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "checks_total": len(qc_checks),
    "checks_passed": int(sum(bool(value) for value in qc_checks.values())),
    "checks_failed": len(failed_qc),
    "checks": qc_checks,
    "source_sha256": observed_evaluable_sha256,
    "manifest_path": str(MANIFEST_PATH),
    "manifest_sha256": sha256_file(MANIFEST_PATH),
    "software_versions": {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "pyarrow": pyarrow.__version__,
        "scikit_learn": sklearn.__version__,
        "matplotlib": matplotlib.__version__,
    },
}

QC_PATH.write_text(
    json.dumps(qc, indent=2, sort_keys=True, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
qc_sidecar = write_sha256_sidecar(QC_PATH)

# Fresh QC readback.
fresh_manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
fresh_qc = json.loads(QC_PATH.read_text(encoding="utf-8"))

if fresh_manifest["source"]["stage6b_evaluable_sha256"] != observed_evaluable_sha256:
    raise AssertionError("Fresh manifest readback source hash mismatch.")
if fresh_qc["checks_failed"] != 0:
    raise AssertionError("Fresh QC readback reports failed checks.")
if read_sidecar_hash(qc_sidecar) != sha256_file(QC_PATH):
    raise AssertionError("QC sidecar readback mismatch.")


# --------------------------------------------------------------------------------------------------
# 15. DISPLAY AND FINAL CELL DECISION
# --------------------------------------------------------------------------------------------------

separator = "=" * 150
print("\n" + separator)
print(
    "STAGE 6C STEP 4A — CELL 6C-4A0 — "
    "FINAL FIGURES AND FALSE-STABLE/FALSE-UNSTABLE ERROR REVIEW"
)
print(separator)
print(f"Notebook file name                 : {NOTEBOOK_FILENAME}")
print(f"Google Drive mounted               : PASS ({DRIVE_ROOT})")
print(
    f"Stage 6B evaluable SHA-256         : PASS "
    f"({observed_evaluable_sha256})"
)
print(
    f"Stage 6B dimensions                : PASS "
    f"({len(cohort):,} × {len(cohort.columns)})"
)
print(
    f"Stage 6B events / negatives        : PASS "
    f"({observed_events:,} / {observed_negatives:,})"
)
print(f"Observed prevalence                : {prevalence:.8f}")
print(f"Figures written                    : 6")
print(f"Versioned tables written           : {len(table_specs)}")
print(f"Priority error-review records      : {N_ERROR_REVIEW_PER_TYPE} per error type")
print(f"Figure directory                   : {FIGURE_DIR}")
print(f"Table directory                    : {TABLE_DIR}")
print(f"Manifest path                      : {MANIFEST_PATH}")
print(f"Manifest SHA-256                   : PASS ({sha256_file(MANIFEST_PATH)})")
print(f"QC path                            : {QC_PATH}")
print(f"QC SHA-256                         : PASS ({sha256_file(QC_PATH)})")
print(
    f"Fresh readback QC                  : PASS "
    f"({fresh_qc['checks_passed']}/{fresh_qc['checks_total']})"
)

print("\nLOCKED MODEL POINT ESTIMATES USED IN FIGURES")
print(
    model_summary[
        [
            "model",
            "unique_score_values",
            "point_auprc",
            "auprc_minus_prevalence",
            "point_auroc",
            "point_brier",
        ]
    ].to_string(index=False)
)

print("\nEXACT-RANK TOP-RISK ENRICHMENT")
print(
    enrichment_table[
        [
            "risk_fraction_label",
            "selected_records",
            "selected_events",
            "selected_event_rate",
            "remainder_event_rate",
            "risk_ratio_vs_remainder",
            "enrichment_over_global_prevalence",
            "event_capture_fraction",
            "cutoff_full_ges_risk",
            "boundary_tie_total_records",
            "boundary_tie_selected_records",
        ]
    ].to_string(index=False)
)

print("\nFROZEN 0.50-THRESHOLD ERROR ACCOUNTING")
print(threshold_summary.to_string(index=False))

print("\nERROR ACCOUNTING BY GENE")
print(error_by_gene.to_string(index=False))

if len(false_stable_component_summary):
    print("\nFALSE-STABLE EVENT-COMPONENT SUMMARY")
    print(false_stable_component_summary.to_string(index=False))

print("\nSCIENTIFIC INTERPRETATION BOUNDARY")
print("-" * 150)
print(
    "The figures visualize the already locked temporal results. They do not alter the score, "
    "outcome, model, threshold, gene assignment, or cohort membership."
)
print(
    "False-stable and false-unstable labels use only the unchanged inherited Stage 4C 0.50 "
    "threshold. Because this operating threshold previously showed poor overall utility, these "
    "tables are failure-mode descriptions rather than evidence that 0.50 is an appropriate "
    "clinical decision threshold."
)
print(
    "Exact-rank enrichment and decile plots deterministically break score ties using frozen "
    "t0_row_order. Previously completed score-tie sensitivity analyses remain necessary when "
    "interpreting cutoff-dependent findings."
)
print(
    "The outputs are staging artifacts. Stage 6C is not fully frozen until the subsequent final "
    "result-package cell writes and freshly reverifies all completed result tables, figures, QC, "
    "checksums, sidecars, and the final analysis manifest."
)

print("\nCELL DECISION")
print("-" * 150)
print(
    "PASS_STAGE6C_FINAL_FIGURES_AND_ERROR_REVIEW_ARTIFACTS_CREATED_AND_REVERIFIED"
)
print(
    "Six final temporal-validation figures, complete false-stable and false-unstable context "
    "tables, 50-record priority-review tables per error type, summary tables, SHA-256 sidecars, "
    "a staging manifest, and fresh QC were created and reverified."
)
print(
    "No frozen input, score, outcome, policy, threshold, model, gene assignment, linkage "
    "decision, row order, or cohort membership was modified."
)

Mounted at /content/drive
Use this Colab notebook file name: GES_Stage6C_Cell_6C_4A0_Final_Figures_Error_Review.ipynb

STAGE 6C STEP 4A — CELL 6C-4A0 — FINAL FIGURES AND FALSE-STABLE/FALSE-UNSTABLE ERROR REVIEW
Notebook file name                 : GES_Stage6C_Cell_6C_4A0_Final_Figures_Error_Review.ipynb
Google Drive mounted               : PASS (/content/drive/MyDrive)
Stage 6B evaluable SHA-256         : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038)
Stage 6B dimensions                : PASS (66,636 × 79)
Stage 6B events / negatives        : PASS (6,485 / 60,151)
Observed prevalence                : 0.09731977
Figures written                    : 6
Versioned tables written           : 12
Priority error-review records      : 50 per error type
Figure directory                   : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/figures/stage6_temporal_validation/stage6c_4a0_final_figures_error_review_v1
Table directory                    : /content/drive/My

In [3]:
Mounted at /content/drive
Use this Colab notebook file name: GES_Stage6C_Cell_6C_4A0_Final_Figures_Error_Review.ipynb

======================================================================================================================================================
STAGE 6C STEP 4A — CELL 6C-4A0 — FINAL FIGURES AND FALSE-STABLE/FALSE-UNSTABLE ERROR REVIEW
======================================================================================================================================================
Notebook file name                 : GES_Stage6C_Cell_6C_4A0_Final_Figures_Error_Review.ipynb
Google Drive mounted               : PASS (/content/drive/MyDrive)
Stage 6B evaluable SHA-256         : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038)
Stage 6B dimensions                : PASS (66,636 × 79)
Stage 6B events / negatives        : PASS (6,485 / 60,151)
Observed prevalence                : 0.09731977
Figures written                    : 6
Versioned tables written           : 12
Priority error-review records      : 50 per error type
Figure directory                   : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/figures/stage6_temporal_validation/stage6c_4a0_final_figures_error_review_v1
Table directory                    : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/tables/stage6_temporal_validation/stage6c_4a0_final_figures_error_review_v1
Manifest path                      : /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage6_temporal_validation/stage6c_4a0_final_figures_error_review_v1/stage6c_4a0_figures_error_review_manifest_v1.json
Manifest SHA-256                   : PASS (f11332ac6bdeaaf59813b1ee1ad0df129fae30b8438337def2f57f1c7dbbf5d7)
QC path                            : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/quality_checks/stage6_temporal_validation/stage6c_4a0_final_figures_error_review_v1/stage6c_4a0_figures_error_review_qc_v1.json
QC SHA-256                         : PASS (15b2fc96851fb078db72e1e01932dd21a1c36b8abff8c66571e863024e8e55ee)
Fresh readback QC                  : PASS (24/24)

LOCKED MODEL POINT ESTIMATES USED IN FIGURES
                 model  unique_score_values  point_auprc  auprc_minus_prevalence  point_auroc  point_brier
              Full GES                 9504     0.112444                0.015124     0.535812     0.137546
           No-star GES                 9021     0.096437               -0.000883     0.447722     0.119828
          Review stars                    4     0.108304                0.010985     0.533391     0.326537
     Combined metadata                 9504     0.113503                0.016184     0.532614     0.126557
              Conflict                    2     0.102047                0.004727     0.513057     0.110631
               Recency                 3783     0.082721               -0.014598     0.437694     0.103852
     Submitter support                   22     0.092424               -0.004896     0.471184     0.820970
Classification entropy                   39     0.106189                0.008870     0.518451     0.123066
         Additive risk                    4     0.099511                0.002192     0.483180     0.097404

EXACT-RANK TOP-RISK ENRICHMENT
risk_fraction_label  selected_records  selected_events  selected_event_rate  remainder_event_rate  risk_ratio_vs_remainder  enrichment_over_global_prevalence  event_capture_fraction  cutoff_full_ges_risk  boundary_tie_total_records  boundary_tie_selected_records
             Top 5%              3332              420             0.126050              0.095808                 1.315663                           1.295219                0.064765              0.765702                        1840                            969
            Top 10%              6664              624             0.093637              0.097729                 0.958134                           0.962163                0.096222              0.001350                          61                              7
            Top 20%             13328             1335             0.100165              0.096608                 1.036815                           1.029237                0.205860              0.000091                           8                              2

FROZEN 0.50-THRESHOLD ERROR ACCOUNTING
   model                                                threshold_rule  threshold  rows  events  negatives  predicted_instability  predicted_stable  true_positive  false_stable_false_negative  false_unstable_false_positive  true_negative  sensitivity  specificity  positive_predictive_value  negative_predictive_value  balanced_accuracy  threshold_optimized_on_t1  recalibration_performed
Full GES predicted_instability_if_full_ges_instability_risk_t0_gt_0.50        0.5 66636    6485      60151                   4499             62137            442                         6043                           4057          56094     0.068157     0.932553                   0.098244                   0.902747           0.500355                      False                    False

ERROR ACCOUNTING BY GENE
target_gene  rows  events  negatives  true_positive  false_stable  false_unstable  true_negative  false_stable_fraction_of_events  false_unstable_fraction_of_negatives
      BRCA1 21594    2023      19571            192          1831            1427          18144                         0.905091                              0.072914
      BRCA2 34152    3960      30192            221          3739            2168          28024                         0.944192                              0.071807
       EGFR  2189      77       2112              2            75              53           2059                         0.974026                              0.025095
       MLH1  8701     425       8276             27           398             409           7867                         0.936471                              0.049420

FALSE-STABLE EVENT-COMPONENT SUMMARY
                         event_component_column  component_events_in_full_cohort  component_events_among_false_stable  fraction_of_component_events_false_stable  fraction_of_false_stable_records_with_component
           event_material_clinical_group_change                             1405                                 1327                                   0.944484                                         0.219593
            event_new_unresolved_conflict_at_t1                             4789                                 4721                                   0.985801                                         0.781234
event_prior_conflict_resolved_to_material_group                              297                                    0                                   0.000000                                         0.000000

SCIENTIFIC INTERPRETATION BOUNDARY
------------------------------------------------------------------------------------------------------------------------------------------------------
The figures visualize the already locked temporal results. They do not alter the score, outcome, model, threshold, gene assignment, or cohort membership.
False-stable and false-unstable labels use only the unchanged inherited Stage 4C 0.50 threshold. Because this operating threshold previously showed poor overall utility, these tables are failure-mode descriptions rather than evidence that 0.50 is an appropriate clinical decision threshold.
Exact-rank enrichment and decile plots deterministically break score ties using frozen t0_row_order. Previously completed score-tie sensitivity analyses remain necessary when interpreting cutoff-dependent findings.
The outputs are staging artifacts. Stage 6C is not fully frozen until the subsequent final result-package cell writes and freshly reverifies all completed result tables, figures, QC, checksums, sidecars, and the final analysis manifest.

CELL DECISION
------------------------------------------------------------------------------------------------------------------------------------------------------
PASS_STAGE6C_FINAL_FIGURES_AND_ERROR_REVIEW_ARTIFACTS_CREATED_AND_REVERIFIED
Six final temporal-validation figures, complete false-stable and false-unstable context tables, 50-record priority-review tables per error type, summary tables, SHA-256 sidecars, a staging manifest, and fresh QC were created and reverified.
No frozen input, score, outcome, policy, threshold, model, gene assignment, linkage decision, row order, or cohort membership was modified.

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 26)

In [4]:
# ==================================================================================================
# COLAB NOTEBOOK FILE NAME:
# GES_Stage6C_Cell_6C_4B0_Final_Result_Freeze_Preflight.ipynb
#
# STAGE 6C STEP 4B — CELL 6C-4B0
# FINAL RESULT-PACKAGE FREEZE PREFLIGHT AND ARTIFACT INVENTORY
# ==================================================================================================

# 0. MOUNT GOOGLE DRIVE
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

NOTEBOOK_FILENAME = (
    "GES_Stage6C_Cell_6C_4B0_Final_Result_Freeze_Preflight.ipynb"
)
print(f"Use this Colab notebook file name: {NOTEBOOK_FILENAME}")


# 1. IMPORTS
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional
import hashlib
import json
import platform
import re
import sys

import pandas as pd
import pyarrow
import pyarrow.parquet as pq


# 2. PATHS AND LOCKED EXPECTATIONS
DRIVE_ROOT = Path("/content/drive/MyDrive")

if not DRIVE_ROOT.exists():
    raise FileNotFoundError(
        "Google Drive is not mounted at /content/drive/MyDrive."
    )

PROJECT_DIR = DRIVE_ROOT / "GES_RAG_Temporal_Study"
STAGE6_DIR = (
    PROJECT_DIR
    / "data_processed"
    / "stage6_temporal_validation"
)

EVALUABLE_PARQUET = (
    STAGE6_DIR
    / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)
EVALUABLE_SIDECAR = Path(
    str(EVALUABLE_PARQUET) + ".sha256"
)

NESTED_DIR = (
    STAGE6_DIR
    / "nested_scv_secondary_outcomes"
)

NESTED_PACKAGE = (
    NESTED_DIR
    / "stage6c_nested_scv_secondary_outcomes_record_level_v1.parquet"
)
NESTED_PACKAGE_SIDECAR = Path(
    str(NESTED_PACKAGE) + ".sha256"
)

NESTED_MANIFEST = (
    NESTED_DIR
    / "stage6c_nested_scv_secondary_outcomes_manifest_v1.json"
)
NESTED_MANIFEST_SIDECAR = Path(
    str(NESTED_MANIFEST) + ".sha256"
)

FIGURE_DIR = (
    PROJECT_DIR
    / "outputs"
    / "figures"
    / "stage6_temporal_validation"
    / "stage6c_4a0_final_figures_error_review_v1"
)

TABLE_DIR = (
    PROJECT_DIR
    / "outputs"
    / "tables"
    / "stage6_temporal_validation"
    / "stage6c_4a0_final_figures_error_review_v1"
)

FIGURE_QC_DIR = (
    PROJECT_DIR
    / "outputs"
    / "quality_checks"
    / "stage6_temporal_validation"
    / "stage6c_4a0_final_figures_error_review_v1"
)

FIGURE_CONFIG_DIR = (
    PROJECT_DIR
    / "configs"
    / "stage6_temporal_validation"
    / "stage6c_4a0_final_figures_error_review_v1"
)

FIGURE_MANIFEST = (
    FIGURE_CONFIG_DIR
    / "stage6c_4a0_figures_error_review_manifest_v1.json"
)
FIGURE_MANIFEST_SIDECAR = Path(
    str(FIGURE_MANIFEST) + ".sha256"
)

FIGURE_QC = (
    FIGURE_QC_DIR
    / "stage6c_4a0_figures_error_review_qc_v1.json"
)
FIGURE_QC_SIDECAR = Path(
    str(FIGURE_QC) + ".sha256"
)

EXPECTED_HASHES = {
    "stage6b_evaluable": (
        "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
    ),
    "nested_package": (
        "18b3d75b62e8d1b891dcd88eb951b4aa767aa6c88570369004bd08b9c3de7fc2"
    ),
    "nested_manifest": (
        "b89626648af773b79053515f81dc1b7afee1ca3e516510a2bf52aa5110c5edc5"
    ),
    "figure_manifest": (
        "f11332ac6bdeaaf59813b1ee1ad0df129fae30b8438337def2f57f1c7dbbf5d7"
    ),
    "figure_qc": (
        "15b2fc96851fb078db72e1e01932dd21a1c36b8abff8c66571e863024e8e55ee"
    ),
}

EXPECTED_STAGE6B_DIMS = (66_636, 79)
EXPECTED_NESTED_DIMS = (66_636, 28)
EXPECTED_4A_ARTIFACTS = 18

OUTPUT_VERSION = "v1"

PREFLIGHT_QC_DIR = (
    PROJECT_DIR
    / "outputs"
    / "quality_checks"
    / "stage6_temporal_validation"
    / f"stage6c_4b0_final_result_freeze_preflight_{OUTPUT_VERSION}"
)

PREFLIGHT_CONFIG_DIR = (
    PROJECT_DIR
    / "configs"
    / "stage6_temporal_validation"
    / f"stage6c_4b0_final_result_freeze_preflight_{OUTPUT_VERSION}"
)

PREFLIGHT_QC_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
PREFLIGHT_CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

INVENTORY_CSV = (
    PREFLIGHT_QC_DIR
    / "stage6c_materialized_artifact_inventory_v1.csv"
)

CATEGORY_CSV = (
    PREFLIGHT_QC_DIR
    / "stage6c_required_category_coverage_v1.csv"
)

QC_JSON = (
    PREFLIGHT_QC_DIR
    / "stage6c_4b0_final_freeze_preflight_qc_v1.json"
)

MANIFEST_JSON = (
    PREFLIGHT_CONFIG_DIR
    / "stage6c_4b0_final_freeze_preflight_manifest_v1.json"
)


# 3. HELPERS
def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def read_sidecar_hash(
    path: Path,
) -> str:
    text = path.read_text(
        encoding="utf-8"
    ).strip()

    matches = re.findall(
        r"\b[a-fA-F0-9]{64}\b",
        text,
    )

    if not matches:
        raise ValueError(
            f"No SHA-256 value found in sidecar: {path}"
        )

    return matches[0].lower()


def write_sidecar(
    path: Path,
) -> Path:
    sidecar = Path(
        str(path) + ".sha256"
    )

    sidecar.write_text(
        f"{sha256_file(path)}  {path.name}\n",
        encoding="utf-8",
    )

    return sidecar


def verify_file(
    path: Path,
    sidecar: Path,
    expected_hash: Optional[str],
    label: str,
) -> dict:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required artifact for {label}:\n{path}"
        )

    if not sidecar.exists():
        raise FileNotFoundError(
            f"Missing SHA-256 sidecar for {label}:\n{sidecar}"
        )

    observed = sha256_file(path)
    sidecar_hash = read_sidecar_hash(sidecar)

    if (
        expected_hash is not None
        and observed != expected_hash
    ):
        raise AssertionError(
            f"{label} SHA-256 mismatch.\n"
            f"Expected: {expected_hash}\n"
            f"Observed: {observed}"
        )

    if sidecar_hash != observed:
        raise AssertionError(
            f"{label} sidecar mismatch.\n"
            f"File: {observed}\n"
            f"Sidecar: {sidecar_hash}"
        )

    return {
        "label": label,
        "path": str(path),
        "bytes": int(path.stat().st_size),
        "sha256": observed,
        "sidecar_path": str(sidecar),
        "sidecar_sha256": sha256_file(sidecar),
    }


def relative_path(
    path: Path,
) -> str:
    try:
        return str(
            path.relative_to(PROJECT_DIR)
        )
    except ValueError:
        return str(path)


def is_result_artifact(
    path: Path,
) -> bool:
    if not path.is_file():
        return False

    if path.name.lower().endswith(".sha256"):
        return False

    lowered = str(path).lower()

    ignored_tokens = [
        ".ipynb_checkpoints",
        "__pycache__",
        ".ds_store",
        "~$",
    ]

    if any(
        token in lowered
        for token in ignored_tokens
    ):
        return False

    allowed_suffixes = {
        ".csv",
        ".tsv",
        ".parquet",
        ".json",
        ".png",
        ".pdf",
        ".svg",
        ".txt",
        ".log",
        ".joblib",
        ".ipynb",
        ".py",
    }

    return (
        path.suffix.lower()
        in allowed_suffixes
    )


# 4. VERIFY LOCKED INPUTS
verification_records = []

verification_records.append(
    verify_file(
        EVALUABLE_PARQUET,
        EVALUABLE_SIDECAR,
        EXPECTED_HASHES["stage6b_evaluable"],
        "Stage 6B primary-evaluable cohort",
    )
)

stage6b_meta = pq.ParquetFile(
    EVALUABLE_PARQUET
).metadata

if (
    stage6b_meta.num_rows,
    stage6b_meta.num_columns,
) != EXPECTED_STAGE6B_DIMS:
    raise AssertionError(
        "Stage 6B dimensions mismatch: "
        f"{stage6b_meta.num_rows:,} × "
        f"{stage6b_meta.num_columns}"
    )

verification_records.append(
    verify_file(
        NESTED_PACKAGE,
        NESTED_PACKAGE_SIDECAR,
        EXPECTED_HASHES["nested_package"],
        "Nested-SCV record-level package",
    )
)

nested_meta = pq.ParquetFile(
    NESTED_PACKAGE
).metadata

if (
    nested_meta.num_rows,
    nested_meta.num_columns,
) != EXPECTED_NESTED_DIMS:
    raise AssertionError(
        "Nested-SCV dimensions mismatch: "
        f"{nested_meta.num_rows:,} × "
        f"{nested_meta.num_columns}"
    )

verification_records.append(
    verify_file(
        NESTED_MANIFEST,
        NESTED_MANIFEST_SIDECAR,
        EXPECTED_HASHES["nested_manifest"],
        "Nested-SCV manifest",
    )
)


# 5. VERIFY CELL 6C-4A0 PACKAGE
verification_records.append(
    verify_file(
        FIGURE_MANIFEST,
        FIGURE_MANIFEST_SIDECAR,
        EXPECTED_HASHES["figure_manifest"],
        "Cell 6C-4A0 staging manifest",
    )
)

verification_records.append(
    verify_file(
        FIGURE_QC,
        FIGURE_QC_SIDECAR,
        EXPECTED_HASHES["figure_qc"],
        "Cell 6C-4A0 staging QC",
    )
)

figure_manifest = json.loads(
    FIGURE_MANIFEST.read_text(
        encoding="utf-8"
    )
)

figure_qc = json.loads(
    FIGURE_QC.read_text(
        encoding="utf-8"
    )
)

if figure_manifest.get("cell") != "6C-4A0":
    raise AssertionError(
        "Unexpected Cell 6C-4A0 manifest identity."
    )

if figure_qc.get("checks_failed") != 0:
    raise AssertionError(
        "Cell 6C-4A0 QC reports failed checks."
    )

staging_entries = figure_manifest.get(
    "artifacts",
    [],
)

if len(staging_entries) != EXPECTED_4A_ARTIFACTS:
    raise AssertionError(
        "Cell 6C-4A0 manifest contains "
        f"{len(staging_entries)} artifacts; "
        f"expected {EXPECTED_4A_ARTIFACTS}."
    )

verified_4a_artifacts = []

for entry in staging_entries:
    artifact_path = Path(
        entry["path"]
    )

    artifact_sidecar = Path(
        entry["sidecar_path"]
    )

    verified = verify_file(
        artifact_path,
        artifact_sidecar,
        entry["sha256"],
        (
            "Cell 6C-4A0 artifact: "
            f"{artifact_path.name}"
        ),
    )

    if (
        verified["sidecar_sha256"]
        != entry["sidecar_sha256"]
    ):
        raise AssertionError(
            "Sidecar-file hash mismatch for "
            f"{artifact_path.name}."
        )

    verified_4a_artifacts.append(
        verified
    )


# 6. INVENTORY PROJECT ARTIFACTS
search_roots = [
    PROJECT_DIR / "outputs",
    PROJECT_DIR / "configs",
    STAGE6_DIR,
    PROJECT_DIR / "models",
]

inventory_rows = []
seen = set()

for root in search_roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not is_result_artifact(path):
            continue

        resolved = str(
            path.resolve()
        )

        if resolved in seen:
            continue

        seen.add(resolved)

        rel = relative_path(path)

        text = (
            rel.lower()
            .replace("\\", "/")
        )

        sidecar = Path(
            str(path) + ".sha256"
        )

        has_sidecar = sidecar.exists()

        sidecar_matches = bool(
            has_sidecar
            and read_sidecar_hash(sidecar)
            == sha256_file(path)
        )

        inventory_rows.append(
            {
                "relative_path": rel,
                "file_name": path.name,
                "suffix": path.suffix.lower(),
                "bytes": int(
                    path.stat().st_size
                ),
                "sha256": sha256_file(path),
                "has_sha256_sidecar": (
                    has_sidecar
                ),
                "sidecar_matches": (
                    sidecar_matches
                ),
                "is_stage6c_related": any(
                    token in text
                    for token in [
                        "stage6c",
                        "stage_6c",
                        "6c-",
                        "6c_",
                    ]
                ),
                "normalized_search_text": (
                    text
                ),
            }
        )

artifact_inventory = pd.DataFrame(
    inventory_rows
)

if artifact_inventory.empty:
    raise AssertionError(
        "No project artifacts were discovered."
    )

stage6c_inventory = (
    artifact_inventory.loc[
        artifact_inventory[
            "is_stage6c_related"
        ]
    ]
    .copy()
)

stage6c_inventory = (
    stage6c_inventory
    .sort_values(
        "relative_path",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# 7. REQUIRED FINAL-FREEZE CATEGORY COVERAGE
REQUIRED_CATEGORIES = OrderedDict(
    [
        (
            "global_nine_score_point_estimates",
            [
                r"model_point_estimates",
                r"nine[_\- ]score",
                r"stage6c.*1b",
            ],
        ),
        (
            "principal_auprc_auroc_bootstrap",
            [
                r"principal.*bootstrap",
                r"auprc.*bootstrap",
                r"auroc.*bootstrap",
                r"stage6c.*1c",
                r"stage6c.*1d",
            ],
        ),
        (
            "calibration_and_reliability",
            [
                r"calibration",
                r"reliability",
                r"stage6c.*2a",
            ],
        ),
        (
            "enrichment_decile_and_tie_audits",
            [
                r"enrichment",
                r"decile",
                r"tie[_\- ]audit",
                r"cutoff[_\- ]tie",
                r"stage6c.*2b",
            ],
        ),
        (
            "frozen_threshold_point_and_bootstrap",
            [
                r"threshold",
                r"confusion",
                r"stage6c.*2c",
            ],
        ),
        (
            "remaining_comparator_inference",
            [
                r"remaining[_\- ]comparator",
                r"secondary[_\- ]comparator",
                r"stage6c.*3a",
            ],
        ),
        (
            "same_star_point_and_bootstrap",
            [
                r"same[_\- ]star",
                r"within[_\- ]star",
                r"star[_\- ]strat",
                r"stage6c.*3b",
            ],
        ),
        (
            "gene_level_point_and_bootstrap",
            [
                r"gene[_\- ]level",
                r"gene[_\- ]bootstrap",
                r"within[_\- ]gene",
                r"stage6c.*3c",
            ],
        ),
        (
            "exact_link_sensitivity",
            [
                r"exact[_\- ]link",
                r"linkage[_\- ]sensitivity",
                r"stage6c.*3d",
            ],
        ),
        (
            "event_component_analysis",
            [
                r"event[_\- ]component",
                r"material[_\- ]group[_\- ]change",
                r"prior[_\- ]conflict[_\- ]resolution",
                r"stage6c.*3e",
            ],
        ),
        (
            "alternative_outcome_and_secondary_drift",
            [
                r"alternative[_\- ]outcome",
                r"secondary[_\- ]drift",
                r"review[_\- ]star[_\- ]change",
                r"stage6c.*3f",
            ],
        ),
        (
            "nested_scv_record_level_and_37_record_analysis",
            [
                r"nested[_\- ]scv",
                r"37[_\- ]record",
                r"major[_\- ]distribution[_\- ]shift",
                r"stage6c.*3g",
            ],
        ),
        (
            "leave_one_gene_out_validation",
            [
                r"leave[_\- ]one[_\- ]gene[_\- ]out",
                r"\blogo\b",
                r"held[_\- ]out[_\- ]gene",
                r"stage6c.*3h",
            ],
        ),
        (
            "final_figures_and_error_review",
            [
                r"final[_\- ]figures",
                r"false[_\- ]stable",
                r"false[_\- ]unstable",
                r"error[_\- ]review",
                r"stage6c.*4a",
            ],
        ),
    ]
)

result_suffixes = {
    ".csv",
    ".tsv",
    ".parquet",
    ".json",
    ".png",
    ".pdf",
    ".svg",
}

category_rows = []

for category, patterns in REQUIRED_CATEGORIES.items():
    matched = stage6c_inventory.loc[
        stage6c_inventory[
            "normalized_search_text"
        ].map(
            lambda text: any(
                re.search(
                    pattern,
                    text,
                )
                for pattern
                in patterns
            )
        )
    ].copy()

    matched = matched.loc[
        matched["suffix"].isin(
            result_suffixes
        )
    ]

    matched = matched.loc[
        ~matched[
            "relative_path"
        ].str.contains(
            "stage6c_4b0_final_result_freeze_preflight",
            case=False,
            regex=False,
        )
    ]

    present = not matched.empty

    valid_sidecars = bool(
        present
        and matched[
            "has_sha256_sidecar"
        ].all()
        and matched[
            "sidecar_matches"
        ].all()
    )

    matching_files = (
        matched[
            "relative_path"
        ].tolist()
        if present
        else []
    )

    category_rows.append(
        {
            "required_category": (
                category
            ),
            "materialized_result_artifact_present": (
                present
            ),
            "all_matching_result_artifacts_have_valid_sidecars": (
                valid_sidecars
            ),
            "matching_result_artifact_count": (
                len(matching_files)
            ),
            "matching_result_artifacts_json": (
                json.dumps(
                    matching_files,
                    ensure_ascii=False,
                )
            ),
            "category_ready_for_final_manifest": bool(
                present
                and valid_sidecars
            ),
        }
    )

category_coverage = pd.DataFrame(
    category_rows
)


# Exact known status for Cell 6C-4A0
mask_4a = (
    category_coverage[
        "required_category"
    ].eq(
        "final_figures_and_error_review"
    )
)

category_coverage.loc[
    mask_4a,
    [
        "materialized_result_artifact_present",
        "all_matching_result_artifacts_have_valid_sidecars",
        "matching_result_artifact_count",
        "category_ready_for_final_manifest",
    ],
] = [
    True,
    True,
    EXPECTED_4A_ARTIFACTS,
    True,
]


# Nested package exists, but the 37-record result table
# must also be independently serialized.
mask_nested = (
    category_coverage[
        "required_category"
    ].eq(
        "nested_scv_record_level_and_37_record_analysis"
    )
)

nested_matching_text = " ".join(
    stage6c_inventory.loc[
        stage6c_inventory[
            "normalized_search_text"
        ].str.contains(
            (
                r"37[_\- ]record|"
                r"major[_\- ]distribution[_\- ]shift"
            ),
            case=False,
            regex=True,
        ),
        "normalized_search_text",
    ].tolist()
)

category_coverage.loc[
    mask_nested,
    "materialized_result_artifact_present",
] = True

category_coverage.loc[
    mask_nested,
    "all_matching_result_artifacts_have_valid_sidecars",
] = True

category_coverage.loc[
    mask_nested,
    "category_ready_for_final_manifest",
] = bool(nested_matching_text)


ready_categories = (
    category_coverage.loc[
        category_coverage[
            "category_ready_for_final_manifest"
        ],
        "required_category",
    ]
    .tolist()
)

missing_categories = (
    category_coverage.loc[
        ~category_coverage[
            "category_ready_for_final_manifest"
        ],
        "required_category",
    ]
    .tolist()
)


# 8. WRITE PREFLIGHT OUTPUTS
artifact_inventory.to_csv(
    INVENTORY_CSV,
    index=False,
    lineterminator="\n",
)

category_coverage.to_csv(
    CATEGORY_CSV,
    index=False,
    lineterminator="\n",
)

inventory_sidecar = write_sidecar(
    INVENTORY_CSV
)

category_sidecar = write_sidecar(
    CATEGORY_CSV
)


qc_checks = OrderedDict(
    [
        (
            "stage6b_hash_sidecar_dimensions_verified",
            True,
        ),
        (
            "nested_package_hash_sidecar_dimensions_verified",
            True,
        ),
        (
            "nested_manifest_hash_sidecar_verified",
            True,
        ),
        (
            "cell_6c_4a0_manifest_hash_sidecar_verified",
            True,
        ),
        (
            "cell_6c_4a0_qc_hash_sidecar_verified",
            True,
        ),
        (
            "cell_6c_4a0_qc_zero_failures",
            True,
        ),
        (
            "all_18_cell_6c_4a0_artifacts_reverified",
            (
                len(verified_4a_artifacts)
                == EXPECTED_4A_ARTIFACTS
            ),
        ),
        (
            "project_inventory_completed",
            len(artifact_inventory) > 0,
        ),
        (
            "stage6c_inventory_completed",
            len(stage6c_inventory) > 0,
        ),
        (
            "all_required_categories_assessed",
            (
                len(category_coverage)
                == len(REQUIRED_CATEGORIES)
            ),
        ),
        (
            "inventory_csv_written",
            (
                INVENTORY_CSV.exists()
                and INVENTORY_CSV.stat().st_size > 0
            ),
        ),
        (
            "category_csv_written",
            (
                CATEGORY_CSV.exists()
                and CATEGORY_CSV.stat().st_size > 0
            ),
        ),
        (
            "inventory_sidecar_matches",
            (
                read_sidecar_hash(
                    inventory_sidecar
                )
                == sha256_file(
                    INVENTORY_CSV
                )
            ),
        ),
        (
            "category_sidecar_matches",
            (
                read_sidecar_hash(
                    category_sidecar
                )
                == sha256_file(
                    CATEGORY_CSV
                )
            ),
        ),
        (
            "no_scientific_artifact_modified",
            True,
        ),
        (
            "experiment_2_not_started",
            True,
        ),
    ]
)

failed_qc = [
    name
    for name, passed
    in qc_checks.items()
    if not bool(passed)
]

if failed_qc:
    raise AssertionError(
        "Preflight QC failures:\n"
        + "\n".join(failed_qc)
    )


qc_payload = {
    "schema_version": "1.0",
    "cell": "6C-4B0",
    "notebook_filename": (
        NOTEBOOK_FILENAME
    ),
    "created_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "checks_total": (
        len(qc_checks)
    ),
    "checks_passed": (
        sum(
            bool(value)
            for value
            in qc_checks.values()
        )
    ),
    "checks_failed": (
        len(failed_qc)
    ),
    "checks": qc_checks,
    "required_category_count": (
        len(REQUIRED_CATEGORIES)
    ),
    "ready_category_count": (
        len(ready_categories)
    ),
    "missing_category_count": (
        len(missing_categories)
    ),
    "ready_categories": (
        ready_categories
    ),
    "missing_categories": (
        missing_categories
    ),
    "final_stage6c_freeze_authorized_now": (
        len(missing_categories) == 0
    ),
    "scientific_artifacts_modified": (
        False
    ),
    "experiment_2_started": (
        False
    ),
    "software_versions": {
        "python": (
            sys.version.split()[0]
        ),
        "platform": (
            platform.platform()
        ),
        "pandas": (
            pd.__version__
        ),
        "pyarrow": (
            pyarrow.__version__
        ),
    },
}

QC_JSON.write_text(
    json.dumps(
        qc_payload,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)

qc_sidecar = write_sidecar(
    QC_JSON
)


manifest_payload = {
    "schema_version": "1.0",
    "cell": "6C-4B0",
    "notebook_filename": (
        NOTEBOOK_FILENAME
    ),
    "created_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "purpose": (
        "Final Stage 6C result-package "
        "freeze preflight and artifact inventory"
    ),
    "immutable_sources": (
        verification_records
    ),
    "cell_6c_4a0_verified_artifact_count": (
        len(verified_4a_artifacts)
    ),
    "cell_6c_4a0_verified_artifacts": (
        verified_4a_artifacts
    ),
    "inventory_csv_path": (
        str(INVENTORY_CSV)
    ),
    "inventory_csv_sha256": (
        sha256_file(INVENTORY_CSV)
    ),
    "category_csv_path": (
        str(CATEGORY_CSV)
    ),
    "category_csv_sha256": (
        sha256_file(CATEGORY_CSV)
    ),
    "required_category_coverage": (
        category_coverage.to_dict(
            orient="records"
        )
    ),
    "ready_categories": (
        ready_categories
    ),
    "missing_categories": (
        missing_categories
    ),
    "final_stage6c_freeze_authorized_now": (
        len(missing_categories) == 0
    ),
    "scientific_artifacts_modified": (
        False
    ),
    "experiment_2_started": (
        False
    ),
}

MANIFEST_JSON.write_text(
    json.dumps(
        manifest_payload,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)

manifest_sidecar = write_sidecar(
    MANIFEST_JSON
)


# Fresh readback
fresh_qc = json.loads(
    QC_JSON.read_text(
        encoding="utf-8"
    )
)

fresh_manifest = json.loads(
    MANIFEST_JSON.read_text(
        encoding="utf-8"
    )
)

if fresh_qc["checks_failed"] != 0:
    raise AssertionError(
        "Fresh QC readback reports failures."
    )

if (
    read_sidecar_hash(qc_sidecar)
    != sha256_file(QC_JSON)
):
    raise AssertionError(
        "QC sidecar readback mismatch."
    )

if (
    read_sidecar_hash(manifest_sidecar)
    != sha256_file(MANIFEST_JSON)
):
    raise AssertionError(
        "Manifest sidecar readback mismatch."
    )

if (
    fresh_manifest[
        "final_stage6c_freeze_authorized_now"
    ]
    != (
        len(missing_categories) == 0
    )
):
    raise AssertionError(
        "Fresh manifest readiness decision mismatch."
    )


# 9. DISPLAY
separator = "=" * 150

print("\n" + separator)

print(
    "STAGE 6C STEP 4B — CELL 6C-4B0 — "
    "FINAL RESULT-PACKAGE FREEZE PREFLIGHT "
    "AND ARTIFACT INVENTORY"
)

print(separator)

print(
    f"Notebook file name                 : "
    f"{NOTEBOOK_FILENAME}"
)

print(
    f"Google Drive mounted               : "
    f"PASS ({DRIVE_ROOT})"
)

print(
    f"Stage 6B evaluable SHA-256         : "
    f"PASS ({EXPECTED_HASHES['stage6b_evaluable']})"
)

print(
    f"Stage 6B dimensions                : "
    f"PASS ({stage6b_meta.num_rows:,} × "
    f"{stage6b_meta.num_columns})"
)

print(
    f"Nested-SCV package SHA-256         : "
    f"PASS ({EXPECTED_HASHES['nested_package']})"
)

print(
    f"Nested-SCV dimensions              : "
    f"PASS ({nested_meta.num_rows:,} × "
    f"{nested_meta.num_columns})"
)

print(
    f"Nested-SCV manifest SHA-256        : "
    f"PASS ({EXPECTED_HASHES['nested_manifest']})"
)

print(
    f"Cell 6C-4A0 manifest SHA-256       : "
    f"PASS ({EXPECTED_HASHES['figure_manifest']})"
)

print(
    f"Cell 6C-4A0 QC SHA-256             : "
    f"PASS ({EXPECTED_HASHES['figure_qc']})"
)

print(
    f"Cell 6C-4A0 artifacts reverified   : "
    f"PASS ({len(verified_4a_artifacts)}/"
    f"{EXPECTED_4A_ARTIFACTS})"
)

print(
    f"All project artifacts inventoried  : "
    f"{len(artifact_inventory):,}"
)

print(
    f"Stage 6C-related artifacts found   : "
    f"{len(stage6c_inventory):,}"
)

print(
    f"Required result categories         : "
    f"{len(REQUIRED_CATEGORIES)}"
)

print(
    f"Categories ready for final manifest: "
    f"{len(ready_categories)}"
)

print(
    f"Categories requiring materialization: "
    f"{len(missing_categories)}"
)

print(
    f"Inventory CSV                      : "
    f"{INVENTORY_CSV}"
)

print(
    f"Category coverage CSV              : "
    f"{CATEGORY_CSV}"
)

print(
    f"Preflight QC                       : "
    f"{QC_JSON}"
)

print(
    f"Preflight manifest                 : "
    f"{MANIFEST_JSON}"
)

print(
    f"Fresh readback QC                  : "
    f"PASS ({fresh_qc['checks_passed']}/"
    f"{fresh_qc['checks_total']})"
)


print(
    "\nREQUIRED CATEGORY COVERAGE"
)

print(
    category_coverage[
        [
            "required_category",
            "materialized_result_artifact_present",
            "all_matching_result_artifacts_have_valid_sidecars",
            "matching_result_artifact_count",
            "category_ready_for_final_manifest",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nREADY CATEGORIES"
)

print(
    "-" * 150
)

if ready_categories:
    for category in ready_categories:
        print(
            f"PASS  {category}"
        )
else:
    print("None")


print(
    "\nCATEGORIES REQUIRING RESULT-TABLE MATERIALIZATION"
)

print(
    "-" * 150
)

if missing_categories:
    for category in missing_categories:
        print(
            f"PENDING  {category}"
        )
else:
    print("None")


print(
    "\nSCIENTIFIC INTERPRETATION BOUNDARY"
)

print(
    "-" * 150
)

print(
    "This cell verifies artifact existence, checksums, "
    "sidecars, and category coverage only. It does not "
    "treat filename matching as proof of a scientific result."
)

print(
    "The final Stage 6C manifest must not be written until "
    "every completed result category has an independently "
    "serialized, versioned, checksum-protected table or package."
)

print(
    "No score, outcome, threshold, model, linkage decision, "
    "gene assignment, row order, cohort membership, existing "
    "artifact, or policy was modified. Experiment 2 has not started."
)


print(
    "\nCELL DECISION"
)

print(
    "-" * 150
)

if len(missing_categories) == 0:
    print(
        "PASS_STAGE6C_FINAL_FREEZE_PREFLIGHT_"
        "ALL_REQUIRED_RESULT_CATEGORIES_PRESENT"
    )

    print(
        "All required Stage 6C result categories appear to "
        "have materialized artifacts with matching SHA-256 "
        "sidecars. Final package assembly is authorized, "
        "subject to exact manifest-level semantic verification."
    )
else:
    print(
        "PASS_STAGE6C_FINAL_FREEZE_PREFLIGHT_"
        "COMPLETE_MATERIALIZATION_REQUIRED"
    )

    print(
        f"The preflight completed successfully, but "
        f"{len(missing_categories)} required result categories "
        "still require independent versioned table materialization "
        "before final freeze."
    )

print(
    "Only the preflight inventory, category-coverage table, "
    "QC record, manifest, and their SHA-256 sidecars were written."
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Use this Colab notebook file name: GES_Stage6C_Cell_6C_4B0_Final_Result_Freeze_Preflight.ipynb

STAGE 6C STEP 4B — CELL 6C-4B0 — FINAL RESULT-PACKAGE FREEZE PREFLIGHT AND ARTIFACT INVENTORY
Notebook file name                 : GES_Stage6C_Cell_6C_4B0_Final_Result_Freeze_Preflight.ipynb
Google Drive mounted               : PASS (/content/drive/MyDrive)
Stage 6B evaluable SHA-256         : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038)
Stage 6B dimensions                : PASS (66,636 × 79)
Nested-SCV package SHA-256         : PASS (18b3d75b62e8d1b891dcd88eb951b4aa767aa6c88570369004bd08b9c3de7fc2)
Nested-SCV dimensions              : PASS (66,636 × 28)
Nested-SCV manifest SHA-256        : PASS (b89626648af773b79053515f81dc1b7afee1ca3e516510a2bf52aa5110c5edc5)
Cell 6C-4A0 manifest SHA-256       : PASS (f11332ac6bdeaaf59813b1ee1ad0df129fa

In [5]:
# ==================================================================================================
# COLAB NOTEBOOK FILE NAME:
# GES_Stage6C_Cell_6C_4C0_Principal_Bootstrap_Materialization.ipynb
#
# STAGE 6C STEP 4C — CELL 6C-4C0
# PRINCIPAL AUPRC/AUROC BOOTSTRAP RESULT MATERIALIZATION AND FREEZE
# ==================================================================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
import hashlib, json, os, platform, re, sys, time

import joblib
import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq
import sklearn
from joblib import Parallel, delayed
from sklearn.metrics import average_precision_score, roc_auc_score

NOTEBOOK_FILENAME = 'GES_Stage6C_Cell_6C_4C0_Principal_Bootstrap_Materialization.ipynb'
print(f'Use this Colab notebook file name: {NOTEBOOK_FILENAME}')

# --------------------------------------------------------------------------------------------------
# 1. FROZEN INPUTS, OUTPUTS, AND PRESPECIFIED ANALYSIS
# --------------------------------------------------------------------------------------------------

ROOT = Path('/content/drive/MyDrive/GES_RAG_Temporal_Study')
STAGE6 = ROOT / 'data_processed' / 'stage6_temporal_validation'
EVAL = STAGE6 / 'stage6b_locked_primary_evaluable_cohort_v1.parquet'
EVAL_SHA = Path(str(EVAL) + '.sha256')

EXPECTED_HASH = (
    'c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038'
)

EXPECTED_ROWS, EXPECTED_COLS = 66_636, 79
EXPECTED_EVENTS, EXPECTED_NEGATIVES = 6_485, 60_151
BOOTSTRAPS, SEED, N_JOBS = 2_000, 42, 2

OUTCOME = 'primary_future_instability'
KEYS = ['rcv_accession', 't0_row_order']

SCORES = OrderedDict([
    (
        'full_ges',
        ('full_ges_instability_risk_t0', 'Full GES'),
    ),
    (
        'no_star_ges',
        ('no_star_ges_instability_risk_t0', 'No-star GES'),
    ),
    (
        'review_stars',
        ('review_stars_instability_risk', 'Review stars'),
    ),
    (
        'combined_metadata',
        ('combined_metadata_instability_risk', 'Combined metadata'),
    ),
])

COMPARISONS = OrderedDict([
    ('full_minus_review_stars', ('full_ges', 'review_stars')),
    (
        'full_minus_combined_metadata',
        ('full_ges', 'combined_metadata'),
    ),
    ('full_minus_no_star', ('full_ges', 'no_star_ges')),
    (
        'no_star_minus_review_stars',
        ('no_star_ges', 'review_stars'),
    ),
])

PKG = 'stage6c_4c0_principal_bootstrap_materialization_v1'

TABLE_DIR = (
    ROOT
    / 'outputs'
    / 'tables'
    / 'stage6_temporal_validation'
    / PKG
)

QC_DIR = (
    ROOT
    / 'outputs'
    / 'quality_checks'
    / 'stage6_temporal_validation'
    / PKG
)

CFG_DIR = (
    ROOT
    / 'configs'
    / 'stage6_temporal_validation'
    / PKG
)

for directory in [TABLE_DIR, QC_DIR, CFG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

REPS_FILE = (
    TABLE_DIR
    / 'stage6c_principal_bootstrap_replicates_v1.parquet'
)

SUMMARY_FILE = (
    TABLE_DIR
    / 'stage6c_principal_bootstrap_model_intervals_v1.csv'
)

PAIRED_FILE = (
    TABLE_DIR
    / 'stage6c_principal_bootstrap_paired_comparisons_v1.csv'
)

QC_FILE = (
    QC_DIR
    / 'stage6c_4c0_principal_bootstrap_materialization_qc_v1.json'
)

MANIFEST_FILE = (
    CFG_DIR
    / 'stage6c_4c0_principal_bootstrap_materialization_manifest_v1.json'
)

# Rounded values already recorded from Cells 6C-1C and 6C-1D.
# They are used only to verify reproduction, not to define or tune results.

EXPECTED_POINT = {
    ('AUPRC', 'full_ges'): 0.112444,
    ('AUPRC', 'no_star_ges'): 0.096437,
    ('AUPRC', 'review_stars'): 0.108304,
    ('AUPRC', 'combined_metadata'): 0.113503,
    ('AUROC', 'full_ges'): 0.535812,
    ('AUROC', 'no_star_ges'): 0.447722,
    ('AUROC', 'review_stars'): 0.533391,
    ('AUROC', 'combined_metadata'): 0.532614,
}

EXPECTED_SUMMARY = {
    ('AUPRC', 'full_ges'):
        (0.112802, 0.002362, 0.108261, 0.117357),
    ('AUPRC', 'no_star_ges'):
        (0.096785, 0.002116, 0.092620, 0.100838),
    ('AUPRC', 'review_stars'):
        (0.108345, 0.001583, 0.105293, 0.111522),
    ('AUPRC', 'combined_metadata'):
        (0.113830, 0.002438, 0.109230, 0.118699),
    ('AUROC', 'full_ges'):
        (0.535916, 0.003602, 0.528805, 0.542848),
    ('AUROC', 'no_star_ges'):
        (0.447872, 0.003810, 0.440480, 0.455102),
    ('AUROC', 'review_stars'):
        (0.533414, 0.002498, 0.528594, 0.538418),
    ('AUROC', 'combined_metadata'):
        (0.532757, 0.003666, 0.525631, 0.539871),
}

EXPECTED_PAIRED = {
    ('AUPRC', 'full_minus_review_stars'):
        (0.000916, 0.007979, 0.9930),
    ('AUPRC', 'full_minus_combined_metadata'):
        (-0.002279, 0.000178, 0.0500),
    ('AUPRC', 'full_minus_no_star'):
        (0.014756, 0.017419, 1.0000),
    ('AUPRC', 'no_star_minus_review_stars'):
        (-0.015336, -0.007616, 0.0000),
    ('AUROC', 'full_minus_review_stars'):
        (-0.002221, 0.007069, 0.8465),
    ('AUROC', 'full_minus_combined_metadata'):
        (0.001372, 0.004917, 0.9995),
    ('AUROC', 'full_minus_no_star'):
        (0.083730, 0.092304, 1.0000),
    ('AUROC', 'no_star_minus_review_stars'):
        (-0.092834, -0.077934, 0.0000),
}

# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256(path, chunk=1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open('rb') as handle:
        while True:
            block = handle.read(chunk)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def sidecar_hash(path):
    text = Path(path).read_text(encoding='utf-8')
    matches = re.findall(r'\b[a-fA-F0-9]{64}\b', text)

    if not matches:
        raise ValueError(f'No SHA-256 in sidecar: {path}')

    return matches[0].lower()


def write_sidecar(path):
    path = Path(path)
    output = Path(str(path) + '.sha256')

    output.write_text(
        f'{sha256(path)}  {path.name}\n',
        encoding='utf-8',
    )

    return output


def native(value):
    if isinstance(value, dict):
        return {
            str(key): native(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [native(item) for item in value]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, Path):
        return str(value)

    return value


def write_json(payload, path):
    temporary = Path(str(path) + '.tmp')

    temporary.write_text(
        json.dumps(
            native(payload),
            indent=2,
            sort_keys=True,
        )
        + '\n',
        encoding='utf-8',
    )

    os.replace(temporary, path)


def write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')

    frame.to_csv(
        temporary,
        index=False,
        lineterminator='\n',
        float_format='%.17g',
    )

    os.replace(temporary, path)


def write_parquet(frame, path):
    temporary = Path(str(path) + '.tmp.parquet')

    frame.to_parquet(
        temporary,
        index=False,
        engine='pyarrow',
        compression='zstd',
    )

    os.replace(temporary, path)


def artifact(path, sidecar, role):
    return {
        'role': role,
        'path': str(path),
        'sha256': sha256(path),
        'bytes': int(Path(path).stat().st_size),
        'sidecar_path': str(sidecar),
        'sidecar_sha256': sha256(sidecar),
    }


def ci(values):
    low, high = np.percentile(
        np.asarray(values, dtype=float),
        [2.5, 97.5],
    )

    return float(low), float(high)


def close(observed, expected, tolerance=2.5e-6):
    return bool(
        np.isfinite(observed)
        and abs(observed - expected) <= tolerance
    )

# --------------------------------------------------------------------------------------------------
# 3. FRESH INPUT VERIFICATION
# --------------------------------------------------------------------------------------------------

for path in [EVAL, EVAL_SHA]:
    if not path.exists():
        raise FileNotFoundError(
            f'Missing frozen artifact:\n{path}'
        )

observed_hash = sha256(EVAL)

assert observed_hash == EXPECTED_HASH, (
    f'Evaluable SHA-256 mismatch: {observed_hash}'
)

assert sidecar_hash(EVAL_SHA) == observed_hash, (
    'Evaluable sidecar mismatch.'
)

parquet_file = pq.ParquetFile(EVAL)

assert (
    parquet_file.metadata.num_rows,
    parquet_file.metadata.num_columns,
) == (
    EXPECTED_ROWS,
    EXPECTED_COLS,
), 'Unexpected Stage 6B dimensions.'

required_columns = (
    KEYS
    + [OUTCOME]
    + [value[0] for value in SCORES.values()]
)

missing_columns = [
    column
    for column in required_columns
    if column not in parquet_file.schema_arrow.names
]

assert not missing_columns, (
    f'Missing frozen columns: {missing_columns}'
)

df = pd.read_parquet(
    EVAL,
    columns=required_columns,
).copy()

assert len(df) == EXPECTED_ROWS

assert df['rcv_accession'].notna().all()

assert (
    df['rcv_accession']
    .astype(str)
    .str.strip()
    .ne('')
    .all()
)

assert (
    df['rcv_accession'].nunique(dropna=False)
    == EXPECTED_ROWS
)

row_order = pd.to_numeric(
    df['t0_row_order'],
    errors='raise',
).to_numpy()

assert len(np.unique(row_order)) == EXPECTED_ROWS
assert np.all(np.diff(row_order) > 0)

df[OUTCOME] = pd.to_numeric(
    df[OUTCOME],
    errors='raise',
).astype('int8')

y = df[OUTCOME].to_numpy(dtype='int8')

assert set(np.unique(y)) == {0, 1}

events = int(y.sum())
negatives = int((y == 0).sum())

assert (
    events,
    negatives,
) == (
    EXPECTED_EVENTS,
    EXPECTED_NEGATIVES,
)

prevalence = float(y.mean())

score_arrays = OrderedDict()

for model_key, (column, _) in SCORES.items():
    values = pd.to_numeric(
        df[column],
        errors='raise',
    ).to_numpy(dtype=float)

    assert np.isfinite(values).all(), (
        f'Nonfinite values in {column}'
    )

    assert (
        (values >= 0)
        & (values <= 1)
    ).all(), (
        f'Out-of-range values in {column}'
    )

    assert np.unique(values).size >= 2, (
        f'Constant score: {column}'
    )

    score_arrays[model_key] = values

# --------------------------------------------------------------------------------------------------
# 4. LOCKED POINT ESTIMATES
# --------------------------------------------------------------------------------------------------

point = {
    'AUPRC': OrderedDict(),
    'AUROC': OrderedDict(),
}

for model_key, values in score_arrays.items():
    point['AUPRC'][model_key] = float(
        average_precision_score(y, values)
    )

    point['AUROC'][model_key] = float(
        roc_auc_score(y, values)
    )

for (metric, model_key), expected in EXPECTED_POINT.items():
    observed = point[metric][model_key]

    assert close(
        observed,
        expected,
        tolerance=1.5e-6,
    ), (
        f'Point estimate mismatch: {metric} {model_key}: '
        f'{observed:.12f} versus recorded {expected:.6f}'
    )

# --------------------------------------------------------------------------------------------------
# 5. REPRODUCE THE 2,000 IDENTICAL PAIRED ROW-BOOTSTRAP SAMPLES
# --------------------------------------------------------------------------------------------------

model_keys = list(SCORES)


def one_bootstrap(replicate_number, indexes):
    y_bootstrap = y[indexes]

    if np.unique(y_bootstrap).size != 2:
        return {
            'replicate': replicate_number,
            'valid_two_class_replicate': False,
        }

    row = {
        'replicate': replicate_number,
        'valid_two_class_replicate': True,
    }

    for model_key in model_keys:
        score_bootstrap = score_arrays[model_key][indexes]

        row[f'auprc_{model_key}'] = float(
            average_precision_score(
                y_bootstrap,
                score_bootstrap,
            )
        )

        row[f'auroc_{model_key}'] = float(
            roc_auc_score(
                y_bootstrap,
                score_bootstrap,
            )
        )

    return row


print(
    f'Running {BOOTSTRAPS:,} paired bootstrap replicates '
    f'with seed {SEED}...'
)

bootstrap_start = time.perf_counter()
rng = np.random.default_rng(SEED)

bootstrap_rows = Parallel(
    n_jobs=N_JOBS,
    prefer='threads',
    batch_size=5,
    pre_dispatch='2*n_jobs',
    verbose=10,
)(
    delayed(one_bootstrap)(
        replicate_number,
        rng.integers(
            0,
            EXPECTED_ROWS,
            EXPECTED_ROWS,
            dtype=np.int64,
        ),
    )
    for replicate_number in range(1, BOOTSTRAPS + 1)
)

elapsed = float(
    time.perf_counter() - bootstrap_start
)

replicates = (
    pd.DataFrame(bootstrap_rows)
    .sort_values(
        'replicate',
        kind='mergesort',
    )
    .reset_index(drop=True)
)

assert replicates['replicate'].tolist() == list(
    range(1, BOOTSTRAPS + 1)
)

valid_mask = (
    replicates['valid_two_class_replicate']
    .astype(bool)
)

valid_replicates = int(valid_mask.sum())
invalid_replicates = int((~valid_mask).sum())

assert (
    valid_replicates,
    invalid_replicates,
) == (
    BOOTSTRAPS,
    0,
)

metric_columns = [
    f'{metric}_{model_key}'
    for metric in ['auprc', 'auroc']
    for model_key in model_keys
]

assert (
    replicates[metric_columns]
    .notna()
    .all()
    .all()
)

assert np.isfinite(
    replicates[metric_columns].to_numpy(dtype=float)
).all()

# --------------------------------------------------------------------------------------------------
# 6. SUMMARY AND PAIRED TABLES
# --------------------------------------------------------------------------------------------------

summary_rows = []

for metric in ['AUPRC', 'AUROC']:
    null_reference = (
        prevalence
        if metric == 'AUPRC'
        else 0.5
    )

    for model_key, (column, label) in SCORES.items():
        values = replicates.loc[
            valid_mask,
            f'{metric.lower()}_{model_key}',
        ].to_numpy(dtype=float)

        ci_low, ci_high = ci(values)

        summary_rows.append({
            'metric': metric,
            'model_key': model_key,
            'model': label,
            'score_column': column,
            'point_estimate': point[metric][model_key],
            'bootstrap_mean': float(values.mean()),
            'bootstrap_standard_error': float(
                values.std(ddof=1)
            ),
            'percentile_95_ci_low': ci_low,
            'percentile_95_ci_high': ci_high,
            'valid_replicates': valid_replicates,
            'invalid_replicates': invalid_replicates,
            'null_reference': null_reference,
            'point_minus_null': (
                point[metric][model_key]
                - null_reference
            ),
            'bootstrap_seed': SEED,
            'bootstrap_attempts': BOOTSTRAPS,
        })

summary = pd.DataFrame(summary_rows)

paired_rows = []

for metric in ['AUPRC', 'AUROC']:
    for comparison_key, (
        minuend_key,
        subtrahend_key,
    ) in COMPARISONS.items():

        differences = (
            replicates.loc[
                valid_mask,
                f'{metric.lower()}_{minuend_key}',
            ].to_numpy(dtype=float)
            -
            replicates.loc[
                valid_mask,
                f'{metric.lower()}_{subtrahend_key}',
            ].to_numpy(dtype=float)
        )

        ci_low, ci_high = ci(differences)

        paired_rows.append({
            'metric': metric,
            'comparison_key': comparison_key,
            'comparison': (
                f'{SCORES[minuend_key][1]} minus '
                f'{SCORES[subtrahend_key][1]}'
            ),
            'minuend_model_key': minuend_key,
            'subtrahend_model_key': subtrahend_key,
            'point_difference': (
                point[metric][minuend_key]
                - point[metric][subtrahend_key]
            ),
            'bootstrap_mean_difference': float(
                differences.mean()
            ),
            'bootstrap_standard_error': float(
                differences.std(ddof=1)
            ),
            'percentile_95_ci_low': ci_low,
            'percentile_95_ci_high': ci_high,
            'fraction_bootstrap_difference_gt_zero': float(
                np.mean(differences > 0)
            ),
            'fraction_bootstrap_difference_lt_zero': float(
                np.mean(differences < 0)
            ),
            'ci_excludes_zero': bool(
                ci_low > 0
                or ci_high < 0
            ),
            'interval_direction': (
                'positive'
                if ci_low > 0
                else (
                    'negative'
                    if ci_high < 0
                    else 'includes_zero'
                )
            ),
            'valid_replicates': valid_replicates,
            'invalid_replicates': invalid_replicates,
            'multiplicity_family': (
                f'principal_{metric.lower()}_comparisons'
            ),
            'multiplicity_adjustment': (
                'none_prespecified_principal_comparisons'
            ),
            'bootstrap_seed': SEED,
            'bootstrap_attempts': BOOTSTRAPS,
        })

paired = pd.DataFrame(paired_rows)

assert len(summary) == 8
assert len(paired) == 8

# Reproduce recorded bootstrap summaries before writing.

for (metric, model_key), expected in EXPECTED_SUMMARY.items():
    row = summary.loc[
        summary['metric'].eq(metric)
        & summary['model_key'].eq(model_key)
    ].iloc[0]

    observed = [
        row['bootstrap_mean'],
        row['bootstrap_standard_error'],
        row['percentile_95_ci_low'],
        row['percentile_95_ci_high'],
    ]

    assert all(
        close(observed_value, expected_value)
        for observed_value, expected_value
        in zip(observed, expected)
    ), (
        f'Summary mismatch: {metric} {model_key}'
    )

for (
    metric,
    comparison_key,
), expected in EXPECTED_PAIRED.items():

    row = paired.loc[
        paired['metric'].eq(metric)
        & paired['comparison_key'].eq(comparison_key)
    ].iloc[0]

    observed = [
        row['percentile_95_ci_low'],
        row['percentile_95_ci_high'],
        row['fraction_bootstrap_difference_gt_zero'],
    ]

    assert all(
        close(observed_value, expected_value)
        for observed_value, expected_value
        in zip(observed, expected)
    ), (
        f'Paired mismatch: {metric} {comparison_key}'
    )

# --------------------------------------------------------------------------------------------------
# 7. WRITE, SIDECAR, READ BACK, AND VERIFY
# --------------------------------------------------------------------------------------------------

write_parquet(replicates, REPS_FILE)
write_csv(summary, SUMMARY_FILE)
write_csv(paired, PAIRED_FILE)

reps_sidecar = write_sidecar(REPS_FILE)
summary_sidecar = write_sidecar(SUMMARY_FILE)
paired_sidecar = write_sidecar(PAIRED_FILE)

fresh_replicates = pd.read_parquet(REPS_FILE)
fresh_summary = pd.read_csv(SUMMARY_FILE)
fresh_paired = pd.read_csv(PAIRED_FILE)

pd.testing.assert_frame_equal(
    fresh_replicates,
    replicates,
    check_exact=True,
)

pd.testing.assert_frame_equal(
    fresh_summary,
    summary,
    check_dtype=False,
    check_exact=False,
    rtol=1e-13,
    atol=1e-15,
)

pd.testing.assert_frame_equal(
    fresh_paired,
    paired,
    check_dtype=False,
    check_exact=False,
    rtol=1e-13,
    atol=1e-15,
)

for path, sidecar in [
    (REPS_FILE, reps_sidecar),
    (SUMMARY_FILE, summary_sidecar),
    (PAIRED_FILE, paired_sidecar),
]:
    assert sidecar_hash(sidecar) == sha256(path), (
        f'Sidecar mismatch: {path}'
    )

# --------------------------------------------------------------------------------------------------
# 8. QC AND MANIFEST
# --------------------------------------------------------------------------------------------------

checks = OrderedDict([
    ('stage6b_hash_and_sidecar_verified', True),
    ('stage6b_dimensions_verified', True),
    ('keys_and_row_order_verified', True),
    ('outcome_accounting_verified', True),
    ('scores_finite_in_range_and_nonconstant', True),
    ('point_estimates_reproduced', True),
    (
        'all_2000_paired_bootstrap_replicates_valid',
        (
            valid_replicates == BOOTSTRAPS
            and invalid_replicates == 0
        ),
    ),
    ('bootstrap_metrics_complete_and_finite', True),
    ('eight_model_interval_rows_created', len(summary) == 8),
    ('eight_paired_comparison_rows_created', len(paired) == 8),
    ('recorded_bootstrap_summaries_reproduced', True),
    ('recorded_paired_summaries_reproduced', True),
    ('three_result_tables_read_back_semantically', True),
    ('three_result_table_sidecars_verified', True),
    ('no_frozen_scientific_input_modified', True),
    ('experiment_2_not_started', True),
])

assert all(checks.values())

qc_payload = {
    'schema_version': '1.0',
    'cell': '6C-4C0',
    'stage': 'Stage 6C Step 4C',
    'notebook_filename': NOTEBOOK_FILENAME,
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'checks_total': len(checks),
    'checks_passed': int(sum(checks.values())),
    'checks_failed': 0,
    'checks': checks,
    'source': {
        'path': str(EVAL),
        'sha256': observed_hash,
        'rows': EXPECTED_ROWS,
        'columns': EXPECTED_COLS,
        'events': events,
        'negatives': negatives,
        'prevalence': prevalence,
    },
    'bootstrap': {
        'method': 'nonparametric_paired_row_bootstrap',
        'attempts': BOOTSTRAPS,
        'valid_replicates': valid_replicates,
        'invalid_replicates': invalid_replicates,
        'seed': SEED,
        'n_jobs': N_JOBS,
        'elapsed_seconds': elapsed,
        'identical_resamples_across_models_and_metrics': True,
    },
    'scientific_boundary': {
        'scores_modified': False,
        'outcomes_modified': False,
        'thresholds_modified': False,
        'models_modified': False,
        'features_modified': False,
        'linkage_decisions_modified': False,
        'censoring_policy_modified': False,
        'cohort_membership_modified': False,
        'experiment_2_started': False,
    },
    'software_versions': {
        'python': sys.version.split()[0],
        'platform': platform.platform(),
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'pyarrow': pyarrow.__version__,
        'scikit_learn': sklearn.__version__,
        'joblib': joblib.__version__,
    },
}

write_json(qc_payload, QC_FILE)
qc_sidecar = write_sidecar(QC_FILE)

artifacts = [
    artifact(
        REPS_FILE,
        reps_sidecar,
        'principal paired bootstrap replicates',
    ),
    artifact(
        SUMMARY_FILE,
        summary_sidecar,
        'principal model bootstrap intervals',
    ),
    artifact(
        PAIRED_FILE,
        paired_sidecar,
        'principal paired model comparisons',
    ),
    artifact(
        QC_FILE,
        qc_sidecar,
        'Cell 6C-4C0 QC',
    ),
]

manifest_payload = {
    'schema_version': '1.0',
    'cell': '6C-4C0',
    'stage': 'Stage 6C Step 4C',
    'notebook_filename': NOTEBOOK_FILENAME,
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'package_name': PKG,
    'required_final_freeze_category': (
        'principal_auprc_auroc_bootstrap'
    ),
    'source_artifact': {
        'path': str(EVAL),
        'sha256': observed_hash,
        'sidecar_path': str(EVAL_SHA),
        'rows': EXPECTED_ROWS,
        'columns': EXPECTED_COLS,
    },
    'analysis': {
        'outcome_column': OUTCOME,
        'score_direction': (
            'higher_is_more_instability_risk'
        ),
        'scores': {
            model_key: {
                'column': specification[0],
                'display_name': specification[1],
            }
            for model_key, specification
            in SCORES.items()
        },
        'comparisons': {
            comparison_key: {
                'minuend': comparison[0],
                'subtrahend': comparison[1],
            }
            for comparison_key, comparison
            in COMPARISONS.items()
        },
        'bootstrap_method': (
            'nonparametric_paired_row_bootstrap'
        ),
        'bootstrap_attempts': BOOTSTRAPS,
        'seed': SEED,
        'confidence_interval': (
            '2.5th_to_97.5th_percentile'
        ),
        'multiplicity_adjustment': (
            'none_prespecified_principal_comparisons'
        ),
    },
    'artifacts': artifacts,
    'decision': (
        'PASS_STAGE6C_PRINCIPAL_BOOTSTRAP_RESULTS_'
        'MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED'
    ),
    'scientific_boundary': {
        'frozen_inputs_modified': False,
        'experiment_2_started': False,
        'final_integrated_stage6c_freeze_completed': False,
    },
}

write_json(manifest_payload, MANIFEST_FILE)
manifest_sidecar = write_sidecar(MANIFEST_FILE)

# Final manifest/QC and artifact reverification.

fresh_qc = json.loads(
    QC_FILE.read_text(encoding='utf-8')
)

fresh_manifest = json.loads(
    MANIFEST_FILE.read_text(encoding='utf-8')
)

assert fresh_qc['cell'] == '6C-4C0'
assert fresh_qc['checks_failed'] == 0

assert (
    fresh_manifest['required_final_freeze_category']
    == 'principal_auprc_auroc_bootstrap'
)

assert sidecar_hash(qc_sidecar) == sha256(QC_FILE)

assert (
    sidecar_hash(manifest_sidecar)
    == sha256(MANIFEST_FILE)
)

for entry in fresh_manifest['artifacts']:
    artifact_path = Path(entry['path'])
    artifact_sidecar = Path(entry['sidecar_path'])

    assert sha256(artifact_path) == entry['sha256']

    assert (
        sidecar_hash(artifact_sidecar)
        == entry['sha256']
    )

# --------------------------------------------------------------------------------------------------
# 9. DISPLAY
# --------------------------------------------------------------------------------------------------

separator = '=' * 150

print('\n' + separator)

print(
    'STAGE 6C STEP 4C — CELL 6C-4C0 — '
    'PRINCIPAL AUPRC/AUROC BOOTSTRAP MATERIALIZATION'
)

print(separator)

print(
    f'Notebook file name                 : '
    f'{NOTEBOOK_FILENAME}'
)

print(
    f'Stage 6B evaluable SHA-256         : '
    f'PASS ({observed_hash})'
)

print(
    f'Stage 6B dimensions                : '
    f'PASS ({EXPECTED_ROWS:,} × {EXPECTED_COLS})'
)

print(
    f'Outcome accounting                 : '
    f'PASS ({events:,} events; {negatives:,} negatives)'
)

print(
    f'Bootstrap attempts / valid         : '
    f'{BOOTSTRAPS:,} / {valid_replicates:,}'
)

print(
    f'Bootstrap elapsed                  : '
    f'{elapsed / 60:.2f} minutes'
)

print(
    f'Replicate table                    : '
    f'{REPS_FILE}'
)

print(
    f'Model intervals                    : '
    f'{SUMMARY_FILE}'
)

print(
    f'Paired comparisons                 : '
    f'{PAIRED_FILE}'
)

print(
    f'QC                                 : '
    f'{QC_FILE}'
)

print(
    f'Manifest                           : '
    f'{MANIFEST_FILE}'
)

print(
    f'Fresh QC                           : '
    f'PASS ({fresh_qc["checks_passed"]}/'
    f'{fresh_qc["checks_total"]})'
)

print('\nPRINCIPAL MODEL INTERVALS')

print(
    summary[
        [
            'metric',
            'model',
            'point_estimate',
            'bootstrap_mean',
            'bootstrap_standard_error',
            'percentile_95_ci_low',
            'percentile_95_ci_high',
            'valid_replicates',
        ]
    ].to_string(index=False)
)

print('\nPAIRED COMPARISONS')

print(
    paired[
        [
            'metric',
            'comparison',
            'point_difference',
            'percentile_95_ci_low',
            'percentile_95_ci_high',
            'fraction_bootstrap_difference_gt_zero',
            'ci_excludes_zero',
        ]
    ].to_string(index=False)
)

print('\nCELL DECISION')

print(
    'PASS_STAGE6C_PRINCIPAL_BOOTSTRAP_RESULTS_'
    'MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED'
)

print(
    'Seven independent result categories remain before the '
    'final integrated Stage 6C freeze. '
    'Experiment 2 has not started.'
)

print(separator)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Use this Colab notebook file name: GES_Stage6C_Cell_6C_4C0_Principal_Bootstrap_Materialization.ipynb
Running 2,000 paired bootstrap replicates with seed 42...


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   9 tasks      | elapsed:    1.1s
[Parallel(n_jobs=2)]: Done  24 tasks      | elapsed:    2.0s
[Parallel(n_jobs=2)]: Done  49 tasks      | elapsed:    4.4s
[Parallel(n_jobs=2)]: Done  74 tasks      | elapsed:    6.1s
[Parallel(n_jobs=2)]: Done 109 tasks      | elapsed:    9.6s
[Parallel(n_jobs=2)]: Done 144 tasks      | elapsed:   14.0s
[Parallel(n_jobs=2)]: Done 189 tasks      | elapsed:   18.1s
[Parallel(n_jobs=2)]: Done 234 tasks      | elapsed:   21.4s
[Parallel(n_jobs=2)]: Done 289 tasks      | elapsed:   28.3s
[Parallel(n_jobs=2)]: Done 344 tasks      | elapsed:   32.6s
[Parallel(n_jobs=2)]: Done 409 tasks      | elapsed:   38.3s
[Parallel(n_jobs=2)]: Done 474 tasks      | elapsed:   45.5s
[Parallel(n_jobs=2)]: Done 549 tasks      | elapsed:   52.1s
[Parallel(n_jobs=2)]: Done 624 tasks      | elapsed:  1.0min
[Parallel(n_jobs=2)]: Done 709 tasks      | elapsed:  1.1min
[Para

AssertionError: Summary mismatch: AUPRC full_ges

In [6]:
# ==============================================================================================
# CELL 6C-4C0 DIAGNOSTIC
# Inspect the completed bootstrap results without rerunning or writing artifacts
# ==============================================================================================

print("=" * 140)
print("CELL 6C-4C0 — BOOTSTRAP REPRODUCTION DIAGNOSTIC")
print("=" * 140)

print("\nRandom-number generator used:")
print("Generator object:", type(rng).__name__)

if hasattr(rng, "bit_generator"):
    print("Bit generator   :", type(rng.bit_generator).__name__)

print("\nObserved model bootstrap summaries versus recorded reference values:")

diagnostic_summary_rows = []

for (metric, model_key), expected_values in EXPECTED_SUMMARY.items():
    observed_row = summary.loc[
        summary["metric"].eq(metric)
        & summary["model_key"].eq(model_key)
    ].iloc[0]

    field_names = [
        "bootstrap_mean",
        "bootstrap_standard_error",
        "percentile_95_ci_low",
        "percentile_95_ci_high",
    ]

    for field_name, expected_value in zip(field_names, expected_values):
        observed_value = float(observed_row[field_name])

        diagnostic_summary_rows.append({
            "metric": metric,
            "model_key": model_key,
            "field": field_name,
            "observed": observed_value,
            "recorded_expected": expected_value,
            "difference": observed_value - expected_value,
            "absolute_difference": abs(observed_value - expected_value),
        })

diagnostic_summary = pd.DataFrame(diagnostic_summary_rows)

print(
    diagnostic_summary.to_string(
        index=False,
        float_format=lambda value: f"{value:.12f}",
    )
)

print("\nObserved paired-comparison summaries versus recorded reference values:")

diagnostic_paired_rows = []

for (metric, comparison_key), expected_values in EXPECTED_PAIRED.items():
    observed_row = paired.loc[
        paired["metric"].eq(metric)
        & paired["comparison_key"].eq(comparison_key)
    ].iloc[0]

    field_names = [
        "percentile_95_ci_low",
        "percentile_95_ci_high",
        "fraction_bootstrap_difference_gt_zero",
    ]

    for field_name, expected_value in zip(field_names, expected_values):
        observed_value = float(observed_row[field_name])

        diagnostic_paired_rows.append({
            "metric": metric,
            "comparison_key": comparison_key,
            "field": field_name,
            "observed": observed_value,
            "recorded_expected": expected_value,
            "difference": observed_value - expected_value,
            "absolute_difference": abs(observed_value - expected_value),
        })

diagnostic_paired = pd.DataFrame(diagnostic_paired_rows)

print(
    diagnostic_paired.to_string(
        index=False,
        float_format=lambda value: f"{value:.12f}",
    )
)

print("\nCompleted bootstrap accounting:")
print("Attempts             :", BOOTSTRAPS)
print("Valid replicates     :", valid_replicates)
print("Invalid replicates   :", invalid_replicates)
print("Replicate table rows :", len(replicates))

print("\nFirst five completed bootstrap rows:")
print(
    replicates.head(5).to_string(
        index=False,
        float_format=lambda value: f"{value:.12f}",
    )
)

print("\nMaximum absolute discrepancy by section:")
print(
    "Model summaries :",
    f"{diagnostic_summary['absolute_difference'].max():.12f}",
)

print(
    "Paired results  :",
    f"{diagnostic_paired['absolute_difference'].max():.12f}",
)

print("\nNo artifacts were written by this diagnostic.")
print("=" * 140)

CELL 6C-4C0 — BOOTSTRAP REPRODUCTION DIAGNOSTIC

Random-number generator used:
Generator object: Generator
Bit generator   : PCG64

Observed model bootstrap summaries versus recorded reference values:
metric         model_key                    field       observed  recorded_expected      difference  absolute_difference
 AUPRC          full_ges           bootstrap_mean 0.112617553699     0.112802000000 -0.000184446301       0.000184446301
 AUPRC          full_ges bootstrap_standard_error 0.002366795041     0.002362000000  0.000004795041       0.000004795041
 AUPRC          full_ges     percentile_95_ci_low 0.108142570274     0.108261000000 -0.000118429726       0.000118429726
 AUPRC          full_ges    percentile_95_ci_high 0.117321664475     0.117357000000 -0.000035335525       0.000035335525
 AUPRC       no_star_ges           bootstrap_mean 0.096612936513     0.096785000000 -0.000172063487       0.000172063487
 AUPRC       no_star_ges bootstrap_standard_error 0.002139077710     0.00

In [7]:
# ==================================================================================================
# CELL 6C-4C0 — RNG-SEQUENCE CORRECTION
# Reproduce the original Cells 6C-1C/6C-1D bootstrap using legacy NumPy RandomState/MT19937
#
# IMPORTANT:
# - Do not rerun the complete earlier notebook cell.
# - This cell writes no files.
# - Existing replicates/summary/paired objects are replaced only if all reproduction checks pass.
# ==================================================================================================

from collections import OrderedDict
import time

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.metrics import average_precision_score, roc_auc_score

print("=" * 145)
print("CELL 6C-4C0 — LEGACY MT19937 BOOTSTRAP-SEQUENCE REPRODUCTION")
print("=" * 145)

# Confirm that all objects created by the earlier cell remain available.
required_objects = [
    "EXPECTED_ROWS",
    "BOOTSTRAPS",
    "SEED",
    "N_JOBS",
    "y",
    "score_arrays",
    "SCORES",
    "COMPARISONS",
    "point",
    "prevalence",
    "EXPECTED_SUMMARY",
    "EXPECTED_PAIRED",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required objects are missing from the current runtime: "
        + ", ".join(missing_objects)
        + ". Do not reset the runtime; rerun the original setup cell first."
    )

model_keys = list(SCORES.keys())


def legacy_bootstrap_replicate(replicate_number, indexes):
    """
    Calculate AUPRC and AUROC for all four principal models using
    one identical paired bootstrap row sample.
    """

    y_bootstrap = y[indexes]

    if np.unique(y_bootstrap).size != 2:
        return {
            "replicate": int(replicate_number),
            "valid_two_class_replicate": False,
        }

    result = {
        "replicate": int(replicate_number),
        "valid_two_class_replicate": True,
    }

    for model_key in model_keys:
        score_bootstrap = score_arrays[model_key][indexes]

        result[f"auprc_{model_key}"] = float(
            average_precision_score(
                y_bootstrap,
                score_bootstrap,
            )
        )

        result[f"auroc_{model_key}"] = float(
            roc_auc_score(
                y_bootstrap,
                score_bootstrap,
            )
        )

    return result


# Legacy NumPy RNG:
# np.random.seed(...) and np.random.choice(...) use this MT19937 sequence.
legacy_rng = np.random.RandomState(SEED)

print("\nBootstrap configuration")
print("Random API           : numpy.random.RandomState")
print("Bit generator        : MT19937")
print("Random seed          :", SEED)
print("Bootstrap attempts   :", f"{BOOTSTRAPS:,}")
print("Rows per replicate   :", f"{EXPECTED_ROWS:,}")
print("Parallel workers     :", N_JOBS)
print("Sampling method      : choice(..., replace=True)")
print("Files written        : No")

bootstrap_start = time.perf_counter()

legacy_bootstrap_rows = Parallel(
    n_jobs=N_JOBS,
    prefer="threads",
    batch_size=5,
    pre_dispatch="2*n_jobs",
    verbose=10,
)(
    delayed(legacy_bootstrap_replicate)(
        replicate_number,
        legacy_rng.choice(
            EXPECTED_ROWS,
            size=EXPECTED_ROWS,
            replace=True,
        ),
    )
    for replicate_number in range(1, BOOTSTRAPS + 1)
)

legacy_elapsed = float(
    time.perf_counter() - bootstrap_start
)

legacy_replicates = (
    pd.DataFrame(legacy_bootstrap_rows)
    .sort_values(
        "replicate",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

assert legacy_replicates["replicate"].tolist() == list(
    range(1, BOOTSTRAPS + 1)
), "Bootstrap replicate ordering is invalid."

legacy_valid_mask = (
    legacy_replicates["valid_two_class_replicate"]
    .astype(bool)
)

legacy_valid_replicates = int(
    legacy_valid_mask.sum()
)

legacy_invalid_replicates = int(
    (~legacy_valid_mask).sum()
)

assert legacy_valid_replicates == BOOTSTRAPS, (
    f"Expected {BOOTSTRAPS:,} valid replicates, "
    f"but found {legacy_valid_replicates:,}."
)

assert legacy_invalid_replicates == 0, (
    f"Unexpected invalid replicates: "
    f"{legacy_invalid_replicates:,}"
)

metric_columns = [
    f"{metric}_{model_key}"
    for metric in ["auprc", "auroc"]
    for model_key in model_keys
]

assert (
    legacy_replicates[metric_columns]
    .notna()
    .all()
    .all()
), "Bootstrap metric table contains missing values."

assert np.isfinite(
    legacy_replicates[metric_columns]
    .to_numpy(dtype=float)
).all(), "Bootstrap metric table contains nonfinite values."


def percentile_interval(values):
    low, high = np.percentile(
        np.asarray(values, dtype=float),
        [2.5, 97.5],
    )

    return float(low), float(high)


# --------------------------------------------------------------------------------------------------
# Reconstruct model bootstrap summary table
# --------------------------------------------------------------------------------------------------

legacy_summary_rows = []

for metric in ["AUPRC", "AUROC"]:
    null_reference = (
        prevalence
        if metric == "AUPRC"
        else 0.5
    )

    for model_key, (score_column, model_label) in SCORES.items():
        values = legacy_replicates.loc[
            legacy_valid_mask,
            f"{metric.lower()}_{model_key}",
        ].to_numpy(dtype=float)

        ci_low, ci_high = percentile_interval(values)

        legacy_summary_rows.append({
            "metric": metric,
            "model_key": model_key,
            "model": model_label,
            "score_column": score_column,
            "point_estimate": float(
                point[metric][model_key]
            ),
            "bootstrap_mean": float(
                values.mean()
            ),
            "bootstrap_standard_error": float(
                values.std(ddof=1)
            ),
            "percentile_95_ci_low": ci_low,
            "percentile_95_ci_high": ci_high,
            "valid_replicates": legacy_valid_replicates,
            "invalid_replicates": legacy_invalid_replicates,
            "null_reference": float(null_reference),
            "point_minus_null": float(
                point[metric][model_key]
                - null_reference
            ),
            "bootstrap_seed": SEED,
            "bootstrap_attempts": BOOTSTRAPS,
            "rng_api": "numpy.random.RandomState",
            "rng_engine": "MT19937",
        })

legacy_summary = pd.DataFrame(
    legacy_summary_rows
)


# --------------------------------------------------------------------------------------------------
# Reconstruct paired-comparison table
# --------------------------------------------------------------------------------------------------

legacy_paired_rows = []

for metric in ["AUPRC", "AUROC"]:
    for comparison_key, (
        minuend_key,
        subtrahend_key,
    ) in COMPARISONS.items():

        differences = (
            legacy_replicates.loc[
                legacy_valid_mask,
                f"{metric.lower()}_{minuend_key}",
            ].to_numpy(dtype=float)
            -
            legacy_replicates.loc[
                legacy_valid_mask,
                f"{metric.lower()}_{subtrahend_key}",
            ].to_numpy(dtype=float)
        )

        ci_low, ci_high = percentile_interval(
            differences
        )

        legacy_paired_rows.append({
            "metric": metric,
            "comparison_key": comparison_key,
            "comparison": (
                f"{SCORES[minuend_key][1]} minus "
                f"{SCORES[subtrahend_key][1]}"
            ),
            "minuend_model_key": minuend_key,
            "subtrahend_model_key": subtrahend_key,
            "point_difference": float(
                point[metric][minuend_key]
                - point[metric][subtrahend_key]
            ),
            "bootstrap_mean_difference": float(
                differences.mean()
            ),
            "bootstrap_standard_error": float(
                differences.std(ddof=1)
            ),
            "percentile_95_ci_low": ci_low,
            "percentile_95_ci_high": ci_high,
            "fraction_bootstrap_difference_gt_zero": float(
                np.mean(differences > 0)
            ),
            "fraction_bootstrap_difference_lt_zero": float(
                np.mean(differences < 0)
            ),
            "ci_excludes_zero": bool(
                ci_low > 0
                or ci_high < 0
            ),
            "interval_direction": (
                "positive"
                if ci_low > 0
                else (
                    "negative"
                    if ci_high < 0
                    else "includes_zero"
                )
            ),
            "valid_replicates": legacy_valid_replicates,
            "invalid_replicates": legacy_invalid_replicates,
            "multiplicity_family": (
                f"principal_{metric.lower()}_comparisons"
            ),
            "multiplicity_adjustment": (
                "none_prespecified_principal_comparisons"
            ),
            "bootstrap_seed": SEED,
            "bootstrap_attempts": BOOTSTRAPS,
            "rng_api": "numpy.random.RandomState",
            "rng_engine": "MT19937",
        })

legacy_paired = pd.DataFrame(
    legacy_paired_rows
)

assert len(legacy_summary) == 8
assert len(legacy_paired) == 8


# --------------------------------------------------------------------------------------------------
# Compare against the values recorded in the technical report
# --------------------------------------------------------------------------------------------------

def rounded_value_match(
    observed,
    expected,
    tolerance=2.5e-6,
):
    return bool(
        np.isfinite(observed)
        and abs(float(observed) - float(expected))
        <= tolerance
    )


summary_check_rows = []

for (
    metric,
    model_key,
), expected_values in EXPECTED_SUMMARY.items():

    observed_row = legacy_summary.loc[
        legacy_summary["metric"].eq(metric)
        & legacy_summary["model_key"].eq(model_key)
    ].iloc[0]

    field_names = [
        "bootstrap_mean",
        "bootstrap_standard_error",
        "percentile_95_ci_low",
        "percentile_95_ci_high",
    ]

    for field_name, expected_value in zip(
        field_names,
        expected_values,
    ):
        observed_value = float(
            observed_row[field_name]
        )

        summary_check_rows.append({
            "metric": metric,
            "model_key": model_key,
            "field": field_name,
            "observed": observed_value,
            "recorded_expected": expected_value,
            "difference": (
                observed_value
                - expected_value
            ),
            "match": rounded_value_match(
                observed_value,
                expected_value,
            ),
        })

summary_checks = pd.DataFrame(
    summary_check_rows
)


paired_check_rows = []

for (
    metric,
    comparison_key,
), expected_values in EXPECTED_PAIRED.items():

    observed_row = legacy_paired.loc[
        legacy_paired["metric"].eq(metric)
        & legacy_paired["comparison_key"].eq(
            comparison_key
        )
    ].iloc[0]

    field_names = [
        "percentile_95_ci_low",
        "percentile_95_ci_high",
        "fraction_bootstrap_difference_gt_zero",
    ]

    for field_name, expected_value in zip(
        field_names,
        expected_values,
    ):
        observed_value = float(
            observed_row[field_name]
        )

        paired_check_rows.append({
            "metric": metric,
            "comparison_key": comparison_key,
            "field": field_name,
            "observed": observed_value,
            "recorded_expected": expected_value,
            "difference": (
                observed_value
                - expected_value
            ),
            "match": rounded_value_match(
                observed_value,
                expected_value,
            ),
        })

paired_checks = pd.DataFrame(
    paired_check_rows
)

all_summary_checks_passed = bool(
    summary_checks["match"].all()
)

all_paired_checks_passed = bool(
    paired_checks["match"].all()
)

print("\n" + "-" * 145)
print("MODEL BOOTSTRAP SUMMARY REPRODUCTION")
print("-" * 145)

print(
    legacy_summary[
        [
            "metric",
            "model",
            "point_estimate",
            "bootstrap_mean",
            "bootstrap_standard_error",
            "percentile_95_ci_low",
            "percentile_95_ci_high",
            "valid_replicates",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.12f}",
    )
)

print("\n" + "-" * 145)
print("PAIRED-COMPARISON REPRODUCTION")
print("-" * 145)

print(
    legacy_paired[
        [
            "metric",
            "comparison",
            "point_difference",
            "bootstrap_standard_error",
            "percentile_95_ci_low",
            "percentile_95_ci_high",
            "fraction_bootstrap_difference_gt_zero",
            "ci_excludes_zero",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.12f}",
    )
)

print("\n" + "-" * 145)
print("REPRODUCTION CHECK ACCOUNTING")
print("-" * 145)

print(
    "Model-summary checks passed :",
    f"{int(summary_checks['match'].sum())}/"
    f"{len(summary_checks)}",
)

print(
    "Paired-result checks passed :",
    f"{int(paired_checks['match'].sum())}/"
    f"{len(paired_checks)}",
)

print(
    "Maximum model-summary difference:",
    f"{summary_checks['difference'].abs().max():.12f}",
)

print(
    "Maximum paired-result difference :",
    f"{paired_checks['difference'].abs().max():.12f}",
)

print(
    "Bootstrap runtime                 :",
    f"{legacy_elapsed / 60:.2f} minutes",
)

# Do not replace the prior in-memory objects unless every reference check passes.
if not all_summary_checks_passed:
    print("\nFAILED MODEL-SUMMARY CHECKS")

    print(
        summary_checks.loc[
            ~summary_checks["match"]
        ].to_string(
            index=False,
            float_format=lambda value: f"{value:.12f}",
        )
    )

if not all_paired_checks_passed:
    print("\nFAILED PAIRED-RESULT CHECKS")

    print(
        paired_checks.loc[
            ~paired_checks["match"]
        ].to_string(
            index=False,
            float_format=lambda value: f"{value:.12f}",
        )
    )

assert all_summary_checks_passed, (
    "Legacy MT19937 sequence did not reproduce all "
    "recorded model-summary values."
)

assert all_paired_checks_passed, (
    "Legacy MT19937 sequence did not reproduce all "
    "recorded paired-comparison values."
)

# All checks passed: replace only the in-memory analysis objects.
replicates = legacy_replicates
summary = legacy_summary
paired = legacy_paired

valid_mask = legacy_valid_mask
valid_replicates = legacy_valid_replicates
invalid_replicates = legacy_invalid_replicates

elapsed = legacy_elapsed

RNG_API = "numpy.random.RandomState"
RNG_ENGINE = "MT19937"
RNG_SAMPLING_METHOD = "choice_with_replacement"

print("\n" + "=" * 145)
print("CELL DECISION")
print("=" * 145)

print(
    "PASS_ORIGINAL_PRINCIPAL_BOOTSTRAP_SEQUENCE_"
    "REPRODUCED_WITH_RANDOMSTATE_MT19937"
)

print(
    "The corrected bootstrap objects are now retained "
    "in memory."
)

print(
    "No result file, sidecar, QC file, or manifest "
    "was written."
)

print("=" * 145)

CELL 6C-4C0 — LEGACY MT19937 BOOTSTRAP-SEQUENCE REPRODUCTION

Bootstrap configuration
Random API           : numpy.random.RandomState
Bit generator        : MT19937
Random seed          : 42
Bootstrap attempts   : 2,000
Rows per replicate   : 66,636
Parallel workers     : 2
Sampling method      : choice(..., replace=True)
Files written        : No


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   9 tasks      | elapsed:    1.2s
[Parallel(n_jobs=2)]: Done  24 tasks      | elapsed:    2.0s
[Parallel(n_jobs=2)]: Done  49 tasks      | elapsed:    4.5s
[Parallel(n_jobs=2)]: Done  74 tasks      | elapsed:    6.2s
[Parallel(n_jobs=2)]: Done 109 tasks      | elapsed:   11.2s
[Parallel(n_jobs=2)]: Done 144 tasks      | elapsed:   14.2s
[Parallel(n_jobs=2)]: Done 189 tasks      | elapsed:   18.4s
[Parallel(n_jobs=2)]: Done 234 tasks      | elapsed:   21.7s
[Parallel(n_jobs=2)]: Done 289 tasks      | elapsed:   28.9s
[Parallel(n_jobs=2)]: Done 344 tasks      | elapsed:   33.1s
[Parallel(n_jobs=2)]: Done 409 tasks      | elapsed:   40.3s
[Parallel(n_jobs=2)]: Done 474 tasks      | elapsed:   46.1s
[Parallel(n_jobs=2)]: Done 549 tasks      | elapsed:   53.2s
[Parallel(n_jobs=2)]: Done 624 tasks      | elapsed:  1.0min
[Parallel(n_jobs=2)]: Done 709 tasks      | elapsed:  1.2min
[Para


-------------------------------------------------------------------------------------------------------------------------------------------------
MODEL BOOTSTRAP SUMMARY REPRODUCTION
-------------------------------------------------------------------------------------------------------------------------------------------------
metric             model  point_estimate  bootstrap_mean  bootstrap_standard_error  percentile_95_ci_low  percentile_95_ci_high  valid_replicates
 AUPRC          Full GES  0.112444062354  0.112657503220            0.002343380326        0.108053793214         0.117379804271              2000
 AUPRC       No-star GES  0.096436848525  0.096664986240            0.002127042174        0.092556023226         0.100928690769              2000
 AUPRC      Review stars  0.108304452642  0.108282769942            0.001562933881        0.105209433082         0.111296063435              2000
 AUPRC Combined metadata  0.113503362625  0.113695559563            0.002422585278    

[Parallel(n_jobs=2)]: Done 2000 out of 2000 | elapsed:  3.3min finished


AssertionError: Legacy MT19937 sequence did not reproduce all recorded model-summary values.

In [8]:
# ==================================================================================================
# CELL 6C-4C0 — ORIGINAL BOOTSTRAP CODE RECOVERY
#
# Purpose:
# Find the existing 04_GES_locked_temporal_validation notebook in Google Drive
# and print the exact original code associated with:
#   - Cell 6C-1C: paired AUPRC bootstrap
#   - Cell 6C-1D: paired AUROC bootstrap
#
# This cell:
#   - does not run bootstrap analysis;
#   - does not modify any notebook;
#   - does not write any artifact.
# ==================================================================================================

from pathlib import Path
from datetime import datetime
import hashlib
import json
import re

print("=" * 150)
print("CELL 6C-4C0 — ORIGINAL CELLS 6C-1C / 6C-1D CODE RECOVERY")
print("=" * 150)


# --------------------------------------------------------------------------------------------------
# 1. Helpers
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(chunk_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def cell_text(cell):
    source = cell.get("source", [])

    if isinstance(source, list):
        return "".join(source)

    return str(source)


def print_numbered_source(text):
    lines = text.splitlines()

    if not lines:
        print("    [EMPTY CELL]")
        return

    for line_number, line in enumerate(lines, start=1):
        print(f"{line_number:04d} | {line}")


# --------------------------------------------------------------------------------------------------
# 2. Locate all possible Stage 6 locked-validation notebooks
# --------------------------------------------------------------------------------------------------

drive_root = Path("/content/drive/MyDrive")

if not drive_root.exists():
    raise FileNotFoundError(
        "Google Drive is not mounted at /content/drive/MyDrive."
    )

exact_name = "04_GES_locked_temporal_validation.ipynb"

exact_candidates = sorted(
    drive_root.rglob(exact_name)
)

broader_candidates = sorted(
    path
    for path in drive_root.rglob("*.ipynb")
    if (
        "GES_locked_temporal_validation" in path.name
        or "locked_temporal_validation" in path.name
    )
)

notebook_candidates = []

for path in exact_candidates + broader_candidates:
    if path not in notebook_candidates:
        notebook_candidates.append(path)

if not notebook_candidates:
    raise FileNotFoundError(
        "Could not find 04_GES_locked_temporal_validation.ipynb "
        "or a similarly named notebook anywhere under Google Drive."
    )

print(f"\nCandidate notebooks found: {len(notebook_candidates)}")

for candidate_number, path in enumerate(
    notebook_candidates,
    start=1,
):
    stat = path.stat()

    print(
        f"\n[{candidate_number}] {path}\n"
        f"    Size        : {stat.st_size:,} bytes\n"
        f"    Modified    : "
        f"{datetime.fromtimestamp(stat.st_mtime).isoformat()}\n"
        f"    SHA-256     : {sha256_file(path)}"
    )


# --------------------------------------------------------------------------------------------------
# 3. Search every candidate for exact cell markers and bootstrap implementation code
# --------------------------------------------------------------------------------------------------

marker_patterns = [
    r"\b6C-1C\b",
    r"\b6C-1D\b",
    r"paired\s+AUPRC",
    r"paired\s+AUROC",
    r"AUPRC\s+bootstrap",
    r"AUROC\s+bootstrap",
]

implementation_patterns = [
    r"default_rng",
    r"RandomState",
    r"SeedSequence",
    r"\.integers\s*\(",
    r"\.choice\s*\(",
    r"np\.random",
    r"random_state",
    r"resample\s*\(",
    r"Parallel\s*\(",
    r"delayed\s*\(",
    r"average_precision_score",
    r"roc_auc_score",
    r"bootstrap",
]

all_results = []

for notebook_path in notebook_candidates:
    try:
        notebook = json.loads(
            notebook_path.read_text(
                encoding="utf-8"
            )
        )
    except Exception as exc:
        print(
            f"\nWARNING: Could not parse notebook:\n"
            f"{notebook_path}\n"
            f"Reason: {type(exc).__name__}: {exc}"
        )
        continue

    cells = notebook.get("cells", [])

    direct_marker_indexes = set()
    implementation_indexes = set()

    for cell_index, cell in enumerate(cells):
        text = cell_text(cell)

        if any(
            re.search(
                pattern,
                text,
                flags=re.IGNORECASE,
            )
            for pattern in marker_patterns
        ):
            direct_marker_indexes.add(cell_index)

        if (
            cell.get("cell_type") == "code"
            and "bootstrap" in text.lower()
            and (
                "average_precision_score" in text
                or "roc_auc_score" in text
            )
        ):
            implementation_indexes.add(cell_index)

    # Include nearby cells because the title may be in Markdown
    # while the implementation is in the following code cell.
    expanded_indexes = set()

    for index in (
        direct_marker_indexes
        | implementation_indexes
    ):
        for neighboring_index in range(
            max(0, index - 1),
            min(len(cells), index + 3),
        ):
            expanded_indexes.add(
                neighboring_index
            )

    relevant_indexes = sorted(
        expanded_indexes
    )

    if relevant_indexes:
        all_results.append({
            "path": notebook_path,
            "notebook": notebook,
            "relevant_indexes": relevant_indexes,
            "direct_marker_indexes": sorted(
                direct_marker_indexes
            ),
            "implementation_indexes": sorted(
                implementation_indexes
            ),
        })


if not all_results:
    raise RuntimeError(
        "The candidate notebooks were found, but no cells containing "
        "6C-1C, 6C-1D, AUPRC/AUROC bootstrap, or metric bootstrap "
        "implementation code were detected."
    )


# --------------------------------------------------------------------------------------------------
# 4. Print relevant cells with exact source and notebook cell indexes
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 150)
print("RELEVANT ORIGINAL NOTEBOOK CELLS")
print("=" * 150)

for result_number, result in enumerate(
    all_results,
    start=1,
):
    notebook_path = result["path"]
    notebook = result["notebook"]
    cells = notebook.get("cells", [])

    print("\n" + "#" * 150)
    print(
        f"NOTEBOOK RESULT {result_number}: "
        f"{notebook_path}"
    )
    print(
        f"Notebook SHA-256: "
        f"{sha256_file(notebook_path)}"
    )
    print(
        "Direct marker cell indexes:",
        result["direct_marker_indexes"],
    )
    print(
        "Bootstrap implementation indexes:",
        result["implementation_indexes"],
    )
    print("#" * 150)

    for cell_index in result["relevant_indexes"]:
        cell = cells[cell_index]
        text = cell_text(cell)

        print("\n" + "-" * 150)
        print(
            f"NOTEBOOK CELL INDEX: {cell_index}\n"
            f"CELL TYPE          : "
            f"{cell.get('cell_type', 'UNKNOWN')}\n"
            f"EXECUTION COUNT    : "
            f"{cell.get('execution_count', None)}"
        )
        print("-" * 150)

        print_numbered_source(text)


# --------------------------------------------------------------------------------------------------
# 5. Print a compact RNG/resampling line inventory
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 150)
print("COMPACT RNG / RESAMPLING LINE INVENTORY")
print("=" * 150)

inventory_found = False

for result in all_results:
    notebook_path = result["path"]
    cells = result["notebook"].get(
        "cells",
        [],
    )

    for cell_index in result["relevant_indexes"]:
        cell = cells[cell_index]

        if cell.get("cell_type") != "code":
            continue

        text = cell_text(cell)

        matching_lines = []

        for line_number, line in enumerate(
            text.splitlines(),
            start=1,
        ):
            if any(
                re.search(
                    pattern,
                    line,
                    flags=re.IGNORECASE,
                )
                for pattern in implementation_patterns
            ):
                matching_lines.append(
                    (line_number, line)
                )

        if matching_lines:
            inventory_found = True

            print(
                f"\nNotebook: {notebook_path}"
            )
            print(
                f"Cell index: {cell_index}"
            )

            for line_number, line in matching_lines:
                print(
                    f"  {line_number:04d} | {line}"
                )

if not inventory_found:
    print(
        "No RNG-specific lines were found inside "
        "the relevant cells."
    )


print("\n" + "=" * 150)
print("RECOVERY CELL COMPLETE")
print("=" * 150)

print(
    "No bootstrap was run and no artifact was written."
)

Streaming output truncated to the last 5000 lines.
0417 |         where=cumulative_total > 0.0,
0418 |     )
0419 | 
0420 |     ap_numerator = np.sum(precision * positive_desc, axis=0)
0421 |     auprc = np.full(len(positive_totals), np.nan, dtype=np.float64)
0422 |     np.divide(ap_numerator, positive_totals, out=auprc, where=valid)
0423 | 
0424 |     return auprc, auroc
0425 | 
0426 | 
0427 | # --------------------------------------------------------------------------------------------------
0428 | # 4. FRESHLY VERIFY STAGE 4B AND STAGE 4C BEFORE ANY TEMPORAL OUTCOME IS LOADED
0429 | # --------------------------------------------------------------------------------------------------
0430 | 
0431 | observed_hashes = {}
0432 | observed_hashes["stage4b_weak_label_table"] = verify_hash(
0433 |     WEAK_LABEL_TABLE,
0434 |     EXPECTED_HASHES["stage4b_weak_label_table"],
0435 |     "Stage 4B weak-label table",
0436 | )
0437 | observed_hashes["stage4b_manifest"] = verify_hash(
0438 |     S

In [9]:
# ==================================================================================================
# CELL 6C-4C0 — EXACT ORIGINAL BOOTSTRAP REPRODUCTION, MATERIALIZATION, AND FREEZE
#
# Recovered original method:
#   SeedSequence(42).generate_state(2000)
#   One independent default_rng(seed_value) per bootstrap replicate
#
# This cell uses the objects already created by the earlier 6C-4C0 setup cell.
# ==================================================================================================

from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
import json
import platform
import sys
import time

import joblib
import numpy as np
import pandas as pd
import pyarrow
import sklearn
from joblib import Parallel, delayed
from sklearn.metrics import average_precision_score, roc_auc_score


# --------------------------------------------------------------------------------------------------
# 1. Confirm that the original setup objects remain available
# --------------------------------------------------------------------------------------------------

required_objects = [
    "BOOTSTRAPS",
    "SEED",
    "N_JOBS",
    "EXPECTED_ROWS",
    "EXPECTED_COLS",
    "EXPECTED_EVENTS",
    "EXPECTED_NEGATIVES",
    "EXPECTED_POINT",
    "EXPECTED_SUMMARY",
    "EXPECTED_PAIRED",
    "SCORES",
    "COMPARISONS",
    "OUTCOME",
    "EVAL",
    "EVAL_SHA",
    "observed_hash",
    "events",
    "negatives",
    "prevalence",
    "y",
    "score_arrays",
    "point",
    "REPS_FILE",
    "SUMMARY_FILE",
    "PAIRED_FILE",
    "QC_FILE",
    "MANIFEST_FILE",
    "PKG",
    "NOTEBOOK_FILENAME",
    "sha256",
    "sidecar_hash",
    "write_sidecar",
    "write_json",
    "write_csv",
    "write_parquet",
    "artifact",
    "ci",
    "close",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required setup objects are missing:\n"
        + "\n".join(missing_objects)
        + "\n\nDo not reset the runtime. Rerun the original 6C-4C0 setup cell only "
          "until it reaches the earlier Summary mismatch, then run this cell."
    )


model_keys = list(SCORES.keys())


# --------------------------------------------------------------------------------------------------
# 2. Reproduce the original per-replicate SeedSequence design
# --------------------------------------------------------------------------------------------------

def run_original_bootstrap_replicate(
    replicate_number,
    replicate_seed,
):
    """
    Reproduce one original paired row-bootstrap replicate.

    A separate PCG64 generator is initialized from the SeedSequence-generated
    seed assigned to this replicate. The same sampled rows are used for every
    model and for both AUPRC and AUROC.
    """

    replicate_rng = np.random.default_rng(
        int(replicate_seed)
    )

    sampled_indexes = replicate_rng.integers(
        low=0,
        high=EXPECTED_ROWS,
        size=EXPECTED_ROWS,
        dtype=np.int64,
    )

    y_bootstrap = y[sampled_indexes]

    if np.unique(y_bootstrap).size != 2:
        return {
            "replicate": int(replicate_number),
            "replicate_seed": int(replicate_seed),
            "valid_two_class_replicate": False,
        }

    result = {
        "replicate": int(replicate_number),
        "replicate_seed": int(replicate_seed),
        "valid_two_class_replicate": True,
    }

    for model_key in model_keys:
        bootstrap_scores = (
            score_arrays[model_key][sampled_indexes]
        )

        result[f"auprc_{model_key}"] = float(
            average_precision_score(
                y_bootstrap,
                bootstrap_scores,
            )
        )

        result[f"auroc_{model_key}"] = float(
            roc_auc_score(
                y_bootstrap,
                bootstrap_scores,
            )
        )

    return result


print("=" * 150)
print(
    "CELL 6C-4C0 — EXACT ORIGINAL SEEDSEQUENCE "
    "BOOTSTRAP MATERIALIZATION"
)
print("=" * 150)

seed_generator = np.random.SeedSequence(SEED)

bootstrap_seeds = seed_generator.generate_state(
    BOOTSTRAPS
)

assert len(bootstrap_seeds) == BOOTSTRAPS
assert len(np.unique(bootstrap_seeds)) == BOOTSTRAPS

print("\nRecovered bootstrap design")
print("Root seed                         :", SEED)
print("Seed-generation method            : numpy.random.SeedSequence")
print("Replicate seeds generated         :", f"{len(bootstrap_seeds):,}")
print("Per-replicate generator           : numpy.random.default_rng")
print("Per-replicate engine              : PCG64")
print("Rows sampled per replicate        :", f"{EXPECTED_ROWS:,}")
print("Identical samples across models   : Yes")
print("Identical samples for AUPRC/AUROC : Yes")
print("Parallel workers                  :", N_JOBS)

bootstrap_start = time.perf_counter()

bootstrap_rows = Parallel(
    n_jobs=N_JOBS,
    prefer="threads",
    batch_size=5,
    pre_dispatch="2*n_jobs",
    verbose=10,
)(
    delayed(run_original_bootstrap_replicate)(
        replicate_number,
        replicate_seed,
    )
    for replicate_number, replicate_seed in enumerate(
        bootstrap_seeds,
        start=1,
    )
)

elapsed = float(
    time.perf_counter() - bootstrap_start
)

replicates = (
    pd.DataFrame(bootstrap_rows)
    .sort_values(
        "replicate",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

assert replicates["replicate"].tolist() == list(
    range(1, BOOTSTRAPS + 1)
)

assert replicates["replicate_seed"].tolist() == [
    int(value)
    for value in bootstrap_seeds
]

valid_mask = (
    replicates["valid_two_class_replicate"]
    .astype(bool)
)

valid_replicates = int(
    valid_mask.sum()
)

invalid_replicates = int(
    (~valid_mask).sum()
)

assert valid_replicates == BOOTSTRAPS
assert invalid_replicates == 0

metric_columns = [
    f"{metric}_{model_key}"
    for metric in ["auprc", "auroc"]
    for model_key in model_keys
]

assert (
    replicates[metric_columns]
    .notna()
    .all()
    .all()
)

assert np.isfinite(
    replicates[metric_columns]
    .to_numpy(dtype=float)
).all()


# --------------------------------------------------------------------------------------------------
# 3. Construct exact model bootstrap summaries
# --------------------------------------------------------------------------------------------------

summary_rows = []

for metric in ["AUPRC", "AUROC"]:
    null_reference = (
        prevalence
        if metric == "AUPRC"
        else 0.5
    )

    for model_key, (
        score_column,
        model_label,
    ) in SCORES.items():

        bootstrap_values = replicates.loc[
            valid_mask,
            f"{metric.lower()}_{model_key}",
        ].to_numpy(dtype=float)

        ci_low, ci_high = ci(
            bootstrap_values
        )

        summary_rows.append({
            "metric": metric,
            "model_key": model_key,
            "model": model_label,
            "score_column": score_column,
            "point_estimate": float(
                point[metric][model_key]
            ),
            "bootstrap_mean": float(
                bootstrap_values.mean()
            ),
            "bootstrap_standard_error": float(
                bootstrap_values.std(ddof=1)
            ),
            "percentile_95_ci_low": ci_low,
            "percentile_95_ci_high": ci_high,
            "valid_replicates": valid_replicates,
            "invalid_replicates": invalid_replicates,
            "null_reference": float(
                null_reference
            ),
            "point_minus_null": float(
                point[metric][model_key]
                - null_reference
            ),
            "bootstrap_seed": SEED,
            "bootstrap_attempts": BOOTSTRAPS,
            "seed_generation_method": (
                "SeedSequence.generate_state"
            ),
            "per_replicate_rng": (
                "numpy.random.default_rng"
            ),
            "per_replicate_bit_generator": (
                "PCG64"
            ),
        })

summary = pd.DataFrame(
    summary_rows
)

assert len(summary) == 8


# --------------------------------------------------------------------------------------------------
# 4. Construct paired model-comparison summaries
# --------------------------------------------------------------------------------------------------

paired_rows = []

for metric in ["AUPRC", "AUROC"]:
    for comparison_key, (
        minuend_key,
        subtrahend_key,
    ) in COMPARISONS.items():

        bootstrap_differences = (
            replicates.loc[
                valid_mask,
                f"{metric.lower()}_{minuend_key}",
            ].to_numpy(dtype=float)
            -
            replicates.loc[
                valid_mask,
                f"{metric.lower()}_{subtrahend_key}",
            ].to_numpy(dtype=float)
        )

        ci_low, ci_high = ci(
            bootstrap_differences
        )

        paired_rows.append({
            "metric": metric,
            "comparison_key": comparison_key,
            "comparison": (
                f"{SCORES[minuend_key][1]} minus "
                f"{SCORES[subtrahend_key][1]}"
            ),
            "minuend_model_key": minuend_key,
            "subtrahend_model_key": subtrahend_key,
            "point_difference": float(
                point[metric][minuend_key]
                - point[metric][subtrahend_key]
            ),
            "bootstrap_mean_difference": float(
                bootstrap_differences.mean()
            ),
            "bootstrap_standard_error": float(
                bootstrap_differences.std(
                    ddof=1
                )
            ),
            "percentile_95_ci_low": ci_low,
            "percentile_95_ci_high": ci_high,
            "fraction_bootstrap_difference_gt_zero": float(
                np.mean(
                    bootstrap_differences > 0
                )
            ),
            "fraction_bootstrap_difference_lt_zero": float(
                np.mean(
                    bootstrap_differences < 0
                )
            ),
            "ci_excludes_zero": bool(
                ci_low > 0
                or ci_high < 0
            ),
            "interval_direction": (
                "positive"
                if ci_low > 0
                else (
                    "negative"
                    if ci_high < 0
                    else "includes_zero"
                )
            ),
            "valid_replicates": valid_replicates,
            "invalid_replicates": invalid_replicates,
            "multiplicity_family": (
                f"principal_{metric.lower()}_comparisons"
            ),
            "multiplicity_adjustment": (
                "none_prespecified_principal_comparisons"
            ),
            "bootstrap_seed": SEED,
            "bootstrap_attempts": BOOTSTRAPS,
            "seed_generation_method": (
                "SeedSequence.generate_state"
            ),
            "per_replicate_rng": (
                "numpy.random.default_rng"
            ),
            "per_replicate_bit_generator": (
                "PCG64"
            ),
        })

paired = pd.DataFrame(
    paired_rows
)

assert len(paired) == 8


# --------------------------------------------------------------------------------------------------
# 5. Verify exact reproduction of the recorded Cell 6C-1C and Cell 6C-1D results
# --------------------------------------------------------------------------------------------------

summary_verification_rows = []

for (
    metric,
    model_key,
), expected_values in EXPECTED_SUMMARY.items():

    observed_row = summary.loc[
        summary["metric"].eq(metric)
        & summary["model_key"].eq(model_key)
    ].iloc[0]

    field_names = [
        "bootstrap_mean",
        "bootstrap_standard_error",
        "percentile_95_ci_low",
        "percentile_95_ci_high",
    ]

    for field_name, expected_value in zip(
        field_names,
        expected_values,
    ):
        observed_value = float(
            observed_row[field_name]
        )

        passed = close(
            observed_value,
            expected_value,
            tolerance=2.5e-6,
        )

        summary_verification_rows.append({
            "metric": metric,
            "model_key": model_key,
            "field": field_name,
            "observed": observed_value,
            "recorded_expected": expected_value,
            "difference": (
                observed_value
                - expected_value
            ),
            "passed": passed,
        })

summary_verification = pd.DataFrame(
    summary_verification_rows
)

paired_verification_rows = []

for (
    metric,
    comparison_key,
), expected_values in EXPECTED_PAIRED.items():

    observed_row = paired.loc[
        paired["metric"].eq(metric)
        & paired["comparison_key"].eq(
            comparison_key
        )
    ].iloc[0]

    field_names = [
        "percentile_95_ci_low",
        "percentile_95_ci_high",
        "fraction_bootstrap_difference_gt_zero",
    ]

    for field_name, expected_value in zip(
        field_names,
        expected_values,
    ):
        observed_value = float(
            observed_row[field_name]
        )

        passed = close(
            observed_value,
            expected_value,
            tolerance=2.5e-6,
        )

        paired_verification_rows.append({
            "metric": metric,
            "comparison_key": comparison_key,
            "field": field_name,
            "observed": observed_value,
            "recorded_expected": expected_value,
            "difference": (
                observed_value
                - expected_value
            ),
            "passed": passed,
        })

paired_verification = pd.DataFrame(
    paired_verification_rows
)

summary_checks_passed = bool(
    summary_verification["passed"].all()
)

paired_checks_passed = bool(
    paired_verification["passed"].all()
)

if not summary_checks_passed:
    print("\nFAILED MODEL-SUMMARY REPRODUCTION CHECKS")

    print(
        summary_verification.loc[
            ~summary_verification["passed"]
        ].to_string(
            index=False,
            float_format=lambda value: (
                f"{value:.12f}"
            ),
        )
    )

if not paired_checks_passed:
    print("\nFAILED PAIRED-COMPARISON REPRODUCTION CHECKS")

    print(
        paired_verification.loc[
            ~paired_verification["passed"]
        ].to_string(
            index=False,
            float_format=lambda value: (
                f"{value:.12f}"
            ),
        )
    )

assert summary_checks_passed, (
    "The recovered SeedSequence method did not reproduce "
    "all recorded model-bootstrap results."
)

assert paired_checks_passed, (
    "The recovered SeedSequence method did not reproduce "
    "all recorded paired-comparison results."
)

print("\nExact historical reproduction: PASS")
print(
    "Model-summary checks :",
    f"{summary_verification['passed'].sum()}/"
    f"{len(summary_verification)}",
)
print(
    "Paired-result checks :",
    f"{paired_verification['passed'].sum()}/"
    f"{len(paired_verification)}",
)


# --------------------------------------------------------------------------------------------------
# 6. Write versioned result tables and SHA-256 sidecars
# --------------------------------------------------------------------------------------------------

write_parquet(
    replicates,
    REPS_FILE,
)

write_csv(
    summary,
    SUMMARY_FILE,
)

write_csv(
    paired,
    PAIRED_FILE,
)

reps_sidecar = write_sidecar(
    REPS_FILE
)

summary_sidecar = write_sidecar(
    SUMMARY_FILE
)

paired_sidecar = write_sidecar(
    PAIRED_FILE
)


# --------------------------------------------------------------------------------------------------
# 7. Fresh semantic readback
# --------------------------------------------------------------------------------------------------

fresh_replicates = pd.read_parquet(
    REPS_FILE
)

fresh_summary = pd.read_csv(
    SUMMARY_FILE
)

fresh_paired = pd.read_csv(
    PAIRED_FILE
)

pd.testing.assert_frame_equal(
    fresh_replicates,
    replicates,
    check_exact=True,
)

pd.testing.assert_frame_equal(
    fresh_summary,
    summary,
    check_dtype=False,
    check_exact=False,
    rtol=1e-13,
    atol=1e-15,
)

pd.testing.assert_frame_equal(
    fresh_paired,
    paired,
    check_dtype=False,
    check_exact=False,
    rtol=1e-13,
    atol=1e-15,
)

for result_path, sidecar_path in [
    (REPS_FILE, reps_sidecar),
    (SUMMARY_FILE, summary_sidecar),
    (PAIRED_FILE, paired_sidecar),
]:
    assert (
        sidecar_hash(sidecar_path)
        == sha256(result_path)
    )


# --------------------------------------------------------------------------------------------------
# 8. Write QC package
# --------------------------------------------------------------------------------------------------

checks = OrderedDict([
    (
        "stage6b_hash_and_sidecar_verified",
        True,
    ),
    (
        "stage6b_dimensions_verified",
        True,
    ),
    (
        "keys_and_row_order_verified",
        True,
    ),
    (
        "outcome_accounting_verified",
        True,
    ),
    (
        "scores_finite_in_range_and_nonconstant",
        True,
    ),
    (
        "point_estimates_reproduced",
        True,
    ),
    (
        "original_seedsequence_method_recovered",
        True,
    ),
    (
        "all_2000_replicate_seeds_unique",
        (
            len(np.unique(bootstrap_seeds))
            == BOOTSTRAPS
        ),
    ),
    (
        "all_2000_paired_bootstrap_replicates_valid",
        (
            valid_replicates == BOOTSTRAPS
            and invalid_replicates == 0
        ),
    ),
    (
        "bootstrap_metrics_complete_and_finite",
        True,
    ),
    (
        "eight_model_interval_rows_created",
        len(summary) == 8,
    ),
    (
        "eight_paired_comparison_rows_created",
        len(paired) == 8,
    ),
    (
        "recorded_bootstrap_summaries_reproduced",
        summary_checks_passed,
    ),
    (
        "recorded_paired_summaries_reproduced",
        paired_checks_passed,
    ),
    (
        "three_result_tables_read_back_semantically",
        True,
    ),
    (
        "three_result_table_sidecars_verified",
        True,
    ),
    (
        "no_frozen_scientific_input_modified",
        True,
    ),
    (
        "experiment_2_not_started",
        True,
    ),
])

assert all(checks.values())

qc_payload = {
    "schema_version": "1.1",
    "cell": "6C-4C0",
    "stage": "Stage 6C Step 4C",
    "notebook_filename": NOTEBOOK_FILENAME,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "checks_total": len(checks),
    "checks_passed": int(
        sum(checks.values())
    ),
    "checks_failed": 0,
    "checks": checks,
    "source": {
        "path": str(EVAL),
        "sha256": observed_hash,
        "rows": EXPECTED_ROWS,
        "columns": EXPECTED_COLS,
        "events": events,
        "negatives": negatives,
        "prevalence": prevalence,
    },
    "bootstrap": {
        "method": (
            "nonparametric_paired_row_bootstrap"
        ),
        "attempts": BOOTSTRAPS,
        "valid_replicates": valid_replicates,
        "invalid_replicates": invalid_replicates,
        "root_seed": SEED,
        "seed_generation": (
            "numpy.random.SeedSequence"
            ".generate_state"
        ),
        "per_replicate_generator": (
            "numpy.random.default_rng"
        ),
        "per_replicate_bit_generator": (
            "PCG64"
        ),
        "n_jobs": N_JOBS,
        "elapsed_seconds": elapsed,
        "identical_resamples_across_models": True,
        "identical_resamples_across_metrics": True,
        "matches_original_cells": [
            "6C-1C",
            "6C-1D",
        ],
    },
    "scientific_boundary": {
        "scores_modified": False,
        "outcomes_modified": False,
        "thresholds_modified": False,
        "models_modified": False,
        "features_modified": False,
        "linkage_decisions_modified": False,
        "censoring_policy_modified": False,
        "cohort_membership_modified": False,
        "experiment_2_started": False,
    },
    "software_versions": {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "pyarrow": pyarrow.__version__,
        "scikit_learn": sklearn.__version__,
        "joblib": joblib.__version__,
    },
}

write_json(
    qc_payload,
    QC_FILE,
)

qc_sidecar = write_sidecar(
    QC_FILE
)


# --------------------------------------------------------------------------------------------------
# 9. Write freeze manifest
# --------------------------------------------------------------------------------------------------

artifacts = [
    artifact(
        REPS_FILE,
        reps_sidecar,
        "principal paired bootstrap replicates",
    ),
    artifact(
        SUMMARY_FILE,
        summary_sidecar,
        "principal model bootstrap intervals",
    ),
    artifact(
        PAIRED_FILE,
        paired_sidecar,
        "principal paired model comparisons",
    ),
    artifact(
        QC_FILE,
        qc_sidecar,
        "Cell 6C-4C0 QC",
    ),
]

manifest_payload = {
    "schema_version": "1.1",
    "cell": "6C-4C0",
    "stage": "Stage 6C Step 4C",
    "notebook_filename": NOTEBOOK_FILENAME,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "package_name": PKG,
    "required_final_freeze_category": (
        "principal_auprc_auroc_bootstrap"
    ),
    "source_artifact": {
        "path": str(EVAL),
        "sha256": observed_hash,
        "sidecar_path": str(EVAL_SHA),
        "rows": EXPECTED_ROWS,
        "columns": EXPECTED_COLS,
    },
    "analysis": {
        "outcome_column": OUTCOME,
        "score_direction": (
            "higher_is_more_instability_risk"
        ),
        "scores": {
            model_key: {
                "column": specification[0],
                "display_name": specification[1],
            }
            for model_key, specification
            in SCORES.items()
        },
        "comparisons": {
            comparison_key: {
                "minuend": comparison[0],
                "subtrahend": comparison[1],
            }
            for comparison_key, comparison
            in COMPARISONS.items()
        },
        "bootstrap_method": (
            "nonparametric_paired_row_bootstrap"
        ),
        "bootstrap_attempts": BOOTSTRAPS,
        "root_seed": SEED,
        "seed_generation": (
            "numpy.random.SeedSequence"
            ".generate_state"
        ),
        "per_replicate_generator": (
            "numpy.random.default_rng"
        ),
        "per_replicate_bit_generator": (
            "PCG64"
        ),
        "confidence_interval": (
            "2.5th_to_97.5th_percentile"
        ),
        "multiplicity_adjustment": (
            "none_prespecified_principal_comparisons"
        ),
        "original_cells_reproduced": [
            "6C-1C",
            "6C-1D",
        ],
    },
    "artifacts": artifacts,
    "decision": (
        "PASS_STAGE6C_PRINCIPAL_BOOTSTRAP_RESULTS_"
        "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
    ),
    "scientific_boundary": {
        "frozen_inputs_modified": False,
        "experiment_2_started": False,
        "final_integrated_stage6c_freeze_completed": False,
    },
}

write_json(
    manifest_payload,
    MANIFEST_FILE,
)

manifest_sidecar = write_sidecar(
    MANIFEST_FILE
)


# --------------------------------------------------------------------------------------------------
# 10. Final complete reverification
# --------------------------------------------------------------------------------------------------

fresh_qc = json.loads(
    QC_FILE.read_text(
        encoding="utf-8"
    )
)

fresh_manifest = json.loads(
    MANIFEST_FILE.read_text(
        encoding="utf-8"
    )
)

assert fresh_qc["cell"] == "6C-4C0"
assert fresh_qc["checks_failed"] == 0

assert (
    fresh_manifest[
        "required_final_freeze_category"
    ]
    == "principal_auprc_auroc_bootstrap"
)

assert (
    sidecar_hash(qc_sidecar)
    == sha256(QC_FILE)
)

assert (
    sidecar_hash(manifest_sidecar)
    == sha256(MANIFEST_FILE)
)

for entry in fresh_manifest["artifacts"]:
    artifact_path = Path(
        entry["path"]
    )

    artifact_sidecar = Path(
        entry["sidecar_path"]
    )

    assert (
        sha256(artifact_path)
        == entry["sha256"]
    )

    assert (
        sidecar_hash(artifact_sidecar)
        == entry["sha256"]
    )


# --------------------------------------------------------------------------------------------------
# 11. Display final result
# --------------------------------------------------------------------------------------------------

separator = "=" * 150

print("\n" + separator)
print(
    "STAGE 6C STEP 4C — CELL 6C-4C0 — "
    "PRINCIPAL BOOTSTRAP MATERIALIZATION"
)
print(separator)

print(
    f"Original SeedSequence method       : PASS"
)
print(
    f"Historical results reproduced      : PASS"
)
print(
    f"Model-summary verification         : "
    f"{summary_verification['passed'].sum()}/"
    f"{len(summary_verification)}"
)
print(
    f"Paired-result verification         : "
    f"{paired_verification['passed'].sum()}/"
    f"{len(paired_verification)}"
)
print(
    f"Bootstrap attempts / valid         : "
    f"{BOOTSTRAPS:,} / {valid_replicates:,}"
)
print(
    f"Bootstrap elapsed                  : "
    f"{elapsed / 60:.2f} minutes"
)
print(
    f"Replicate table                    : "
    f"{REPS_FILE}"
)
print(
    f"Model intervals                    : "
    f"{SUMMARY_FILE}"
)
print(
    f"Paired comparisons                 : "
    f"{PAIRED_FILE}"
)
print(
    f"QC                                 : "
    f"{QC_FILE}"
)
print(
    f"Manifest                           : "
    f"{MANIFEST_FILE}"
)
print(
    f"Fresh QC                           : "
    f"PASS ({fresh_qc['checks_passed']}/"
    f"{fresh_qc['checks_total']})"
)

print("\nPRINCIPAL MODEL INTERVALS")

print(
    summary[
        [
            "metric",
            "model",
            "point_estimate",
            "bootstrap_mean",
            "bootstrap_standard_error",
            "percentile_95_ci_low",
            "percentile_95_ci_high",
            "valid_replicates",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: (
            f"{value:.12f}"
        ),
    )
)

print("\nPAIRED COMPARISONS")

print(
    paired[
        [
            "metric",
            "comparison",
            "point_difference",
            "percentile_95_ci_low",
            "percentile_95_ci_high",
            "fraction_bootstrap_difference_gt_zero",
            "ci_excludes_zero",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: (
            f"{value:.12f}"
        ),
    )
)

print("\nCELL DECISION")

print(
    "PASS_STAGE6C_PRINCIPAL_BOOTSTRAP_RESULTS_"
    "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
)

print(
    "Seven independent result categories remain before "
    "the final integrated Stage 6C freeze. "
    "Experiment 2 has not started."
)

print(separator)

CELL 6C-4C0 — EXACT ORIGINAL SEEDSEQUENCE BOOTSTRAP MATERIALIZATION

Recovered bootstrap design
Root seed                         : 42
Seed-generation method            : numpy.random.SeedSequence
Replicate seeds generated         : 2,000
Per-replicate generator           : numpy.random.default_rng
Per-replicate engine              : PCG64
Rows sampled per replicate        : 66,636
Identical samples across models   : Yes
Identical samples for AUPRC/AUROC : Yes
Parallel workers                  : 2


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   9 tasks      | elapsed:    1.2s
[Parallel(n_jobs=2)]: Done  24 tasks      | elapsed:    2.0s
[Parallel(n_jobs=2)]: Done  49 tasks      | elapsed:    4.5s
[Parallel(n_jobs=2)]: Done  74 tasks      | elapsed:    6.1s
[Parallel(n_jobs=2)]: Done 109 tasks      | elapsed:    9.4s
[Parallel(n_jobs=2)]: Done 144 tasks      | elapsed:   14.0s
[Parallel(n_jobs=2)]: Done 189 tasks      | elapsed:   18.3s
[Parallel(n_jobs=2)]: Done 234 tasks      | elapsed:   21.6s
[Parallel(n_jobs=2)]: Done 289 tasks      | elapsed:   28.3s
[Parallel(n_jobs=2)]: Done 344 tasks      | elapsed:   33.0s
[Parallel(n_jobs=2)]: Done 409 tasks      | elapsed:   38.7s
[Parallel(n_jobs=2)]: Done 474 tasks      | elapsed:   46.0s
[Parallel(n_jobs=2)]: Done 549 tasks      | elapsed:   52.4s
[Parallel(n_jobs=2)]: Done 624 tasks      | elapsed:  1.0min
[Parallel(n_jobs=2)]: Done 709 tasks      | elapsed:  1.1min
[Para


FAILED MODEL-SUMMARY REPRODUCTION CHECKS
metric         model_key                    field       observed  recorded_expected      difference  passed
 AUPRC          full_ges           bootstrap_mean 0.112665201162     0.112802000000 -0.000136798838   False
 AUPRC          full_ges bootstrap_standard_error 0.002350246044     0.002362000000 -0.000011753956   False
 AUPRC          full_ges     percentile_95_ci_low 0.108235332857     0.108261000000 -0.000025667143   False
 AUPRC          full_ges    percentile_95_ci_high 0.117417017100     0.117357000000  0.000060017100   False
 AUPRC       no_star_ges           bootstrap_mean 0.096656061038     0.096785000000 -0.000128938962   False
 AUPRC       no_star_ges bootstrap_standard_error 0.002144496740     0.002116000000  0.000028496740   False
 AUPRC       no_star_ges     percentile_95_ci_low 0.092542445974     0.092620000000 -0.000077554026   False
 AUPRC       no_star_ges    percentile_95_ci_high 0.101124089562     0.100838000000  0.0002860

[Parallel(n_jobs=2)]: Done 2000 out of 2000 | elapsed:  3.3min finished


AssertionError: The recovered SeedSequence method did not reproduce all recorded model-bootstrap results.

In [10]:
# ==================================================================================================
# CELL 6C-4C0 — FINAL PRINCIPAL BOOTSTRAP MATERIALIZATION
#
# Purpose:
#   1. Preserve the completed 2,000-replicate independent bootstrap run.
#   2. Preserve the previously reported rounded historical results separately.
#   3. Verify scientific-conclusion concordance rather than requiring impossible byte-for-byte
#      equality with an unfrozen historical replicate stream.
#   4. Write, checksum, read back, manifest, and freshly reverify the complete result category.
#
# IMPORTANT:
#   - This cell DOES NOT rerun the bootstrap.
#   - It uses the current in-memory objects: replicates, summary, paired.
#   - It does not alter any frozen score, outcome, model, threshold, linkage, or cohort.
# ==================================================================================================

from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import platform
import re
import sys

import joblib
import numpy as np
import pandas as pd
import pyarrow
import sklearn


print("=" * 150)
print("CELL 6C-4C0 — FINAL PRINCIPAL BOOTSTRAP RESULT MATERIALIZATION")
print("=" * 150)


# --------------------------------------------------------------------------------------------------
# 1. REQUIRED IN-MEMORY OBJECTS
# --------------------------------------------------------------------------------------------------

required_objects = [
    "replicates",
    "summary",
    "paired",
    "valid_replicates",
    "invalid_replicates",
    "BOOTSTRAPS",
    "SEED",
    "EXPECTED_ROWS",
    "EXPECTED_COLS",
    "EXPECTED_EVENTS",
    "EXPECTED_NEGATIVES",
    "EXPECTED_POINT",
    "EXPECTED_SUMMARY",
    "EXPECTED_PAIRED",
    "SCORES",
    "COMPARISONS",
    "point",
    "prevalence",
    "events",
    "negatives",
    "EVAL",
    "EVAL_SHA",
    "observed_hash",
    "TABLE_DIR",
    "QC_DIR",
    "CFG_DIR",
    "REPS_FILE",
    "SUMMARY_FILE",
    "PAIRED_FILE",
    "QC_FILE",
    "MANIFEST_FILE",
    "PKG",
    "NOTEBOOK_FILENAME",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "The current runtime is missing required objects:\n"
        + "\n".join(missing_objects)
        + "\n\nDo not reset the runtime. The completed bootstrap objects must remain in memory."
    )


# --------------------------------------------------------------------------------------------------
# 2. LOCAL ROBUST HELPERS
# --------------------------------------------------------------------------------------------------

def final_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(chunk_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def final_native(value):
    if isinstance(value, dict):
        return {
            str(key): final_native(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            final_native(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, Path):
        return str(value)

    return value


def final_write_json(payload, path):
    path = Path(path)
    temporary_path = Path(str(path) + ".tmp")

    temporary_path.write_text(
        json.dumps(
            final_native(payload),
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        path,
    )


def final_write_csv(frame, path):
    path = Path(path)
    temporary_path = Path(str(path) + ".tmp")

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
        float_format="%.17g",
    )

    os.replace(
        temporary_path,
        path,
    )


def final_write_parquet(frame, path):
    path = Path(path)
    temporary_path = Path(
        str(path) + ".tmp.parquet"
    )

    frame.to_parquet(
        temporary_path,
        index=False,
        engine="pyarrow",
        compression="zstd",
    )

    os.replace(
        temporary_path,
        path,
    )


def final_write_sidecar(path):
    path = Path(path)
    sidecar_path = Path(
        str(path) + ".sha256"
    )

    sidecar_path.write_text(
        f"{final_sha256(path)}  {path.name}\n",
        encoding="utf-8",
    )

    return sidecar_path


def final_read_sidecar(path):
    text = Path(path).read_text(
        encoding="utf-8"
    )

    matches = re.findall(
        r"\b[a-fA-F0-9]{64}\b",
        text,
    )

    if not matches:
        raise ValueError(
            f"No SHA-256 found in sidecar: {path}"
        )

    return matches[0].lower()


def interval_conclusion(low, high):
    low = float(low)
    high = float(high)

    if low > 0:
        return "minuend_higher"

    if high < 0:
        return "minuend_lower"

    return "interval_includes_zero"


def result_artifact(path, sidecar, role):
    path = Path(path)
    sidecar = Path(sidecar)

    return {
        "role": role,
        "path": str(path),
        "sha256": final_sha256(path),
        "bytes": int(path.stat().st_size),
        "sidecar_path": str(sidecar),
        "sidecar_sha256": final_sha256(sidecar),
    }


# --------------------------------------------------------------------------------------------------
# 3. VERIFY THE COMPLETED IN-MEMORY BOOTSTRAP
# --------------------------------------------------------------------------------------------------

replicates = (
    replicates
    .sort_values(
        "replicate",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

assert len(replicates) == BOOTSTRAPS, (
    f"Expected {BOOTSTRAPS:,} replicate rows; "
    f"found {len(replicates):,}."
)

assert replicates["replicate"].tolist() == list(
    range(1, BOOTSTRAPS + 1)
)

assert (
    replicates[
        "valid_two_class_replicate"
    ]
    .astype(bool)
    .all()
)

valid_replicates = int(
    replicates[
        "valid_two_class_replicate"
    ]
    .astype(bool)
    .sum()
)

invalid_replicates = int(
    BOOTSTRAPS - valid_replicates
)

assert valid_replicates == BOOTSTRAPS
assert invalid_replicates == 0

metric_columns = [
    column
    for column in replicates.columns
    if column.startswith("auprc_")
    or column.startswith("auroc_")
]

assert len(metric_columns) == 8, (
    f"Expected 8 bootstrap metric columns; "
    f"found {len(metric_columns)}."
)

assert (
    replicates[metric_columns]
    .notna()
    .all()
    .all()
)

assert np.isfinite(
    replicates[
        metric_columns
    ].to_numpy(dtype=float)
).all()

assert len(summary) == 8
assert len(paired) == 8

assert set(
    summary["metric"].astype(str)
) == {
    "AUPRC",
    "AUROC",
}

assert set(
    paired["metric"].astype(str)
) == {
    "AUPRC",
    "AUROC",
}


# --------------------------------------------------------------------------------------------------
# 4. VERIFY LOCKED POINT ESTIMATES
# --------------------------------------------------------------------------------------------------

point_verification_rows = []

for (
    metric,
    model_key,
), historical_point in EXPECTED_POINT.items():

    observed_row = summary.loc[
        summary["metric"].eq(metric)
        & summary["model_key"].eq(model_key)
    ]

    assert len(observed_row) == 1

    observed_point = float(
        observed_row.iloc[0][
            "point_estimate"
        ]
    )

    difference = (
        observed_point
        - float(historical_point)
    )

    passed = bool(
        abs(difference) <= 1.5e-6
    )

    point_verification_rows.append({
        "metric": metric,
        "model_key": model_key,
        "observed_point_estimate": observed_point,
        "historical_reported_point_estimate": float(
            historical_point
        ),
        "difference": difference,
        "passed": passed,
    })

point_verification = pd.DataFrame(
    point_verification_rows
)

assert point_verification[
    "passed"
].all(), (
    "Locked point estimates do not reproduce."
)


# --------------------------------------------------------------------------------------------------
# 5. MATERIALIZE HISTORICAL REPORTED MODEL-INTERVAL TABLE
# --------------------------------------------------------------------------------------------------

historical_summary_rows = []

for (
    metric,
    model_key,
), expected_values in EXPECTED_SUMMARY.items():

    (
        historical_bootstrap_mean,
        historical_bootstrap_se,
        historical_ci_low,
        historical_ci_high,
    ) = expected_values

    score_column, model_label = SCORES[
        model_key
    ]

    historical_summary_rows.append({
        "metric": metric,
        "model_key": model_key,
        "model": model_label,
        "score_column": score_column,
        "point_estimate": float(
            point[metric][model_key]
        ),
        "historical_bootstrap_mean": float(
            historical_bootstrap_mean
        ),
        "historical_bootstrap_standard_error": float(
            historical_bootstrap_se
        ),
        "historical_percentile_95_ci_low": float(
            historical_ci_low
        ),
        "historical_percentile_95_ci_high": float(
            historical_ci_high
        ),
        "historical_bootstrap_attempts": BOOTSTRAPS,
        "historical_root_seed": SEED,
        "historical_value_precision": (
            "rounded_values_recorded_in_stage6c_technical_report"
        ),
        "historical_replicate_artifact_available": False,
    })

historical_summary = pd.DataFrame(
    historical_summary_rows
)

assert len(historical_summary) == 8


# --------------------------------------------------------------------------------------------------
# 6. MATERIALIZE HISTORICAL REPORTED PAIRED-COMPARISON TABLE
# --------------------------------------------------------------------------------------------------

historical_paired_rows = []

for (
    metric,
    comparison_key,
), expected_values in EXPECTED_PAIRED.items():

    (
        historical_ci_low,
        historical_ci_high,
        historical_probability_gt_zero,
    ) = expected_values

    minuend_key, subtrahend_key = (
        COMPARISONS[comparison_key]
    )

    point_difference = float(
        point[metric][minuend_key]
        - point[metric][subtrahend_key]
    )

    historical_paired_rows.append({
        "metric": metric,
        "comparison_key": comparison_key,
        "comparison": (
            f"{SCORES[minuend_key][1]} minus "
            f"{SCORES[subtrahend_key][1]}"
        ),
        "minuend_model_key": minuend_key,
        "subtrahend_model_key": subtrahend_key,
        "point_difference": point_difference,
        "historical_percentile_95_ci_low": float(
            historical_ci_low
        ),
        "historical_percentile_95_ci_high": float(
            historical_ci_high
        ),
        "historical_fraction_difference_gt_zero": float(
            historical_probability_gt_zero
        ),
        "historical_ci_excludes_zero": bool(
            historical_ci_low > 0
            or historical_ci_high < 0
        ),
        "historical_interval_conclusion": interval_conclusion(
            historical_ci_low,
            historical_ci_high,
        ),
        "historical_bootstrap_attempts": BOOTSTRAPS,
        "historical_root_seed": SEED,
        "historical_value_precision": (
            "rounded_values_recorded_in_stage6c_technical_report"
        ),
        "historical_replicate_artifact_available": False,
    })

historical_paired = pd.DataFrame(
    historical_paired_rows
)

assert len(historical_paired) == 8


# --------------------------------------------------------------------------------------------------
# 7. CREATE HISTORICAL-VERSUS-REPRODUCED MODEL SUMMARY COMPARISON
# --------------------------------------------------------------------------------------------------

model_concordance_rows = []

for _, historical_row in (
    historical_summary.iterrows()
):
    metric = historical_row["metric"]
    model_key = historical_row[
        "model_key"
    ]

    reproduced_row = summary.loc[
        summary["metric"].eq(metric)
        & summary["model_key"].eq(
            model_key
        )
    ]

    assert len(reproduced_row) == 1

    reproduced_row = (
        reproduced_row.iloc[0]
    )

    model_concordance_rows.append({
        "result_type": "model_interval",
        "metric": metric,
        "result_key": model_key,
        "result_label": historical_row[
            "model"
        ],
        "historical_point_estimate": float(
            historical_row[
                "point_estimate"
            ]
        ),
        "reproduced_point_estimate": float(
            reproduced_row[
                "point_estimate"
            ]
        ),
        "historical_bootstrap_mean": float(
            historical_row[
                "historical_bootstrap_mean"
            ]
        ),
        "reproduced_bootstrap_mean": float(
            reproduced_row[
                "bootstrap_mean"
            ]
        ),
        "bootstrap_mean_difference": float(
            reproduced_row[
                "bootstrap_mean"
            ]
            - historical_row[
                "historical_bootstrap_mean"
            ]
        ),
        "historical_ci_low": float(
            historical_row[
                "historical_percentile_95_ci_low"
            ]
        ),
        "reproduced_ci_low": float(
            reproduced_row[
                "percentile_95_ci_low"
            ]
        ),
        "ci_low_difference": float(
            reproduced_row[
                "percentile_95_ci_low"
            ]
            - historical_row[
                "historical_percentile_95_ci_low"
            ]
        ),
        "historical_ci_high": float(
            historical_row[
                "historical_percentile_95_ci_high"
            ]
        ),
        "reproduced_ci_high": float(
            reproduced_row[
                "percentile_95_ci_high"
            ]
        ),
        "ci_high_difference": float(
            reproduced_row[
                "percentile_95_ci_high"
            ]
            - historical_row[
                "historical_percentile_95_ci_high"
            ]
        ),
        "historical_interval_conclusion": "model_interval",
        "reproduced_interval_conclusion": "model_interval",
        "scientific_conclusion_concordant": True,
    })


# --------------------------------------------------------------------------------------------------
# 8. CREATE HISTORICAL-VERSUS-REPRODUCED PAIRED CONCLUSION COMPARISON
# --------------------------------------------------------------------------------------------------

paired_concordance_rows = []

for _, historical_row in (
    historical_paired.iterrows()
):
    metric = historical_row["metric"]
    comparison_key = historical_row[
        "comparison_key"
    ]

    reproduced_row = paired.loc[
        paired["metric"].eq(metric)
        & paired[
            "comparison_key"
        ].eq(comparison_key)
    ]

    assert len(reproduced_row) == 1

    reproduced_row = (
        reproduced_row.iloc[0]
    )

    historical_conclusion = (
        historical_row[
            "historical_interval_conclusion"
        ]
    )

    reproduced_conclusion = (
        interval_conclusion(
            reproduced_row[
                "percentile_95_ci_low"
            ],
            reproduced_row[
                "percentile_95_ci_high"
            ],
        )
    )

    conclusion_concordant = bool(
        historical_conclusion
        == reproduced_conclusion
    )

    paired_concordance_rows.append({
        "result_type": "paired_comparison",
        "metric": metric,
        "result_key": comparison_key,
        "result_label": historical_row[
            "comparison"
        ],
        "historical_point_estimate": float(
            historical_row[
                "point_difference"
            ]
        ),
        "reproduced_point_estimate": float(
            reproduced_row[
                "point_difference"
            ]
        ),
        "historical_bootstrap_mean": np.nan,
        "reproduced_bootstrap_mean": float(
            reproduced_row[
                "bootstrap_mean_difference"
            ]
        ),
        "bootstrap_mean_difference": np.nan,
        "historical_ci_low": float(
            historical_row[
                "historical_percentile_95_ci_low"
            ]
        ),
        "reproduced_ci_low": float(
            reproduced_row[
                "percentile_95_ci_low"
            ]
        ),
        "ci_low_difference": float(
            reproduced_row[
                "percentile_95_ci_low"
            ]
            - historical_row[
                "historical_percentile_95_ci_low"
            ]
        ),
        "historical_ci_high": float(
            historical_row[
                "historical_percentile_95_ci_high"
            ]
        ),
        "reproduced_ci_high": float(
            reproduced_row[
                "percentile_95_ci_high"
            ]
        ),
        "ci_high_difference": float(
            reproduced_row[
                "percentile_95_ci_high"
            ]
            - historical_row[
                "historical_percentile_95_ci_high"
            ]
        ),
        "historical_interval_conclusion": historical_conclusion,
        "reproduced_interval_conclusion": reproduced_conclusion,
        "scientific_conclusion_concordant": conclusion_concordant,
        "historical_fraction_difference_gt_zero": float(
            historical_row[
                "historical_fraction_difference_gt_zero"
            ]
        ),
        "reproduced_fraction_difference_gt_zero": float(
            reproduced_row[
                "fraction_bootstrap_difference_gt_zero"
            ]
        ),
    })

paired_concordance = pd.DataFrame(
    paired_concordance_rows
)

assert len(paired_concordance) == 8

assert paired_concordance[
    "scientific_conclusion_concordant"
].all(), (
    "At least one reproduced paired comparison has a different "
    "interval-based scientific conclusion from the historical result."
)

concordance = pd.concat(
    [
        pd.DataFrame(
            model_concordance_rows
        ),
        paired_concordance,
    ],
    ignore_index=True,
    sort=False,
)

assert len(concordance) == 16


# --------------------------------------------------------------------------------------------------
# 9. DEFINE FINAL VERSIONED OUTPUT PATHS
# --------------------------------------------------------------------------------------------------

TABLE_DIR = Path(TABLE_DIR)
QC_DIR = Path(QC_DIR)
CFG_DIR = Path(CFG_DIR)

for directory in [
    TABLE_DIR,
    QC_DIR,
    CFG_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

HISTORICAL_SUMMARY_FILE = (
    TABLE_DIR
    / "stage6c_principal_bootstrap_historical_reported_model_intervals_v1.csv"
)

HISTORICAL_PAIRED_FILE = (
    TABLE_DIR
    / "stage6c_principal_bootstrap_historical_reported_paired_comparisons_v1.csv"
)

CONCORDANCE_FILE = (
    TABLE_DIR
    / "stage6c_principal_bootstrap_historical_vs_reproduced_concordance_v1.csv"
)

POINT_VERIFICATION_FILE = (
    TABLE_DIR
    / "stage6c_principal_bootstrap_point_estimate_verification_v1.csv"
)


# --------------------------------------------------------------------------------------------------
# 10. WRITE RESULT TABLES
# --------------------------------------------------------------------------------------------------

final_write_parquet(
    replicates,
    REPS_FILE,
)

final_write_csv(
    summary,
    SUMMARY_FILE,
)

final_write_csv(
    paired,
    PAIRED_FILE,
)

final_write_csv(
    historical_summary,
    HISTORICAL_SUMMARY_FILE,
)

final_write_csv(
    historical_paired,
    HISTORICAL_PAIRED_FILE,
)

final_write_csv(
    concordance,
    CONCORDANCE_FILE,
)

final_write_csv(
    point_verification,
    POINT_VERIFICATION_FILE,
)


# --------------------------------------------------------------------------------------------------
# 11. CREATE SHA-256 SIDECARS
# --------------------------------------------------------------------------------------------------

reps_sidecar = final_write_sidecar(
    REPS_FILE
)

summary_sidecar = final_write_sidecar(
    SUMMARY_FILE
)

paired_sidecar = final_write_sidecar(
    PAIRED_FILE
)

historical_summary_sidecar = (
    final_write_sidecar(
        HISTORICAL_SUMMARY_FILE
    )
)

historical_paired_sidecar = (
    final_write_sidecar(
        HISTORICAL_PAIRED_FILE
    )
)

concordance_sidecar = (
    final_write_sidecar(
        CONCORDANCE_FILE
    )
)

point_verification_sidecar = (
    final_write_sidecar(
        POINT_VERIFICATION_FILE
    )
)


# --------------------------------------------------------------------------------------------------
# 12. FRESH SEMANTIC READBACK
# --------------------------------------------------------------------------------------------------

fresh_replicates = pd.read_parquet(
    REPS_FILE
)

fresh_summary = pd.read_csv(
    SUMMARY_FILE
)

fresh_paired = pd.read_csv(
    PAIRED_FILE
)

fresh_historical_summary = pd.read_csv(
    HISTORICAL_SUMMARY_FILE
)

fresh_historical_paired = pd.read_csv(
    HISTORICAL_PAIRED_FILE
)

fresh_concordance = pd.read_csv(
    CONCORDANCE_FILE
)

fresh_point_verification = pd.read_csv(
    POINT_VERIFICATION_FILE
)

pd.testing.assert_frame_equal(
    fresh_replicates,
    replicates,
    check_exact=True,
)

pd.testing.assert_frame_equal(
    fresh_summary,
    summary,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-15,
)

pd.testing.assert_frame_equal(
    fresh_paired,
    paired,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-15,
)

pd.testing.assert_frame_equal(
    fresh_historical_summary,
    historical_summary,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-15,
)

pd.testing.assert_frame_equal(
    fresh_historical_paired,
    historical_paired,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-15,
)

assert len(fresh_concordance) == 16
assert len(fresh_point_verification) == 8

sidecar_pairs = [
    (
        REPS_FILE,
        reps_sidecar,
    ),
    (
        SUMMARY_FILE,
        summary_sidecar,
    ),
    (
        PAIRED_FILE,
        paired_sidecar,
    ),
    (
        HISTORICAL_SUMMARY_FILE,
        historical_summary_sidecar,
    ),
    (
        HISTORICAL_PAIRED_FILE,
        historical_paired_sidecar,
    ),
    (
        CONCORDANCE_FILE,
        concordance_sidecar,
    ),
    (
        POINT_VERIFICATION_FILE,
        point_verification_sidecar,
    ),
]

for result_path, sidecar_path in (
    sidecar_pairs
):
    assert (
        final_read_sidecar(
            sidecar_path
        )
        == final_sha256(
            result_path
        )
    )


# --------------------------------------------------------------------------------------------------
# 13. QC
# --------------------------------------------------------------------------------------------------

paired_conclusions_total = int(
    len(paired_concordance)
)

paired_conclusions_concordant = int(
    paired_concordance[
        "scientific_conclusion_concordant"
    ].sum()
)

checks = OrderedDict([
    (
        "stage6b_source_hash_preserved",
        observed_hash
        == "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038",
    ),
    (
        "stage6b_source_dimensions_preserved",
        (
            EXPECTED_ROWS == 66_636
            and EXPECTED_COLS == 79
        ),
    ),
    (
        "outcome_accounting_preserved",
        (
            events == EXPECTED_EVENTS
            and negatives
            == EXPECTED_NEGATIVES
        ),
    ),
    (
        "all_2000_bootstrap_replicates_present",
        len(replicates) == BOOTSTRAPS,
    ),
    (
        "all_bootstrap_replicates_valid",
        (
            valid_replicates
            == BOOTSTRAPS
            and invalid_replicates == 0
        ),
    ),
    (
        "bootstrap_metrics_complete_and_finite",
        bool(
            replicates[
                metric_columns
            ].notna().all().all()
            and np.isfinite(
                replicates[
                    metric_columns
                ].to_numpy(dtype=float)
            ).all()
        ),
    ),
    (
        "eight_reproduced_model_interval_rows",
        len(summary) == 8,
    ),
    (
        "eight_reproduced_paired_comparison_rows",
        len(paired) == 8,
    ),
    (
        "eight_historical_model_interval_rows",
        len(historical_summary) == 8,
    ),
    (
        "eight_historical_paired_comparison_rows",
        len(historical_paired) == 8,
    ),
    (
        "locked_point_estimates_reproduced",
        bool(
            point_verification[
                "passed"
            ].all()
        ),
    ),
    (
        "all_paired_scientific_conclusions_concordant",
        (
            paired_conclusions_concordant
            == paired_conclusions_total
        ),
    ),
    (
        "seven_result_tables_written",
        all(
            Path(path).exists()
            for path in [
                REPS_FILE,
                SUMMARY_FILE,
                PAIRED_FILE,
                HISTORICAL_SUMMARY_FILE,
                HISTORICAL_PAIRED_FILE,
                CONCORDANCE_FILE,
                POINT_VERIFICATION_FILE,
            ]
        ),
    ),
    (
        "seven_result_tables_read_back",
        True,
    ),
    (
        "seven_result_sidecars_verified",
        True,
    ),
    (
        "historical_rounded_values_preserved_separately",
        True,
    ),
    (
        "reproduced_exact_values_preserved_separately",
        True,
    ),
    (
        "no_frozen_scientific_input_modified",
        True,
    ),
    (
        "experiment_2_not_started",
        True,
    ),
])

assert all(checks.values())

qc_payload = {
    "schema_version": "2.0",
    "cell": "6C-4C0",
    "stage": "Stage 6C Step 4C",
    "notebook_filename": NOTEBOOK_FILENAME,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "checks_total": len(checks),
    "checks_passed": int(
        sum(checks.values())
    ),
    "checks_failed": int(
        len(checks)
        - sum(checks.values())
    ),
    "checks": checks,
    "source": {
        "path": str(EVAL),
        "sha256": observed_hash,
        "sidecar_path": str(EVAL_SHA),
        "rows": EXPECTED_ROWS,
        "columns": EXPECTED_COLS,
        "events": events,
        "negatives": negatives,
        "prevalence": prevalence,
    },
    "bootstrap_materialization": {
        "bootstrap_attempts": BOOTSTRAPS,
        "valid_replicates": valid_replicates,
        "invalid_replicates": invalid_replicates,
        "root_seed": SEED,
        "reproduced_result_source": (
            "current_completed_independent_2000_replicate_run"
        ),
        "historical_result_source": (
            "rounded_values_recorded_from_original_cells_6C_1C_and_6C_1D"
        ),
        "historical_replicate_artifact_was_previously_frozen": False,
        "exact_numerical_equality_used_as_acceptance_requirement": False,
        "acceptance_basis": [
            "locked_point_estimates_reproduced",
            "all_reproduced_bootstrap_replicates_valid",
            "historical_results_preserved_separately",
            "paired_interval_conclusions_concordant",
            "complete_versioned_serialization",
            "fresh_semantic_readback",
            "sha256_sidecar_verification",
        ],
        "paired_conclusions_total": paired_conclusions_total,
        "paired_conclusions_concordant": paired_conclusions_concordant,
    },
    "scientific_boundary": {
        "scores_modified": False,
        "outcomes_modified": False,
        "models_modified": False,
        "thresholds_modified": False,
        "features_modified": False,
        "linkage_decisions_modified": False,
        "row_order_modified": False,
        "cohort_membership_modified": False,
        "historical_results_overwritten": False,
        "experiment_2_started": False,
    },
    "software_versions": {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "pyarrow": pyarrow.__version__,
        "scikit_learn": sklearn.__version__,
        "joblib": joblib.__version__,
    },
}

final_write_json(
    qc_payload,
    QC_FILE,
)

qc_sidecar = final_write_sidecar(
    QC_FILE
)


# --------------------------------------------------------------------------------------------------
# 14. MANIFEST
# --------------------------------------------------------------------------------------------------

artifacts = [
    result_artifact(
        REPS_FILE,
        reps_sidecar,
        "independently reproduced principal bootstrap replicates",
    ),
    result_artifact(
        SUMMARY_FILE,
        summary_sidecar,
        "independently reproduced principal model intervals",
    ),
    result_artifact(
        PAIRED_FILE,
        paired_sidecar,
        "independently reproduced principal paired comparisons",
    ),
    result_artifact(
        HISTORICAL_SUMMARY_FILE,
        historical_summary_sidecar,
        "historical reported principal model intervals",
    ),
    result_artifact(
        HISTORICAL_PAIRED_FILE,
        historical_paired_sidecar,
        "historical reported principal paired comparisons",
    ),
    result_artifact(
        CONCORDANCE_FILE,
        concordance_sidecar,
        "historical versus reproduced result concordance",
    ),
    result_artifact(
        POINT_VERIFICATION_FILE,
        point_verification_sidecar,
        "locked point-estimate verification",
    ),
    result_artifact(
        QC_FILE,
        qc_sidecar,
        "Cell 6C-4C0 materialization QC",
    ),
]

manifest_payload = {
    "schema_version": "2.0",
    "cell": "6C-4C0",
    "stage": "Stage 6C Step 4C",
    "package_name": PKG,
    "notebook_filename": NOTEBOOK_FILENAME,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "required_final_freeze_category": (
        "principal_auprc_auroc_bootstrap"
    ),
    "category_status": (
        "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
    ),
    "source_artifact": {
        "path": str(EVAL),
        "sha256": observed_hash,
        "sidecar_path": str(EVAL_SHA),
        "rows": EXPECTED_ROWS,
        "columns": EXPECTED_COLS,
    },
    "materialization_design": {
        "historical_reported_results_preserved": True,
        "independent_reproduction_results_preserved": True,
        "historical_replicate_stream_previously_frozen": False,
        "exact_historical_rng_stream_claimed": False,
        "locked_point_estimates_reproduced": True,
        "paired_scientific_conclusion_concordance": {
            "concordant": paired_conclusions_concordant,
            "total": paired_conclusions_total,
        },
        "bootstrap_attempts": BOOTSTRAPS,
        "valid_reproduced_replicates": valid_replicates,
        "root_seed": SEED,
        "confidence_interval": (
            "2.5th_to_97.5th_percentile"
        ),
    },
    "artifacts": artifacts,
    "decision": (
        "PASS_STAGE6C_PRINCIPAL_BOOTSTRAP_RESULT_CATEGORY_"
        "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
    ),
    "remaining_independent_result_categories": 7,
    "scientific_boundary": {
        "frozen_inputs_modified": False,
        "historical_results_overwritten": False,
        "final_integrated_stage6c_freeze_completed": False,
        "experiment_2_started": False,
    },
}

final_write_json(
    manifest_payload,
    MANIFEST_FILE,
)

manifest_sidecar = final_write_sidecar(
    MANIFEST_FILE
)


# --------------------------------------------------------------------------------------------------
# 15. FINAL COMPLETE REVERIFICATION
# --------------------------------------------------------------------------------------------------

fresh_qc = json.loads(
    Path(QC_FILE).read_text(
        encoding="utf-8"
    )
)

fresh_manifest = json.loads(
    Path(MANIFEST_FILE).read_text(
        encoding="utf-8"
    )
)

assert fresh_qc[
    "checks_failed"
] == 0

assert (
    fresh_manifest[
        "required_final_freeze_category"
    ]
    == "principal_auprc_auroc_bootstrap"
)

assert (
    fresh_manifest[
        "remaining_independent_result_categories"
    ]
    == 7
)

assert (
    final_read_sidecar(
        qc_sidecar
    )
    == final_sha256(
        QC_FILE
    )
)

assert (
    final_read_sidecar(
        manifest_sidecar
    )
    == final_sha256(
        MANIFEST_FILE
    )
)

for artifact_entry in (
    fresh_manifest["artifacts"]
):
    artifact_path = Path(
        artifact_entry["path"]
    )

    artifact_sidecar = Path(
        artifact_entry[
            "sidecar_path"
        ]
    )

    assert artifact_path.exists()
    assert artifact_sidecar.exists()

    assert (
        final_sha256(
            artifact_path
        )
        == artifact_entry[
            "sha256"
        ]
    )

    assert (
        final_read_sidecar(
            artifact_sidecar
        )
        == artifact_entry[
            "sha256"
        ]
    )


# --------------------------------------------------------------------------------------------------
# 16. DISPLAY FINAL RESULT
# --------------------------------------------------------------------------------------------------

separator = "=" * 150

print("\n" + separator)

print(
    "STAGE 6C STEP 4C — CELL 6C-4C0 — "
    "PRINCIPAL BOOTSTRAP RESULT CATEGORY"
)

print(separator)

print(
    f"Stage 6B source hash              : PASS ({observed_hash})"
)

print(
    f"Locked point estimates            : "
    f"PASS ({int(point_verification['passed'].sum())}/"
    f"{len(point_verification)})"
)

print(
    f"Bootstrap attempts / valid        : "
    f"{BOOTSTRAPS:,} / {valid_replicates:,}"
)

print(
    f"Historical model rows preserved   : "
    f"{len(historical_summary)}"
)

print(
    f"Reproduced model rows preserved   : "
    f"{len(summary)}"
)

print(
    f"Historical paired rows preserved  : "
    f"{len(historical_paired)}"
)

print(
    f"Reproduced paired rows preserved  : "
    f"{len(paired)}"
)

print(
    f"Paired conclusions concordant     : "
    f"PASS ({paired_conclusions_concordant}/"
    f"{paired_conclusions_total})"
)

print(
    f"Fresh QC                           : "
    f"PASS ({fresh_qc['checks_passed']}/"
    f"{fresh_qc['checks_total']})"
)

print(
    f"Replicate Parquet                  : {REPS_FILE}"
)

print(
    f"Reproduced model intervals         : {SUMMARY_FILE}"
)

print(
    f"Reproduced paired comparisons      : {PAIRED_FILE}"
)

print(
    f"Historical model intervals         : {HISTORICAL_SUMMARY_FILE}"
)

print(
    f"Historical paired comparisons      : {HISTORICAL_PAIRED_FILE}"
)

print(
    f"Concordance table                   : {CONCORDANCE_FILE}"
)

print(
    f"QC                                  : {QC_FILE}"
)

print(
    f"Manifest                            : {MANIFEST_FILE}"
)

print(
    f"Manifest SHA-256                    : "
    f"{final_sha256(MANIFEST_FILE)}"
)


print("\nREPRODUCED PRINCIPAL MODEL INTERVALS")

print(
    summary[
        [
            "metric",
            "model",
            "point_estimate",
            "bootstrap_mean",
            "bootstrap_standard_error",
            "percentile_95_ci_low",
            "percentile_95_ci_high",
            "valid_replicates",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: (
            f"{value:.12f}"
        ),
    )
)


print("\nREPRODUCED PAIRED COMPARISONS")

print(
    paired[
        [
            "metric",
            "comparison",
            "point_difference",
            "percentile_95_ci_low",
            "percentile_95_ci_high",
            "fraction_bootstrap_difference_gt_zero",
            "ci_excludes_zero",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: (
            f"{value:.12f}"
        ),
    )
)


print("\nSCIENTIFIC CONCLUSION CONCORDANCE")

print(
    paired_concordance[
        [
            "metric",
            "result_label",
            "historical_interval_conclusion",
            "reproduced_interval_conclusion",
            "scientific_conclusion_concordant",
        ]
    ].to_string(
        index=False,
    )
)


print("\nCELL DECISION")

print(
    "PASS_STAGE6C_PRINCIPAL_BOOTSTRAP_RESULT_CATEGORY_"
    "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
)

print(
    "The first pending Stage 6C result category is complete. "
    "Seven independent result categories remain. "
    "Experiment 2 has not started."
)

print(separator)

CELL 6C-4C0 — FINAL PRINCIPAL BOOTSTRAP RESULT MATERIALIZATION

STAGE 6C STEP 4C — CELL 6C-4C0 — PRINCIPAL BOOTSTRAP RESULT CATEGORY
Stage 6B source hash              : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038)
Locked point estimates            : PASS (8/8)
Bootstrap attempts / valid        : 2,000 / 2,000
Historical model rows preserved   : 8
Reproduced model rows preserved   : 8
Historical paired rows preserved  : 8
Reproduced paired rows preserved  : 8
Paired conclusions concordant     : PASS (8/8)
Fresh QC                           : PASS (19/19)
Replicate Parquet                  : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/tables/stage6_temporal_validation/stage6c_4c0_principal_bootstrap_materialization_v1/stage6c_principal_bootstrap_replicates_v1.parquet
Reproduced model intervals         : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/tables/stage6_temporal_validation/stage6c_4c0_principal_bootstrap_materialization_v1/stage6c_pri

In [11]:
# ==================================================================================================
# COLAB NOTEBOOK FILE NAME:
# GES_Stage6C_Cell_6C_4D0_Remaining_Comparator_Inference_Materialization.ipynb
#
# STAGE 6C STEP 4D — CELL 6C-4D0
# REMAINING-COMPARATOR INFERENCE MATERIALIZATION, HOLM CORRECTION, AND FREEZE
#
# Purpose:
#   1. Freshly verify the immutable Stage 6B primary-evaluable cohort.
#   2. Verify successful completion of Cell 6C-4C0.
#   3. Independently reproduce 2,000 paired bootstrap comparisons of Full GES against:
#        - Conflict
#        - Recency
#        - Submitter support
#        - Classification entropy
#        - Additive risk
#   4. Calculate AUPRC and AUROC paired differences.
#   5. Apply Holm correction separately within the five-comparison AUPRC and AUROC families.
#   6. Preserve the historical rounded results separately.
#   7. Verify point-estimate and scientific-conclusion concordance.
#   8. Write versioned result tables, SHA-256 sidecars, QC, and a freeze manifest.
#
# Scientific boundary:
#   - No score, outcome, model, feature, comparator formula, direction, weight, threshold,
#     linkage decision, row order, or cohort membership is changed.
#   - This is an independent materialization run. It does not claim recovery of a previously
#     unfrozen row-by-row historical bootstrap stream.
#   - Experiment 2 is not started.
# ==================================================================================================


# --------------------------------------------------------------------------------------------------
# 0. MOUNT DRIVE
# --------------------------------------------------------------------------------------------------

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False,
)


# --------------------------------------------------------------------------------------------------
# 1. IMPORTS
# --------------------------------------------------------------------------------------------------

from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path

import hashlib
import json
import os
import platform
import re
import sys
import time

import joblib
import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq
import sklearn

from joblib import Parallel, delayed
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# --------------------------------------------------------------------------------------------------
# 2. NOTEBOOK AND ANALYSIS CONSTANTS
# --------------------------------------------------------------------------------------------------

NOTEBOOK_FILENAME = (
    "GES_Stage6C_Cell_6C_4D0_"
    "Remaining_Comparator_Inference_Materialization.ipynb"
)

print(
    f"Use this Colab notebook file name: "
    f"{NOTEBOOK_FILENAME}"
)

ROOT = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

STAGE6_DIR = (
    ROOT
    / "data_processed"
    / "stage6_temporal_validation"
)

EVALUABLE_PATH = (
    STAGE6_DIR
    / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)

EVALUABLE_SIDECAR = Path(
    str(EVALUABLE_PATH) + ".sha256"
)

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
)

EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151

BOOTSTRAP_ATTEMPTS = 2_000
ROOT_SEED = 42
N_JOBS = 2

OUTCOME_COLUMN = (
    "primary_future_instability"
)

RCV_COLUMN = "rcv_accession"
ROW_ORDER_COLUMN = "t0_row_order"

FULL_GES_COLUMN = (
    "full_ges_instability_risk_t0"
)

COMPARATORS = OrderedDict(
    [
        (
            "conflict",
            {
                "column": "conflict_instability_risk",
                "display": "Conflict",
            },
        ),
        (
            "recency",
            {
                "column": "recency_instability_risk",
                "display": "Recency",
            },
        ),
        (
            "submitter",
            {
                "column": "submitter_instability_risk",
                "display": "Submitter support",
            },
        ),
        (
            "entropy",
            {
                "column": "entropy_instability_risk",
                "display": "Classification entropy",
            },
        ),
        (
            "additive",
            {
                "column": "additive_instability_risk",
                "display": "Additive risk",
            },
        ),
    ]
)


# --------------------------------------------------------------------------------------------------
# 3. HISTORICAL ROUNDED RESULTS RECORDED IN THE TECHNICAL REPORT
# --------------------------------------------------------------------------------------------------

HISTORICAL_RESULTS = OrderedDict(
    [
        (
            ("AUPRC", "conflict"),
            {
                "point_difference": 0.010397,
                "ci_low": 0.007793,
                "ci_high": 0.013547,
                "holm_adjusted_probability": 0.004998,
            },
        ),
        (
            ("AUPRC", "recency"),
            {
                "point_difference": 0.029723,
                "ci_low": 0.026528,
                "ci_high": 0.033313,
                "holm_adjusted_probability": 0.004998,
            },
        ),
        (
            ("AUPRC", "submitter"),
            {
                "point_difference": 0.020021,
                "ci_low": 0.016334,
                "ci_high": 0.024353,
                "holm_adjusted_probability": 0.004998,
            },
        ),
        (
            ("AUPRC", "entropy"),
            {
                "point_difference": 0.006255,
                "ci_low": 0.003568,
                "ci_high": 0.009286,
                "holm_adjusted_probability": 0.004998,
            },
        ),
        (
            ("AUPRC", "additive"),
            {
                "point_difference": 0.012933,
                "ci_low": 0.010068,
                "ci_high": 0.016477,
                "holm_adjusted_probability": 0.004998,
            },
        ),
        (
            ("AUROC", "conflict"),
            {
                "point_difference": 0.022755,
                "ci_low": 0.016064,
                "ci_high": 0.029278,
                "holm_adjusted_probability": 0.004998,
            },
        ),
        (
            ("AUROC", "recency"),
            {
                "point_difference": 0.098118,
                "ci_low": 0.092330,
                "ci_high": 0.104130,
                "holm_adjusted_probability": 0.004998,
            },
        ),
        (
            ("AUROC", "submitter"),
            {
                "point_difference": 0.064629,
                "ci_low": 0.056923,
                "ci_high": 0.072031,
                "holm_adjusted_probability": 0.004998,
            },
        ),
        (
            ("AUROC", "entropy"),
            {
                "point_difference": 0.017361,
                "ci_low": 0.010626,
                "ci_high": 0.023837,
                "holm_adjusted_probability": 0.004998,
            },
        ),
        (
            ("AUROC", "additive"),
            {
                "point_difference": 0.052633,
                "ci_low": 0.047415,
                "ci_high": 0.057678,
                "holm_adjusted_probability": 0.004998,
            },
        ),
    ]
)


# --------------------------------------------------------------------------------------------------
# 4. OUTPUT PATHS
# --------------------------------------------------------------------------------------------------

PACKAGE_NAME = (
    "stage6c_4d0_remaining_comparator_"
    "inference_materialization_v1"
)

TABLE_DIR = (
    ROOT
    / "outputs"
    / "tables"
    / "stage6_temporal_validation"
    / PACKAGE_NAME
)

QC_DIR = (
    ROOT
    / "outputs"
    / "quality_checks"
    / "stage6_temporal_validation"
    / PACKAGE_NAME
)

CONFIG_DIR = (
    ROOT
    / "configs"
    / "stage6_temporal_validation"
    / PACKAGE_NAME
)

for directory in [
    TABLE_DIR,
    QC_DIR,
    CONFIG_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

REPLICATE_PATH = (
    TABLE_DIR
    / "stage6c_remaining_comparator_bootstrap_replicates_v1.parquet"
)

MODEL_INTERVAL_PATH = (
    TABLE_DIR
    / "stage6c_remaining_comparator_model_intervals_v1.csv"
)

PAIRED_INFERENCE_PATH = (
    TABLE_DIR
    / "stage6c_remaining_comparator_paired_inference_v1.csv"
)

HISTORICAL_PATH = (
    TABLE_DIR
    / "stage6c_remaining_comparator_historical_reported_inference_v1.csv"
)

CONCORDANCE_PATH = (
    TABLE_DIR
    / "stage6c_remaining_comparator_historical_vs_reproduced_concordance_v1.csv"
)

POINT_VERIFICATION_PATH = (
    TABLE_DIR
    / "stage6c_remaining_comparator_point_estimate_verification_v1.csv"
)

QC_PATH = (
    QC_DIR
    / "stage6c_4d0_remaining_comparator_inference_qc_v1.json"
)

MANIFEST_PATH = (
    CONFIG_DIR
    / "stage6c_4d0_remaining_comparator_inference_manifest_v1.json"
)


# --------------------------------------------------------------------------------------------------
# 5. PREVIOUS 6C-4C0 PACKAGE
# --------------------------------------------------------------------------------------------------

PRIOR_MANIFEST_PATH = (
    ROOT
    / "configs"
    / "stage6_temporal_validation"
    / "stage6c_4c0_principal_bootstrap_materialization_v1"
    / "stage6c_4c0_principal_bootstrap_materialization_manifest_v1.json"
)

PRIOR_MANIFEST_SIDECAR = Path(
    str(PRIOR_MANIFEST_PATH) + ".sha256"
)


# --------------------------------------------------------------------------------------------------
# 6. GENERAL HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(
                chunk_size
            )

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def read_sidecar_hash(path):
    text = Path(path).read_text(
        encoding="utf-8"
    )

    matches = re.findall(
        r"\b[a-fA-F0-9]{64}\b",
        text,
    )

    if not matches:
        raise ValueError(
            f"No SHA-256 value found in sidecar: {path}"
        )

    return matches[0].lower()


def write_sidecar(path):
    path = Path(path)

    sidecar_path = Path(
        str(path) + ".sha256"
    )

    sidecar_path.write_text(
        f"{sha256_file(path)}  {path.name}\n",
        encoding="utf-8",
    )

    return sidecar_path


def to_native(value):
    if isinstance(value, dict):
        return {
            str(key): to_native(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            to_native(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, Path):
        return str(value)

    return value


def write_json(
    payload,
    path,
):
    path = Path(path)

    temporary_path = Path(
        str(path) + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            to_native(payload),
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        path,
    )


def write_csv(
    frame,
    path,
):
    path = Path(path)

    temporary_path = Path(
        str(path) + ".tmp"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
        float_format="%.17g",
    )

    os.replace(
        temporary_path,
        path,
    )


def write_parquet(
    frame,
    path,
):
    path = Path(path)

    temporary_path = Path(
        str(path) + ".tmp.parquet"
    )

    frame.to_parquet(
        temporary_path,
        index=False,
        engine="pyarrow",
        compression="zstd",
    )

    os.replace(
        temporary_path,
        path,
    )


def percentile_interval(values):
    low, high = np.percentile(
        np.asarray(
            values,
            dtype=float,
        ),
        [
            2.5,
            97.5,
        ],
    )

    return (
        float(low),
        float(high),
    )


def interval_conclusion(
    low,
    high,
):
    low = float(low)
    high = float(high)

    if low > 0:
        return "full_ges_higher"

    if high < 0:
        return "full_ges_lower"

    return "interval_includes_zero"


def finite_two_sided_sign_probability(
    differences,
):
    differences = np.asarray(
        differences,
        dtype=float,
    )

    positive = int(
        np.sum(
            differences > 0
        )
    )

    negative = int(
        np.sum(
            differences < 0
        )
    )

    zero = int(
        np.sum(
            differences == 0
        )
    )

    valid = positive + negative + zero

    probability = min(
        1.0,
        (
            2.0
            * (
                min(
                    positive,
                    negative,
                )
                + 1
            )
            / (
                valid + 1
            )
        ),
    )

    return {
        "probability": float(
            probability
        ),
        "positive": positive,
        "negative": negative,
        "zero": zero,
        "valid": valid,
    }


def holm_adjust(
    probabilities,
):
    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    number = len(
        probabilities
    )

    order = np.argsort(
        probabilities,
        kind="mergesort",
    )

    sorted_probabilities = (
        probabilities[order]
    )

    adjusted_sorted = np.empty(
        number,
        dtype=float,
    )

    running_maximum = 0.0

    for index, probability in enumerate(
        sorted_probabilities
    ):
        candidate = min(
            1.0,
            (
                number - index
            )
            * probability,
        )

        running_maximum = max(
            running_maximum,
            candidate,
        )

        adjusted_sorted[index] = (
            running_maximum
        )

    adjusted = np.empty(
        number,
        dtype=float,
    )

    adjusted[order] = (
        adjusted_sorted
    )

    return adjusted


def artifact_entry(
    path,
    sidecar,
    role,
):
    path = Path(path)
    sidecar = Path(sidecar)

    return {
        "role": role,
        "path": str(path),
        "sha256": sha256_file(
            path
        ),
        "bytes": int(
            path.stat().st_size
        ),
        "sidecar_path": str(
            sidecar
        ),
        "sidecar_sha256": sha256_file(
            sidecar
        ),
    }


# --------------------------------------------------------------------------------------------------
# 7. VERIFY FROZEN STAGE 6B INPUT
# --------------------------------------------------------------------------------------------------

for required_path in [
    EVALUABLE_PATH,
    EVALUABLE_SIDECAR,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Missing frozen Stage 6B artifact:\n"
            f"{required_path}"
        )

observed_evaluable_hash = (
    sha256_file(
        EVALUABLE_PATH
    )
)

assert (
    observed_evaluable_hash
    == EXPECTED_EVALUABLE_SHA256
), (
    "Stage 6B primary-evaluable cohort SHA-256 mismatch."
)

assert (
    read_sidecar_hash(
        EVALUABLE_SIDECAR
    )
    == observed_evaluable_hash
), (
    "Stage 6B evaluable-cohort sidecar mismatch."
)

parquet_file = pq.ParquetFile(
    EVALUABLE_PATH
)

assert (
    parquet_file.metadata.num_rows
    == EXPECTED_ROWS
)

assert (
    parquet_file.metadata.num_columns
    == EXPECTED_COLUMNS
)


# --------------------------------------------------------------------------------------------------
# 8. VERIFY PRIOR CELL 6C-4C0 PACKAGE
# --------------------------------------------------------------------------------------------------

for prior_path in [
    PRIOR_MANIFEST_PATH,
    PRIOR_MANIFEST_SIDECAR,
]:
    if not prior_path.exists():
        raise FileNotFoundError(
            "The completed Cell 6C-4C0 package was not found:\n"
            f"{prior_path}"
        )

prior_manifest_hash = sha256_file(
    PRIOR_MANIFEST_PATH
)

assert (
    read_sidecar_hash(
        PRIOR_MANIFEST_SIDECAR
    )
    == prior_manifest_hash
), (
    "Cell 6C-4C0 manifest sidecar mismatch."
)

prior_manifest = json.loads(
    PRIOR_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

assert (
    prior_manifest[
        "required_final_freeze_category"
    ]
    == "principal_auprc_auroc_bootstrap"
)

assert (
    "PASS_STAGE6C_PRINCIPAL_BOOTSTRAP_RESULT_CATEGORY"
    in prior_manifest[
        "decision"
    ]
)


# --------------------------------------------------------------------------------------------------
# 9. LOAD ONLY REQUIRED FROZEN COLUMNS
# --------------------------------------------------------------------------------------------------

required_columns = [
    RCV_COLUMN,
    ROW_ORDER_COLUMN,
    OUTCOME_COLUMN,
    FULL_GES_COLUMN,
] + [
    specification["column"]
    for specification in COMPARATORS.values()
]

missing_columns = [
    column
    for column in required_columns
    if column
    not in parquet_file.schema_arrow.names
]

assert not missing_columns, (
    f"Missing required Stage 6B columns: "
    f"{missing_columns}"
)

cohort = pd.read_parquet(
    EVALUABLE_PATH,
    columns=required_columns,
).copy()

assert len(cohort) == EXPECTED_ROWS

assert (
    cohort[
        RCV_COLUMN
    ].notna().all()
)

assert (
    cohort[
        RCV_COLUMN
    ].nunique(
        dropna=False
    )
    == EXPECTED_ROWS
)

row_order = pd.to_numeric(
    cohort[
        ROW_ORDER_COLUMN
    ],
    errors="raise",
).to_numpy()

assert (
    len(
        np.unique(
            row_order
        )
    )
    == EXPECTED_ROWS
)

assert np.all(
    np.diff(
        row_order
    )
    > 0
)

cohort[
    OUTCOME_COLUMN
] = pd.to_numeric(
    cohort[
        OUTCOME_COLUMN
    ],
    errors="raise",
).astype(
    np.int8
)

outcome = cohort[
    OUTCOME_COLUMN
].to_numpy(
    dtype=np.int8
)

assert set(
    np.unique(
        outcome
    ).tolist()
) == {
    0,
    1,
}

events = int(
    outcome.sum()
)

negatives = int(
    np.sum(
        outcome == 0
    )
)

assert (
    events,
    negatives,
) == (
    EXPECTED_EVENTS,
    EXPECTED_NEGATIVES,
)

prevalence = float(
    outcome.mean()
)


# --------------------------------------------------------------------------------------------------
# 10. VALIDATE SCORE ARRAYS
# --------------------------------------------------------------------------------------------------

score_arrays = OrderedDict()

score_arrays[
    "full_ges"
] = pd.to_numeric(
    cohort[
        FULL_GES_COLUMN
    ],
    errors="raise",
).to_numpy(
    dtype=float
)

for (
    comparator_key,
    comparator_specification,
) in COMPARATORS.items():

    score_arrays[
        comparator_key
    ] = pd.to_numeric(
        cohort[
            comparator_specification[
                "column"
            ]
        ],
        errors="raise",
    ).to_numpy(
        dtype=float
    )

for (
    score_key,
    score_values,
) in score_arrays.items():

    assert np.isfinite(
        score_values
    ).all(), (
        f"Nonfinite score values: "
        f"{score_key}"
    )

    assert np.all(
        (
            score_values >= 0
        )
        & (
            score_values <= 1
        )
    ), (
        f"Score outside [0,1]: "
        f"{score_key}"
    )

    assert (
        np.unique(
            score_values
        ).size
        >= 2
    ), (
        f"Constant score: "
        f"{score_key}"
    )


# --------------------------------------------------------------------------------------------------
# 11. LOCKED POINT ESTIMATES
# --------------------------------------------------------------------------------------------------

point_estimates = {
    "AUPRC": OrderedDict(),
    "AUROC": OrderedDict(),
}

for (
    score_key,
    score_values,
) in score_arrays.items():

    point_estimates[
        "AUPRC"
    ][
        score_key
    ] = float(
        average_precision_score(
            outcome,
            score_values,
        )
    )

    point_estimates[
        "AUROC"
    ][
        score_key
    ] = float(
        roc_auc_score(
            outcome,
            score_values,
        )
    )


# --------------------------------------------------------------------------------------------------
# 12. VERIFY HISTORICAL POINT DIFFERENCES
# --------------------------------------------------------------------------------------------------

point_verification_rows = []

for (
    metric,
    comparator_key,
), historical_values in (
    HISTORICAL_RESULTS.items()
):

    observed_difference = float(
        point_estimates[
            metric
        ][
            "full_ges"
        ]
        - point_estimates[
            metric
        ][
            comparator_key
        ]
    )

    historical_difference = float(
        historical_values[
            "point_difference"
        ]
    )

    difference_from_report = (
        observed_difference
        - historical_difference
    )

    passed = bool(
        abs(
            difference_from_report
        )
        <= 2.0e-6
    )

    point_verification_rows.append(
        {
            "metric": metric,
            "comparator_key": comparator_key,
            "comparator": COMPARATORS[
                comparator_key
            ][
                "display"
            ],
            "full_ges_point_estimate": point_estimates[
                metric
            ][
                "full_ges"
            ],
            "comparator_point_estimate": point_estimates[
                metric
            ][
                comparator_key
            ],
            "observed_full_minus_comparator": observed_difference,
            "historical_reported_difference": historical_difference,
            "difference_from_report": difference_from_report,
            "passed": passed,
        }
    )

point_verification = pd.DataFrame(
    point_verification_rows
)

assert point_verification[
    "passed"
].all(), (
    "At least one locked point difference does not reproduce "
    "the rounded historical result."
)


# --------------------------------------------------------------------------------------------------
# 13. INDEPENDENT 2,000-REPLICATE PAIRED BOOTSTRAP
# --------------------------------------------------------------------------------------------------

model_keys = list(
    score_arrays.keys()
)

bootstrap_seed_values = (
    np.random.SeedSequence(
        ROOT_SEED
    ).generate_state(
        BOOTSTRAP_ATTEMPTS,
        dtype=np.uint64,
    )
)

assert (
    len(
        bootstrap_seed_values
    )
    == BOOTSTRAP_ATTEMPTS
)

assert (
    len(
        np.unique(
            bootstrap_seed_values
        )
    )
    == BOOTSTRAP_ATTEMPTS
)


def run_bootstrap_replicate(
    replicate_number,
    replicate_seed,
):
    replicate_rng = (
        np.random.default_rng(
            int(
                replicate_seed
            )
        )
    )

    sampled_indexes = (
        replicate_rng.integers(
            low=0,
            high=EXPECTED_ROWS,
            size=EXPECTED_ROWS,
            dtype=np.int64,
        )
    )

    bootstrap_outcome = (
        outcome[
            sampled_indexes
        ]
    )

    if (
        np.unique(
            bootstrap_outcome
        ).size
        != 2
    ):
        return {
            "replicate": int(
                replicate_number
            ),
            "replicate_seed": int(
                replicate_seed
            ),
            "valid_two_class_replicate": False,
        }

    result = {
        "replicate": int(
            replicate_number
        ),
        "replicate_seed": int(
            replicate_seed
        ),
        "valid_two_class_replicate": True,
    }

    for model_key in model_keys:
        sampled_scores = (
            score_arrays[
                model_key
            ][
                sampled_indexes
            ]
        )

        result[
            f"auprc_{model_key}"
        ] = float(
            average_precision_score(
                bootstrap_outcome,
                sampled_scores,
            )
        )

        result[
            f"auroc_{model_key}"
        ] = float(
            roc_auc_score(
                bootstrap_outcome,
                sampled_scores,
            )
        )

    return result


print(
    "\nRunning independent remaining-comparator "
    f"bootstrap: {BOOTSTRAP_ATTEMPTS:,} replicates..."
)

bootstrap_start = time.perf_counter()

bootstrap_rows = Parallel(
    n_jobs=N_JOBS,
    prefer="threads",
    batch_size=5,
    pre_dispatch="2*n_jobs",
    verbose=10,
)(
    delayed(
        run_bootstrap_replicate
    )(
        replicate_number,
        replicate_seed,
    )
    for (
        replicate_number,
        replicate_seed,
    ) in enumerate(
        bootstrap_seed_values,
        start=1,
    )
)

bootstrap_elapsed_seconds = float(
    time.perf_counter()
    - bootstrap_start
)

replicates = (
    pd.DataFrame(
        bootstrap_rows
    )
    .sort_values(
        "replicate",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

assert replicates[
    "replicate"
].tolist() == list(
    range(
        1,
        BOOTSTRAP_ATTEMPTS + 1,
    )
)

valid_mask = (
    replicates[
        "valid_two_class_replicate"
    ].astype(
        bool
    )
)

valid_replicates = int(
    valid_mask.sum()
)

invalid_replicates = int(
    (
        ~valid_mask
    ).sum()
)

assert (
    valid_replicates
    == BOOTSTRAP_ATTEMPTS
)

assert invalid_replicates == 0

bootstrap_metric_columns = [
    f"{metric}_{model_key}"
    for metric in [
        "auprc",
        "auroc",
    ]
    for model_key in model_keys
]

assert (
    replicates[
        bootstrap_metric_columns
    ].notna().all().all()
)

assert np.isfinite(
    replicates[
        bootstrap_metric_columns
    ].to_numpy(
        dtype=float
    )
).all()


# --------------------------------------------------------------------------------------------------
# 14. MODEL BOOTSTRAP INTERVAL TABLE
# --------------------------------------------------------------------------------------------------

model_interval_rows = []

for metric in [
    "AUPRC",
    "AUROC",
]:
    for model_key in model_keys:
        bootstrap_values = (
            replicates.loc[
                valid_mask,
                f"{metric.lower()}_{model_key}",
            ].to_numpy(
                dtype=float
            )
        )

        ci_low, ci_high = (
            percentile_interval(
                bootstrap_values
            )
        )

        if model_key == "full_ges":
            model_display = "Full GES"
            score_column = FULL_GES_COLUMN
        else:
            model_display = (
                COMPARATORS[
                    model_key
                ][
                    "display"
                ]
            )

            score_column = (
                COMPARATORS[
                    model_key
                ][
                    "column"
                ]
            )

        model_interval_rows.append(
            {
                "metric": metric,
                "model_key": model_key,
                "model": model_display,
                "score_column": score_column,
                "point_estimate": point_estimates[
                    metric
                ][
                    model_key
                ],
                "bootstrap_mean": float(
                    bootstrap_values.mean()
                ),
                "bootstrap_standard_error": float(
                    bootstrap_values.std(
                        ddof=1
                    )
                ),
                "percentile_95_ci_low": ci_low,
                "percentile_95_ci_high": ci_high,
                "bootstrap_attempts": BOOTSTRAP_ATTEMPTS,
                "valid_replicates": valid_replicates,
                "invalid_replicates": invalid_replicates,
                "root_seed": ROOT_SEED,
                "materialization_sequence": (
                    "independent_seedsequence_generate_state_"
                    "per_replicate_default_rng"
                ),
            }
        )

model_intervals = pd.DataFrame(
    model_interval_rows
)

assert len(
    model_intervals
) == 12


# --------------------------------------------------------------------------------------------------
# 15. PAIRED COMPARATOR INFERENCE
# --------------------------------------------------------------------------------------------------

paired_rows = []

for metric in [
    "AUPRC",
    "AUROC",
]:
    for comparator_key in (
        COMPARATORS.keys()
    ):
        full_values = (
            replicates.loc[
                valid_mask,
                f"{metric.lower()}_full_ges",
            ].to_numpy(
                dtype=float
            )
        )

        comparator_values = (
            replicates.loc[
                valid_mask,
                f"{metric.lower()}_{comparator_key}",
            ].to_numpy(
                dtype=float
            )
        )

        differences = (
            full_values
            - comparator_values
        )

        ci_low, ci_high = (
            percentile_interval(
                differences
            )
        )

        sign_result = (
            finite_two_sided_sign_probability(
                differences
            )
        )

        paired_rows.append(
            {
                "metric": metric,
                "comparator_key": comparator_key,
                "comparator": COMPARATORS[
                    comparator_key
                ][
                    "display"
                ],
                "comparison": (
                    "Full GES minus "
                    + COMPARATORS[
                        comparator_key
                    ][
                        "display"
                    ]
                ),
                "full_ges_point_estimate": point_estimates[
                    metric
                ][
                    "full_ges"
                ],
                "comparator_point_estimate": point_estimates[
                    metric
                ][
                    comparator_key
                ],
                "point_difference": float(
                    point_estimates[
                        metric
                    ][
                        "full_ges"
                    ]
                    - point_estimates[
                        metric
                    ][
                        comparator_key
                    ]
                ),
                "bootstrap_mean_difference": float(
                    differences.mean()
                ),
                "bootstrap_standard_error": float(
                    differences.std(
                        ddof=1
                    )
                ),
                "percentile_95_ci_low": ci_low,
                "percentile_95_ci_high": ci_high,
                "fraction_difference_gt_zero": float(
                    np.mean(
                        differences > 0
                    )
                ),
                "fraction_difference_lt_zero": float(
                    np.mean(
                        differences < 0
                    )
                ),
                "zero_differences": sign_result[
                    "zero"
                ],
                "finite_two_sided_sign_probability": sign_result[
                    "probability"
                ],
                "ci_excludes_zero": bool(
                    ci_low > 0
                    or ci_high < 0
                ),
                "interval_conclusion": interval_conclusion(
                    ci_low,
                    ci_high,
                ),
                "bootstrap_attempts": BOOTSTRAP_ATTEMPTS,
                "valid_replicates": valid_replicates,
                "invalid_replicates": invalid_replicates,
                "root_seed": ROOT_SEED,
            }
        )

paired_inference = pd.DataFrame(
    paired_rows
)

assert len(
    paired_inference
) == 10


# --------------------------------------------------------------------------------------------------
# 16. HOLM CORRECTION SEPARATELY BY METRIC
# --------------------------------------------------------------------------------------------------

paired_inference[
    "holm_family"
] = ""

paired_inference[
    "holm_family_size"
] = 0

paired_inference[
    "holm_adjusted_sign_probability"
] = np.nan

for metric in [
    "AUPRC",
    "AUROC",
]:
    metric_mask = (
        paired_inference[
            "metric"
        ].eq(
            metric
        )
    )

    raw_probabilities = (
        paired_inference.loc[
            metric_mask,
            "finite_two_sided_sign_probability",
        ].to_numpy(
            dtype=float
        )
    )

    adjusted_probabilities = (
        holm_adjust(
            raw_probabilities
        )
    )

    paired_inference.loc[
        metric_mask,
        "holm_family",
    ] = (
        f"remaining_comparator_{metric.lower()}"
    )

    paired_inference.loc[
        metric_mask,
        "holm_family_size",
    ] = len(
        raw_probabilities
    )

    paired_inference.loc[
        metric_mask,
        "holm_adjusted_sign_probability",
    ] = adjusted_probabilities

paired_inference[
    "holm_reject_at_0_05"
] = (
    paired_inference[
        "holm_adjusted_sign_probability"
    ]
    <= 0.05
)

assert (
    paired_inference[
        "holm_family_size"
    ]
    .eq(5)
    .all()
)


# --------------------------------------------------------------------------------------------------
# 17. HISTORICAL RESULT TABLE
# --------------------------------------------------------------------------------------------------

historical_rows = []

for (
    metric,
    comparator_key,
), historical_values in (
    HISTORICAL_RESULTS.items()
):

    historical_rows.append(
        {
            "metric": metric,
            "comparator_key": comparator_key,
            "comparator": COMPARATORS[
                comparator_key
            ][
                "display"
            ],
            "comparison": (
                "Full GES minus "
                + COMPARATORS[
                    comparator_key
                ][
                    "display"
                ]
            ),
            "historical_point_difference": historical_values[
                "point_difference"
            ],
            "historical_percentile_95_ci_low": historical_values[
                "ci_low"
            ],
            "historical_percentile_95_ci_high": historical_values[
                "ci_high"
            ],
            "historical_holm_adjusted_probability": historical_values[
                "holm_adjusted_probability"
            ],
            "historical_interval_conclusion": interval_conclusion(
                historical_values[
                    "ci_low"
                ],
                historical_values[
                    "ci_high"
                ],
            ),
            "historical_value_precision": (
                "rounded_values_recorded_in_technical_report"
            ),
            "historical_bootstrap_attempts": BOOTSTRAP_ATTEMPTS,
            "historical_root_seed": ROOT_SEED,
            "historical_row_level_replicate_artifact_available": False,
        }
    )

historical_results = pd.DataFrame(
    historical_rows
)

assert len(
    historical_results
) == 10


# --------------------------------------------------------------------------------------------------
# 18. HISTORICAL VERSUS REPRODUCED CONCORDANCE
# --------------------------------------------------------------------------------------------------

concordance = historical_results.merge(
    paired_inference,
    on=[
        "metric",
        "comparator_key",
        "comparator",
        "comparison",
    ],
    how="inner",
    validate="one_to_one",
)

assert len(
    concordance
) == 10

concordance[
    "point_difference_delta"
] = (
    concordance[
        "point_difference"
    ]
    - concordance[
        "historical_point_difference"
    ]
)

concordance[
    "point_difference_reproduced"
] = (
    concordance[
        "point_difference_delta"
    ].abs()
    <= 2.0e-6
)

concordance[
    "reproduced_interval_conclusion"
] = concordance.apply(
    lambda row: interval_conclusion(
        row[
            "percentile_95_ci_low"
        ],
        row[
            "percentile_95_ci_high"
        ],
    ),
    axis=1,
)

concordance[
    "interval_conclusion_concordant"
] = (
    concordance[
        "historical_interval_conclusion"
    ]
    == concordance[
        "reproduced_interval_conclusion"
    ]
)

concordance[
    "historical_holm_significant"
] = (
    concordance[
        "historical_holm_adjusted_probability"
    ]
    <= 0.05
)

concordance[
    "reproduced_holm_significant"
] = (
    concordance[
        "holm_adjusted_sign_probability"
    ]
    <= 0.05
)

concordance[
    "holm_conclusion_concordant"
] = (
    concordance[
        "historical_holm_significant"
    ]
    == concordance[
        "reproduced_holm_significant"
    ]
)

concordance[
    "scientific_conclusion_concordant"
] = (
    concordance[
        "point_difference_reproduced"
    ]
    & concordance[
        "interval_conclusion_concordant"
    ]
    & concordance[
        "holm_conclusion_concordant"
    ]
)

assert concordance[
    "point_difference_reproduced"
].all()

assert concordance[
    "interval_conclusion_concordant"
].all()

assert concordance[
    "holm_conclusion_concordant"
].all()

assert concordance[
    "scientific_conclusion_concordant"
].all()

assert paired_inference[
    "ci_excludes_zero"
].all()

assert paired_inference[
    "interval_conclusion"
].eq(
    "full_ges_higher"
).all()

assert paired_inference[
    "holm_reject_at_0_05"
].all()


# --------------------------------------------------------------------------------------------------
# 19. WRITE SCIENTIFIC TABLES
# --------------------------------------------------------------------------------------------------

write_parquet(
    replicates,
    REPLICATE_PATH,
)

write_csv(
    model_intervals,
    MODEL_INTERVAL_PATH,
)

write_csv(
    paired_inference,
    PAIRED_INFERENCE_PATH,
)

write_csv(
    historical_results,
    HISTORICAL_PATH,
)

write_csv(
    concordance,
    CONCORDANCE_PATH,
)

write_csv(
    point_verification,
    POINT_VERIFICATION_PATH,
)


# --------------------------------------------------------------------------------------------------
# 20. CREATE SIDECARS
# --------------------------------------------------------------------------------------------------

replicate_sidecar = write_sidecar(
    REPLICATE_PATH
)

model_interval_sidecar = write_sidecar(
    MODEL_INTERVAL_PATH
)

paired_inference_sidecar = write_sidecar(
    PAIRED_INFERENCE_PATH
)

historical_sidecar = write_sidecar(
    HISTORICAL_PATH
)

concordance_sidecar = write_sidecar(
    CONCORDANCE_PATH
)

point_verification_sidecar = write_sidecar(
    POINT_VERIFICATION_PATH
)


# --------------------------------------------------------------------------------------------------
# 21. FRESH READBACK
# --------------------------------------------------------------------------------------------------

fresh_replicates = pd.read_parquet(
    REPLICATE_PATH
)

fresh_model_intervals = pd.read_csv(
    MODEL_INTERVAL_PATH
)

fresh_paired_inference = pd.read_csv(
    PAIRED_INFERENCE_PATH
)

fresh_historical_results = pd.read_csv(
    HISTORICAL_PATH
)

fresh_concordance = pd.read_csv(
    CONCORDANCE_PATH
)

fresh_point_verification = pd.read_csv(
    POINT_VERIFICATION_PATH
)

pd.testing.assert_frame_equal(
    fresh_replicates,
    replicates,
    check_exact=True,
)

assert len(
    fresh_model_intervals
) == 12

assert len(
    fresh_paired_inference
) == 10

assert len(
    fresh_historical_results
) == 10

assert len(
    fresh_concordance
) == 10

assert len(
    fresh_point_verification
) == 10

assert fresh_paired_inference[
    "ci_excludes_zero"
].astype(
    bool
).all()

assert fresh_paired_inference[
    "holm_reject_at_0_05"
].astype(
    bool
).all()

assert fresh_concordance[
    "scientific_conclusion_concordant"
].astype(
    bool
).all()

result_sidecar_pairs = [
    (
        REPLICATE_PATH,
        replicate_sidecar,
    ),
    (
        MODEL_INTERVAL_PATH,
        model_interval_sidecar,
    ),
    (
        PAIRED_INFERENCE_PATH,
        paired_inference_sidecar,
    ),
    (
        HISTORICAL_PATH,
        historical_sidecar,
    ),
    (
        CONCORDANCE_PATH,
        concordance_sidecar,
    ),
    (
        POINT_VERIFICATION_PATH,
        point_verification_sidecar,
    ),
]

for (
    result_path,
    sidecar_path,
) in result_sidecar_pairs:

    assert (
        read_sidecar_hash(
            sidecar_path
        )
        == sha256_file(
            result_path
        )
    )


# --------------------------------------------------------------------------------------------------
# 22. QC
# --------------------------------------------------------------------------------------------------

checks = OrderedDict(
    [
        (
            "stage6b_evaluable_hash_verified",
            (
                observed_evaluable_hash
                == EXPECTED_EVALUABLE_SHA256
            ),
        ),
        (
            "stage6b_evaluable_sidecar_verified",
            True,
        ),
        (
            "stage6b_dimensions_verified",
            (
                parquet_file.metadata.num_rows
                == EXPECTED_ROWS
                and parquet_file.metadata.num_columns
                == EXPECTED_COLUMNS
            ),
        ),
        (
            "prior_6c_4c0_manifest_verified",
            True,
        ),
        (
            "cohort_keys_and_row_order_verified",
            True,
        ),
        (
            "outcome_accounting_verified",
            (
                events == EXPECTED_EVENTS
                and negatives
                == EXPECTED_NEGATIVES
            ),
        ),
        (
            "six_scores_finite_in_range_and_nonconstant",
            True,
        ),
        (
            "ten_historical_point_differences_reproduced",
            bool(
                point_verification[
                    "passed"
                ].all()
            ),
        ),
        (
            "all_2000_bootstrap_replicates_present",
            (
                len(replicates)
                == BOOTSTRAP_ATTEMPTS
            ),
        ),
        (
            "all_2000_bootstrap_replicates_valid",
            (
                valid_replicates
                == BOOTSTRAP_ATTEMPTS
                and invalid_replicates == 0
            ),
        ),
        (
            "bootstrap_metrics_complete_and_finite",
            True,
        ),
        (
            "twelve_model_interval_rows_created",
            (
                len(model_intervals)
                == 12
            ),
        ),
        (
            "ten_paired_inference_rows_created",
            (
                len(paired_inference)
                == 10
            ),
        ),
        (
            "holm_applied_separately_to_two_five_test_families",
            bool(
                paired_inference[
                    "holm_family_size"
                ].eq(5).all()
            ),
        ),
        (
            "all_ten_paired_intervals_favor_full_ges",
            bool(
                paired_inference[
                    "interval_conclusion"
                ].eq(
                    "full_ges_higher"
                ).all()
            ),
        ),
        (
            "all_ten_holm_corrected_comparisons_significant",
            bool(
                paired_inference[
                    "holm_reject_at_0_05"
                ].all()
            ),
        ),
        (
            "all_ten_scientific_conclusions_concordant",
            bool(
                concordance[
                    "scientific_conclusion_concordant"
                ].all()
            ),
        ),
        (
            "six_scientific_tables_written",
            all(
                path.exists()
                for path in [
                    REPLICATE_PATH,
                    MODEL_INTERVAL_PATH,
                    PAIRED_INFERENCE_PATH,
                    HISTORICAL_PATH,
                    CONCORDANCE_PATH,
                    POINT_VERIFICATION_PATH,
                ]
            ),
        ),
        (
            "six_scientific_tables_read_back",
            True,
        ),
        (
            "six_scientific_table_sidecars_verified",
            True,
        ),
        (
            "historical_results_preserved_separately",
            True,
        ),
        (
            "independent_materialization_results_preserved_separately",
            True,
        ),
        (
            "no_frozen_scientific_input_modified",
            True,
        ),
        (
            "experiment_2_not_started",
            True,
        ),
    ]
)

assert all(
    checks.values()
)

qc_payload = {
    "schema_version": "1.0",
    "cell": "6C-4D0",
    "stage": "Stage 6C Step 4D",
    "notebook_filename": NOTEBOOK_FILENAME,
    "package_name": PACKAGE_NAME,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "checks_total": len(
        checks
    ),
    "checks_passed": int(
        sum(
            checks.values()
        )
    ),
    "checks_failed": int(
        len(
            checks
        )
        - sum(
            checks.values()
        )
    ),
    "checks": checks,
    "source": {
        "path": str(
            EVALUABLE_PATH
        ),
        "sha256": observed_evaluable_hash,
        "sidecar_path": str(
            EVALUABLE_SIDECAR
        ),
        "rows": EXPECTED_ROWS,
        "columns": EXPECTED_COLUMNS,
        "events": events,
        "negatives": negatives,
        "prevalence": prevalence,
    },
    "prior_completed_category": {
        "manifest_path": str(
            PRIOR_MANIFEST_PATH
        ),
        "manifest_sha256": prior_manifest_hash,
        "category": (
            "principal_auprc_auroc_bootstrap"
        ),
        "verified": True,
    },
    "bootstrap": {
        "method": (
            "independent_nonparametric_paired_row_bootstrap"
        ),
        "attempts": BOOTSTRAP_ATTEMPTS,
        "valid_replicates": valid_replicates,
        "invalid_replicates": invalid_replicates,
        "root_seed": ROOT_SEED,
        "seed_generation": (
            "numpy.random.SeedSequence.generate_state"
        ),
        "per_replicate_generator": (
            "numpy.random.default_rng"
        ),
        "per_replicate_bit_generator": (
            "PCG64"
        ),
        "n_jobs": N_JOBS,
        "elapsed_seconds": bootstrap_elapsed_seconds,
        "identical_resamples_across_models": True,
        "identical_resamples_across_metrics": True,
    },
    "multiplicity": {
        "method": "Holm",
        "families": [
            {
                "metric": "AUPRC",
                "number_of_tests": 5,
            },
            {
                "metric": "AUROC",
                "number_of_tests": 5,
            },
        ],
        "raw_probability": (
            "finite_two_sided_bootstrap_sign_probability"
        ),
    },
    "historical_comparison": {
        "historical_values_are_rounded": True,
        "historical_row_level_replicate_artifact_available": False,
        "exact_historical_rng_stream_claimed": False,
        "locked_point_estimates_reproduced": True,
        "paired_scientific_conclusions_concordant": 10,
        "paired_scientific_conclusions_total": 10,
    },
    "scientific_boundary": {
        "scores_modified": False,
        "outcomes_modified": False,
        "models_modified": False,
        "features_modified": False,
        "comparator_formulas_modified": False,
        "score_directions_modified": False,
        "weights_modified": False,
        "thresholds_modified": False,
        "linkage_decisions_modified": False,
        "row_order_modified": False,
        "cohort_membership_modified": False,
        "historical_results_overwritten": False,
        "experiment_2_started": False,
    },
    "software_versions": {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "pyarrow": pyarrow.__version__,
        "scikit_learn": sklearn.__version__,
        "joblib": joblib.__version__,
    },
}

write_json(
    qc_payload,
    QC_PATH,
)

qc_sidecar = write_sidecar(
    QC_PATH
)


# --------------------------------------------------------------------------------------------------
# 23. FREEZE MANIFEST
# --------------------------------------------------------------------------------------------------

artifacts = [
    artifact_entry(
        REPLICATE_PATH,
        replicate_sidecar,
        "independent remaining-comparator bootstrap replicates",
    ),
    artifact_entry(
        MODEL_INTERVAL_PATH,
        model_interval_sidecar,
        "remaining-comparator model bootstrap intervals",
    ),
    artifact_entry(
        PAIRED_INFERENCE_PATH,
        paired_inference_sidecar,
        "remaining-comparator paired inference and Holm correction",
    ),
    artifact_entry(
        HISTORICAL_PATH,
        historical_sidecar,
        "historical reported remaining-comparator inference",
    ),
    artifact_entry(
        CONCORDANCE_PATH,
        concordance_sidecar,
        "historical versus reproduced conclusion concordance",
    ),
    artifact_entry(
        POINT_VERIFICATION_PATH,
        point_verification_sidecar,
        "locked remaining-comparator point-estimate verification",
    ),
    artifact_entry(
        QC_PATH,
        qc_sidecar,
        "Cell 6C-4D0 QC",
    ),
]

manifest_payload = {
    "schema_version": "1.0",
    "cell": "6C-4D0",
    "stage": "Stage 6C Step 4D",
    "notebook_filename": NOTEBOOK_FILENAME,
    "package_name": PACKAGE_NAME,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "required_final_freeze_category": (
        "remaining_comparator_inference"
    ),
    "category_status": (
        "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
    ),
    "source_artifact": {
        "path": str(
            EVALUABLE_PATH
        ),
        "sha256": observed_evaluable_hash,
        "sidecar_path": str(
            EVALUABLE_SIDECAR
        ),
        "rows": EXPECTED_ROWS,
        "columns": EXPECTED_COLUMNS,
    },
    "comparators": {
        comparator_key: {
            "display": specification[
                "display"
            ],
            "score_column": specification[
                "column"
            ],
        }
        for (
            comparator_key,
            specification,
        ) in COMPARATORS.items()
    },
    "analysis": {
        "primary_model": "Full GES",
        "full_ges_score_column": FULL_GES_COLUMN,
        "outcome_column": OUTCOME_COLUMN,
        "metrics": [
            "AUPRC",
            "AUROC",
        ],
        "bootstrap_attempts": BOOTSTRAP_ATTEMPTS,
        "root_seed": ROOT_SEED,
        "confidence_interval": (
            "2.5th_to_97.5th_percentile"
        ),
        "paired_resampling": True,
        "multiplicity_method": "Holm",
        "multiplicity_families": {
            "AUPRC": 5,
            "AUROC": 5,
        },
        "historical_results_preserved": True,
        "independent_reproduction_preserved": True,
        "exact_historical_rng_stream_claimed": False,
        "scientific_conclusions_concordant": {
            "concordant": 10,
            "total": 10,
        },
    },
    "artifacts": artifacts,
    "decision": (
        "PASS_STAGE6C_REMAINING_COMPARATOR_INFERENCE_RESULT_CATEGORY_"
        "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
    ),
    "completed_independent_result_categories": 2,
    "remaining_independent_result_categories": 6,
    "scientific_boundary": {
        "frozen_inputs_modified": False,
        "historical_results_overwritten": False,
        "final_integrated_stage6c_freeze_completed": False,
        "experiment_2_started": False,
    },
}

write_json(
    manifest_payload,
    MANIFEST_PATH,
)

manifest_sidecar = write_sidecar(
    MANIFEST_PATH
)


# --------------------------------------------------------------------------------------------------
# 24. FINAL COMPLETE REVERIFICATION
# --------------------------------------------------------------------------------------------------

fresh_qc = json.loads(
    QC_PATH.read_text(
        encoding="utf-8"
    )
)

fresh_manifest = json.loads(
    MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

assert (
    fresh_qc[
        "checks_failed"
    ]
    == 0
)

assert (
    fresh_manifest[
        "required_final_freeze_category"
    ]
    == "remaining_comparator_inference"
)

assert (
    fresh_manifest[
        "remaining_independent_result_categories"
    ]
    == 6
)

assert (
    read_sidecar_hash(
        qc_sidecar
    )
    == sha256_file(
        QC_PATH
    )
)

assert (
    read_sidecar_hash(
        manifest_sidecar
    )
    == sha256_file(
        MANIFEST_PATH
    )
)

for artifact_record in (
    fresh_manifest[
        "artifacts"
    ]
):
    artifact_path = Path(
        artifact_record[
            "path"
        ]
    )

    artifact_sidecar = Path(
        artifact_record[
            "sidecar_path"
        ]
    )

    assert artifact_path.exists()
    assert artifact_sidecar.exists()

    assert (
        sha256_file(
            artifact_path
        )
        == artifact_record[
            "sha256"
        ]
    )

    assert (
        read_sidecar_hash(
            artifact_sidecar
        )
        == artifact_record[
            "sha256"
        ]
    )


# --------------------------------------------------------------------------------------------------
# 25. DISPLAY FINAL RESULT
# --------------------------------------------------------------------------------------------------

separator = "=" * 155

print(
    "\n" + separator
)

print(
    "STAGE 6C STEP 4D — CELL 6C-4D0 — "
    "REMAINING-COMPARATOR INFERENCE RESULT CATEGORY"
)

print(
    separator
)

print(
    f"Stage 6B source hash                 : "
    f"PASS ({observed_evaluable_hash})"
)

print(
    f"Prior Cell 6C-4C0 manifest           : "
    f"PASS ({prior_manifest_hash})"
)

print(
    f"Locked point differences             : "
    f"PASS ({int(point_verification['passed'].sum())}/"
    f"{len(point_verification)})"
)

print(
    f"Bootstrap attempts / valid           : "
    f"{BOOTSTRAP_ATTEMPTS:,} / {valid_replicates:,}"
)

print(
    f"Bootstrap elapsed                    : "
    f"{bootstrap_elapsed_seconds / 60:.2f} minutes"
)

print(
    f"Paired AUPRC/AUROC comparisons       : "
    f"{len(paired_inference)}"
)

print(
    f"Intervals favoring Full GES          : "
    f"PASS ({int(paired_inference['ci_excludes_zero'].sum())}/"
    f"{len(paired_inference)})"
)

print(
    f"Holm-corrected significant results   : "
    f"PASS ({int(paired_inference['holm_reject_at_0_05'].sum())}/"
    f"{len(paired_inference)})"
)

print(
    f"Scientific conclusions concordant    : "
    f"PASS ({int(concordance['scientific_conclusion_concordant'].sum())}/"
    f"{len(concordance)})"
)

print(
    f"Fresh QC                              : "
    f"PASS ({fresh_qc['checks_passed']}/"
    f"{fresh_qc['checks_total']})"
)

print(
    f"Bootstrap replicate table             : "
    f"{REPLICATE_PATH}"
)

print(
    f"Model interval table                  : "
    f"{MODEL_INTERVAL_PATH}"
)

print(
    f"Paired inference table                : "
    f"{PAIRED_INFERENCE_PATH}"
)

print(
    f"Historical inference table            : "
    f"{HISTORICAL_PATH}"
)

print(
    f"Concordance table                     : "
    f"{CONCORDANCE_PATH}"
)

print(
    f"QC                                    : "
    f"{QC_PATH}"
)

print(
    f"Manifest                              : "
    f"{MANIFEST_PATH}"
)

print(
    f"Manifest SHA-256                      : "
    f"{sha256_file(MANIFEST_PATH)}"
)


print(
    "\nREPRODUCED REMAINING-COMPARATOR INFERENCE"
)

print(
    paired_inference[
        [
            "metric",
            "comparator",
            "point_difference",
            "percentile_95_ci_low",
            "percentile_95_ci_high",
            "fraction_difference_gt_zero",
            "finite_two_sided_sign_probability",
            "holm_adjusted_sign_probability",
            "holm_reject_at_0_05",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: (
            f"{value:.12f}"
        ),
    )
)


print(
    "\nSCIENTIFIC-CONCLUSION CONCORDANCE"
)

print(
    concordance[
        [
            "metric",
            "comparator",
            "historical_interval_conclusion",
            "reproduced_interval_conclusion",
            "historical_holm_significant",
            "reproduced_holm_significant",
            "scientific_conclusion_concordant",
        ]
    ].to_string(
        index=False,
    )
)


print(
    "\nCELL DECISION"
)

print(
    "PASS_STAGE6C_REMAINING_COMPARATOR_INFERENCE_RESULT_CATEGORY_"
    "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
)

print(
    "The second pending Stage 6C result category is complete. "
    "Six independent result categories remain. "
    "Experiment 2 has not started."
)

print(
    separator
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Use this Colab notebook file name: GES_Stage6C_Cell_6C_4D0_Remaining_Comparator_Inference_Materialization.ipynb

Running independent remaining-comparator bootstrap: 2,000 replicates...


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   9 tasks      | elapsed:    2.2s
[Parallel(n_jobs=2)]: Done  24 tasks      | elapsed:    4.1s
[Parallel(n_jobs=2)]: Done  49 tasks      | elapsed:    7.7s
[Parallel(n_jobs=2)]: Done  74 tasks      | elapsed:    9.7s
[Parallel(n_jobs=2)]: Done 109 tasks      | elapsed:   13.7s
[Parallel(n_jobs=2)]: Done 144 tasks      | elapsed:   18.0s
[Parallel(n_jobs=2)]: Done 189 tasks      | elapsed:   24.0s
[Parallel(n_jobs=2)]: Done 234 tasks      | elapsed:   28.1s
[Parallel(n_jobs=2)]: Done 289 tasks      | elapsed:   36.3s
[Parallel(n_jobs=2)]: Done 344 tasks      | elapsed:   41.5s
[Parallel(n_jobs=2)]: Done 409 tasks      | elapsed:   50.5s
[Parallel(n_jobs=2)]: Done 474 tasks      | elapsed:   56.7s
[Parallel(n_jobs=2)]: Done 549 tasks      | elapsed:  1.1min
[Parallel(n_jobs=2)]: Done 624 tasks      | elapsed:  1.2min
[Parallel(n_jobs=2)]: Done 709 tasks      | elapsed:  1.4min
[Para


STAGE 6C STEP 4D — CELL 6C-4D0 — REMAINING-COMPARATOR INFERENCE RESULT CATEGORY
Stage 6B source hash                 : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038)
Prior Cell 6C-4C0 manifest           : PASS (5f260c7167ceb2b137d57119ecdf2d59f004a98cc2aab72b96435e760890e578)
Locked point differences             : PASS (10/10)
Bootstrap attempts / valid           : 2,000 / 2,000
Bootstrap elapsed                    : 3.95 minutes
Paired AUPRC/AUROC comparisons       : 10
Intervals favoring Full GES          : PASS (10/10)
Holm-corrected significant results   : PASS (10/10)
Scientific conclusions concordant    : PASS (10/10)
Fresh QC                              : PASS (24/24)
Bootstrap replicate table             : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/tables/stage6_temporal_validation/stage6c_4d0_remaining_comparator_inference_materialization_v1/stage6c_remaining_comparator_bootstrap_replicates_v1.parquet
Model interval table                  

In [12]:
# ==================================================================================================
# COLAB NOTEBOOK FILE NAME:
# GES_Stage6C_Cell_6C_4E0_Same_Star_Inference_Materialization.ipynb
#
# STAGE 6C STEP 4E — CELL 6C-4E0
# SAME-STAR POINT ESTIMATES, BOOTSTRAP INFERENCE, SPARSE-STRATUM AUDIT,
# MULTIPLICITY TABLES, MATERIALIZATION, AND FREEZE
#
# Purpose:
#   1. Freshly verify the immutable Stage 6B primary-evaluable cohort.
#   2. Verify the completed Cell 6C-4D0 result package.
#   3. Reconstruct the prespecified same-star strata using frozen T0 review stars.
#   4. Preserve the 0-star one-class stratum descriptively.
#   5. Calculate locked point estimates for stars 1, 2, and 3.
#   6. Run 2,000 exact ordinary row-bootstrap attempts per estimable stratum.
#   7. Use identical bootstrap samples across Full GES, No-star GES, and Combined Metadata.
#   8. Preserve one-class bootstrap attempts as invalid rather than silently replacing them.
#   9. Calculate model intervals and Full-GES-minus-comparator paired inference.
#  10. Apply Holm correction separately across six AUPRC and six AUROC comparisons.
#  11. Preserve historical rounded results separately and verify scientific-conclusion concordance.
#  12. Write versioned tables, SHA-256 sidecars, QC, and a freeze manifest.
#
# Scientific boundary:
#   - No score, outcome, review-star value, model, threshold, feature, comparator formula,
#     weight, linkage decision, row order, or cohort membership is changed.
#   - The 3-star stratum remains explicitly classified as extremely sparse.
#   - The 0-star stratum remains descriptive only because it has no negatives.
#   - Experiment 2 is not started.
# ==================================================================================================


# --------------------------------------------------------------------------------------------------
# 0. MOUNT GOOGLE DRIVE
# --------------------------------------------------------------------------------------------------

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False,
)


# --------------------------------------------------------------------------------------------------
# 1. IMPORTS
# --------------------------------------------------------------------------------------------------

from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path

import hashlib
import json
import os
import platform
import re
import sys
import time

import joblib
import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq
import sklearn

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# --------------------------------------------------------------------------------------------------
# 2. CONSTANTS
# --------------------------------------------------------------------------------------------------

NOTEBOOK_FILENAME = (
    "GES_Stage6C_Cell_6C_4E0_"
    "Same_Star_Inference_Materialization.ipynb"
)

print(
    f"Use this Colab notebook file name: "
    f"{NOTEBOOK_FILENAME}"
)

ROOT = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

STAGE6_DIR = (
    ROOT
    / "data_processed"
    / "stage6_temporal_validation"
)

EVALUABLE_PATH = (
    STAGE6_DIR
    / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)

EVALUABLE_SIDECAR = Path(
    str(EVALUABLE_PATH) + ".sha256"
)

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
)

EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151

OUTCOME_COLUMN = "primary_future_instability"
RCV_COLUMN = "rcv_accession"
ROW_ORDER_COLUMN = "t0_row_order"
STAR_COLUMN = "t0_aggregate_review_stars"

BOOTSTRAP_ATTEMPTS = 2_000
BOOTSTRAP_BATCH_SIZE = 50
ROOT_SEED = 42

MODELS = OrderedDict(
    [
        (
            "full_ges",
            {
                "column": "full_ges_instability_risk_t0",
                "display": "Full GES",
            },
        ),
        (
            "no_star_ges",
            {
                "column": "no_star_ges_instability_risk_t0",
                "display": "No-star GES",
            },
        ),
        (
            "combined_metadata",
            {
                "column": "combined_metadata_instability_risk",
                "display": "Combined metadata",
            },
        ),
    ]
)

COMPARATORS = OrderedDict(
    [
        (
            "no_star_ges",
            "No-star GES",
        ),
        (
            "combined_metadata",
            "Combined metadata",
        ),
    ]
)

EXPECTED_STRATA = {
    0: {
        "rows": 36,
        "events": 36,
        "negatives": 0,
        "status": "descriptive_one_class",
    },
    1: {
        "rows": 49_224,
        "events": 4_941,
        "negatives": 44_283,
        "status": "estimable",
    },
    2: {
        "rows": 9_223,
        "events": 1_506,
        "negatives": 7_717,
        "status": "estimable",
    },
    3: {
        "rows": 8_153,
        "events": 2,
        "negatives": 8_151,
        "status": "estimable_extremely_sparse",
    },
}


# --------------------------------------------------------------------------------------------------
# 3. HISTORICAL MODEL RESULTS
# --------------------------------------------------------------------------------------------------

HISTORICAL_MODEL_RESULTS = [
    {
        "stars": 1,
        "model_key": "full_ges",
        "auprc": 0.111256,
        "auprc_ci_low": 0.106629,
        "auprc_ci_high": 0.116832,
        "auroc": 0.509992,
        "auroc_ci_low": 0.501315,
        "auroc_ci_high": 0.518572,
    },
    {
        "stars": 1,
        "model_key": "no_star_ges",
        "auprc": 0.118062,
        "auprc_ci_low": 0.113179,
        "auprc_ci_high": 0.124022,
        "auroc": 0.521198,
        "auroc_ci_low": 0.512278,
        "auroc_ci_high": 0.529951,
    },
    {
        "stars": 1,
        "model_key": "combined_metadata",
        "auprc": 0.114910,
        "auprc_ci_low": 0.109985,
        "auprc_ci_high": 0.120904,
        "auroc": 0.515405,
        "auroc_ci_low": 0.506665,
        "auroc_ci_high": 0.523869,
    },
    {
        "stars": 2,
        "model_key": "full_ges",
        "auprc": 0.141797,
        "auprc_ci_low": 0.134488,
        "auprc_ci_high": 0.150002,
        "auroc": 0.464028,
        "auroc_ci_low": 0.449047,
        "auroc_ci_high": 0.478285,
    },
    {
        "stars": 2,
        "model_key": "no_star_ges",
        "auprc": 0.141431,
        "auprc_ci_low": 0.134092,
        "auprc_ci_high": 0.149792,
        "auroc": 0.461306,
        "auroc_ci_low": 0.446297,
        "auroc_ci_high": 0.475540,
    },
    {
        "stars": 2,
        "model_key": "combined_metadata",
        "auprc": 0.163817,
        "auprc_ci_low": 0.155008,
        "auprc_ci_high": 0.173966,
        "auroc": 0.544343,
        "auroc_ci_low": 0.530439,
        "auroc_ci_high": 0.556962,
    },
    {
        "stars": 3,
        "model_key": "full_ges",
        "auprc": 0.000189,
        "auprc_ci_low": 0.000126,
        "auprc_ci_high": 0.000530,
        "auroc": 0.026561,
        "auroc_ci_low": 0.023082,
        "auroc_ci_high": 0.030241,
    },
    {
        "stars": 3,
        "model_key": "no_star_ges",
        "auprc": 0.000203,
        "auprc_ci_low": 0.000128,
        "auprc_ci_high": 0.000606,
        "auroc": 0.109373,
        "auroc_ci_low": 0.043057,
        "auroc_ci_high": 0.177809,
    },
    {
        "stars": 3,
        "model_key": "combined_metadata",
        "auprc": 0.003433,
        "auprc_ci_low": 0.002137,
        "auprc_ci_high": 0.010203,
        "auroc": 0.947307,
        "auroc_ci_low": 0.941547,
        "auroc_ci_high": 0.953010,
    },
]


# --------------------------------------------------------------------------------------------------
# 4. HISTORICAL PAIRED RESULTS
# --------------------------------------------------------------------------------------------------

HISTORICAL_PAIRED_RESULTS = [
    {
        "stars": 1,
        "metric": "AUPRC",
        "comparator_key": "no_star_ges",
        "point_difference": -0.006806,
        "ci_low": -0.007740,
        "ci_high": -0.005792,
        "holm_adjusted_probability": 0.005997,
    },
    {
        "stars": 1,
        "metric": "AUROC",
        "comparator_key": "no_star_ges",
        "point_difference": -0.011206,
        "ci_low": -0.012354,
        "ci_high": -0.010094,
        "holm_adjusted_probability": 0.005997,
    },
    {
        "stars": 1,
        "metric": "AUPRC",
        "comparator_key": "combined_metadata",
        "point_difference": -0.003655,
        "ci_low": -0.005116,
        "ci_high": -0.002110,
        "holm_adjusted_probability": 0.005997,
    },
    {
        "stars": 1,
        "metric": "AUROC",
        "comparator_key": "combined_metadata",
        "point_difference": -0.005413,
        "ci_low": -0.007466,
        "ci_high": -0.003308,
        "holm_adjusted_probability": 0.005997,
    },
    {
        "stars": 2,
        "metric": "AUPRC",
        "comparator_key": "no_star_ges",
        "point_difference": 0.000366,
        "ci_low": -0.000592,
        "ci_high": 0.000844,
        "holm_adjusted_probability": 0.401799,
    },
    {
        "stars": 2,
        "metric": "AUROC",
        "comparator_key": "no_star_ges",
        "point_difference": 0.002722,
        "ci_low": 0.001641,
        "ci_high": 0.003875,
        "holm_adjusted_probability": 0.005997,
    },
    {
        "stars": 2,
        "metric": "AUPRC",
        "comparator_key": "combined_metadata",
        "point_difference": -0.022020,
        "ci_low": -0.026346,
        "ci_high": -0.018965,
        "holm_adjusted_probability": 0.005997,
    },
    {
        "stars": 2,
        "metric": "AUROC",
        "comparator_key": "combined_metadata",
        "point_difference": -0.080314,
        "ci_low": -0.088438,
        "ci_high": -0.072424,
        "holm_adjusted_probability": 0.005997,
    },
    {
        "stars": 3,
        "metric": "AUPRC",
        "comparator_key": "no_star_ges",
        "point_difference": -0.000014,
        "ci_low": -0.000067,
        "ci_high": -0.000002,
        "holm_adjusted_probability": 0.005997,
    },
    {
        "stars": 3,
        "metric": "AUROC",
        "comparator_key": "no_star_ges",
        "point_difference": -0.082812,
        "ci_low": -0.151006,
        "ci_high": -0.017664,
        "holm_adjusted_probability": 0.005997,
    },
    {
        "stars": 3,
        "metric": "AUPRC",
        "comparator_key": "combined_metadata",
        "point_difference": -0.003244,
        "ci_low": -0.009677,
        "ci_high": -0.002011,
        "holm_adjusted_probability": 0.005997,
    },
    {
        "stars": 3,
        "metric": "AUROC",
        "comparator_key": "combined_metadata",
        "point_difference": -0.920746,
        "ci_low": -0.926993,
        "ci_high": -0.914132,
        "holm_adjusted_probability": 0.005997,
    },
]


# --------------------------------------------------------------------------------------------------
# 5. OUTPUT PATHS
# --------------------------------------------------------------------------------------------------

PACKAGE_NAME = (
    "stage6c_4e0_same_star_"
    "inference_materialization_v1"
)

TABLE_DIR = (
    ROOT
    / "outputs"
    / "tables"
    / "stage6_temporal_validation"
    / PACKAGE_NAME
)

QC_DIR = (
    ROOT
    / "outputs"
    / "quality_checks"
    / "stage6_temporal_validation"
    / PACKAGE_NAME
)

CONFIG_DIR = (
    ROOT
    / "configs"
    / "stage6_temporal_validation"
    / PACKAGE_NAME
)

for directory in [
    TABLE_DIR,
    QC_DIR,
    CONFIG_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

STRATUM_INVENTORY_PATH = (
    TABLE_DIR
    / "stage6c_same_star_stratum_inventory_v1.csv"
)

POINT_ESTIMATE_PATH = (
    TABLE_DIR
    / "stage6c_same_star_point_estimates_v1.csv"
)

REPLICATE_PATH = (
    TABLE_DIR
    / "stage6c_same_star_bootstrap_replicates_v1.parquet"
)

MODEL_INTERVAL_PATH = (
    TABLE_DIR
    / "stage6c_same_star_model_bootstrap_intervals_v1.csv"
)

PAIRED_INFERENCE_PATH = (
    TABLE_DIR
    / "stage6c_same_star_paired_inference_v1.csv"
)

MULTIPLICITY_PATH = (
    TABLE_DIR
    / "stage6c_same_star_holm_multiplicity_v1.csv"
)

SPARSE_AUDIT_PATH = (
    TABLE_DIR
    / "stage6c_same_star_sparse_stratum_audit_v1.csv"
)

HISTORICAL_MODEL_PATH = (
    TABLE_DIR
    / "stage6c_same_star_historical_reported_model_intervals_v1.csv"
)

HISTORICAL_PAIRED_PATH = (
    TABLE_DIR
    / "stage6c_same_star_historical_reported_paired_inference_v1.csv"
)

CONCORDANCE_PATH = (
    TABLE_DIR
    / "stage6c_same_star_historical_vs_reproduced_concordance_v1.csv"
)

QC_PATH = (
    QC_DIR
    / "stage6c_4e0_same_star_inference_qc_v1.json"
)

MANIFEST_PATH = (
    CONFIG_DIR
    / "stage6c_4e0_same_star_inference_manifest_v1.json"
)


# --------------------------------------------------------------------------------------------------
# 6. PRIOR CELL 6C-4D0 MANIFEST
# --------------------------------------------------------------------------------------------------

PRIOR_MANIFEST_PATH = (
    ROOT
    / "configs"
    / "stage6_temporal_validation"
    / "stage6c_4d0_remaining_comparator_inference_materialization_v1"
    / "stage6c_4d0_remaining_comparator_inference_manifest_v1.json"
)

PRIOR_MANIFEST_SIDECAR = Path(
    str(PRIOR_MANIFEST_PATH) + ".sha256"
)


# --------------------------------------------------------------------------------------------------
# 7. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(
                chunk_size
            )

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def read_sidecar_hash(path):
    text = Path(path).read_text(
        encoding="utf-8"
    )

    matches = re.findall(
        r"\b[a-fA-F0-9]{64}\b",
        text,
    )

    if not matches:
        raise ValueError(
            f"No SHA-256 value found in sidecar: {path}"
        )

    return matches[0].lower()


def write_sidecar(path):
    path = Path(path)

    sidecar_path = Path(
        str(path) + ".sha256"
    )

    sidecar_path.write_text(
        f"{sha256_file(path)}  {path.name}\n",
        encoding="utf-8",
    )

    return sidecar_path


def to_native(value):
    if isinstance(value, dict):
        return {
            str(key): to_native(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            to_native(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, Path):
        return str(value)

    return value


def write_json(
    payload,
    path,
):
    path = Path(path)

    temporary_path = Path(
        str(path) + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            to_native(payload),
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        path,
    )


def write_csv(
    frame,
    path,
):
    path = Path(path)

    temporary_path = Path(
        str(path) + ".tmp"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
        float_format="%.17g",
    )

    os.replace(
        temporary_path,
        path,
    )


def write_parquet(
    frame,
    path,
):
    path = Path(path)

    temporary_path = Path(
        str(path) + ".tmp.parquet"
    )

    frame.to_parquet(
        temporary_path,
        index=False,
        engine="pyarrow",
        compression="zstd",
    )

    os.replace(
        temporary_path,
        path,
    )


def percentile_interval(values):
    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]

    low, high = np.percentile(
        values,
        [
            2.5,
            97.5,
        ],
    )

    return (
        float(low),
        float(high),
    )


def interval_conclusion(
    low,
    high,
):
    low = float(low)
    high = float(high)

    if low > 0:
        return "full_ges_higher"

    if high < 0:
        return "full_ges_lower"

    return "interval_includes_zero"


def model_null_conclusion(
    metric,
    low,
    high,
    prevalence,
):
    null_value = (
        float(prevalence)
        if metric == "AUPRC"
        else 0.50
    )

    if low > null_value:
        return "above_null"

    if high < null_value:
        return "below_null"

    return "includes_null"


def finite_two_sided_sign_probability(
    differences,
):
    differences = np.asarray(
        differences,
        dtype=float,
    )

    differences = differences[
        np.isfinite(differences)
    ]

    positive = int(
        np.sum(
            differences > 0
        )
    )

    negative = int(
        np.sum(
            differences < 0
        )
    )

    zero = int(
        np.sum(
            differences == 0
        )
    )

    valid = int(
        len(
            differences
        )
    )

    probability = min(
        1.0,
        (
            2.0
            * (
                min(
                    positive,
                    negative,
                )
                + 1
            )
            / (
                valid + 1
            )
        ),
    )

    return {
        "probability": float(
            probability
        ),
        "positive": positive,
        "negative": negative,
        "zero": zero,
        "valid": valid,
    }


def holm_adjust(probabilities):
    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    number = len(
        probabilities
    )

    order = np.argsort(
        probabilities,
        kind="mergesort",
    )

    sorted_probabilities = (
        probabilities[
            order
        ]
    )

    adjusted_sorted = np.empty(
        number,
        dtype=float,
    )

    running_maximum = 0.0

    for index, probability in enumerate(
        sorted_probabilities
    ):
        candidate = min(
            1.0,
            (
                number - index
            )
            * probability,
        )

        running_maximum = max(
            running_maximum,
            candidate,
        )

        adjusted_sorted[
            index
        ] = running_maximum

    adjusted = np.empty(
        number,
        dtype=float,
    )

    adjusted[
        order
    ] = adjusted_sorted

    return adjusted


def artifact_entry(
    path,
    sidecar,
    role,
):
    path = Path(path)
    sidecar = Path(sidecar)

    return {
        "role": role,
        "path": str(path),
        "sha256": sha256_file(
            path
        ),
        "bytes": int(
            path.stat().st_size
        ),
        "sidecar_path": str(
            sidecar
        ),
        "sidecar_sha256": sha256_file(
            sidecar
        ),
    }


# --------------------------------------------------------------------------------------------------
# 8. EXACT GROUPED WEIGHTED METRIC ENGINE
# --------------------------------------------------------------------------------------------------

def prepare_score_groups(
    outcome,
    score,
):
    outcome = np.asarray(
        outcome,
        dtype=np.int8,
    )

    score = np.asarray(
        score,
        dtype=float,
    )

    order = np.argsort(
        -score,
        kind="mergesort",
    )

    sorted_score = score[
        order
    ]

    sorted_outcome = outcome[
        order
    ]

    group_starts = np.concatenate(
        [
            np.array(
                [0],
                dtype=np.int64,
            ),
            (
                np.flatnonzero(
                    np.diff(
                        sorted_score
                    )
                    != 0
                )
                + 1
            ).astype(
                np.int64
            ),
        ]
    )

    return {
        "order": order,
        "sorted_outcome": sorted_outcome,
        "group_starts": group_starts,
        "unique_score_count": int(
            len(
                group_starts
            )
        ),
    }


def grouped_bootstrap_metrics(
    bootstrap_counts,
    group_specification,
):
    bootstrap_counts = np.asarray(
        bootstrap_counts,
        dtype=np.int64,
    )

    order = group_specification[
        "order"
    ]

    sorted_outcome = group_specification[
        "sorted_outcome"
    ]

    group_starts = group_specification[
        "group_starts"
    ]

    counts_sorted = (
        bootstrap_counts[
            :,
            order,
        ]
    )

    positive_sorted = (
        counts_sorted
        * sorted_outcome[
            None,
            :
        ]
    )

    negative_sorted = (
        counts_sorted
        * (
            1
            - sorted_outcome[
                None,
                :
            ]
        )
    )

    positive_by_group = np.add.reduceat(
        positive_sorted,
        group_starts,
        axis=1,
    )

    negative_by_group = np.add.reduceat(
        negative_sorted,
        group_starts,
        axis=1,
    )

    total_positive = (
        positive_by_group.sum(
            axis=1
        )
    )

    total_negative = (
        negative_by_group.sum(
            axis=1
        )
    )

    valid = (
        total_positive > 0
    ) & (
        total_negative > 0
    )

    cumulative_positive = np.cumsum(
        positive_by_group,
        axis=1,
    )

    cumulative_total = np.cumsum(
        (
            positive_by_group
            + negative_by_group
        ),
        axis=1,
    )

    precision = np.divide(
        cumulative_positive,
        cumulative_total,
        out=np.zeros_like(
            cumulative_positive,
            dtype=float,
        ),
        where=(
            cumulative_total > 0
        ),
    )

    auprc = np.divide(
        np.sum(
            precision
            * positive_by_group,
            axis=1,
        ),
        total_positive,
        out=np.full(
            len(
                total_positive
            ),
            np.nan,
            dtype=float,
        ),
        where=(
            total_positive > 0
        ),
    )

    cumulative_negative = np.cumsum(
        negative_by_group,
        axis=1,
    )

    negative_below_group = (
        total_negative[
            :,
            None,
        ]
        - cumulative_negative
    )

    auroc_numerator = np.sum(
        positive_by_group
        * (
            negative_below_group
            + 0.5
            * negative_by_group
        ),
        axis=1,
    )

    auroc_denominator = (
        total_positive
        * total_negative
    )

    auroc = np.divide(
        auroc_numerator,
        auroc_denominator,
        out=np.full(
            len(
                total_positive
            ),
            np.nan,
            dtype=float,
        ),
        where=(
            auroc_denominator > 0
        ),
    )

    auprc[
        ~valid
    ] = np.nan

    auroc[
        ~valid
    ] = np.nan

    return {
        "auprc": auprc,
        "auroc": auroc,
        "valid": valid,
        "total_positive": total_positive,
        "total_negative": total_negative,
    }


# --------------------------------------------------------------------------------------------------
# 9. VERIFY STAGE 6B INPUT
# --------------------------------------------------------------------------------------------------

for required_path in [
    EVALUABLE_PATH,
    EVALUABLE_SIDECAR,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Missing frozen Stage 6B artifact:\n"
            f"{required_path}"
        )

observed_evaluable_hash = sha256_file(
    EVALUABLE_PATH
)

assert (
    observed_evaluable_hash
    == EXPECTED_EVALUABLE_SHA256
)

assert (
    read_sidecar_hash(
        EVALUABLE_SIDECAR
    )
    == observed_evaluable_hash
)

parquet_file = pq.ParquetFile(
    EVALUABLE_PATH
)

assert (
    parquet_file.metadata.num_rows
    == EXPECTED_ROWS
)

assert (
    parquet_file.metadata.num_columns
    == EXPECTED_COLUMNS
)


# --------------------------------------------------------------------------------------------------
# 10. VERIFY PRIOR CELL 6C-4D0
# --------------------------------------------------------------------------------------------------

for required_path in [
    PRIOR_MANIFEST_PATH,
    PRIOR_MANIFEST_SIDECAR,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Missing completed Cell 6C-4D0 artifact:\n"
            f"{required_path}"
        )

prior_manifest_hash = sha256_file(
    PRIOR_MANIFEST_PATH
)

assert (
    read_sidecar_hash(
        PRIOR_MANIFEST_SIDECAR
    )
    == prior_manifest_hash
)

prior_manifest = json.loads(
    PRIOR_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

assert (
    prior_manifest[
        "required_final_freeze_category"
    ]
    == "remaining_comparator_inference"
)

assert (
    "PASS_STAGE6C_REMAINING_COMPARATOR_INFERENCE_RESULT_CATEGORY"
    in prior_manifest[
        "decision"
    ]
)


# --------------------------------------------------------------------------------------------------
# 11. LOAD REQUIRED FROZEN COLUMNS
# --------------------------------------------------------------------------------------------------

required_columns = [
    RCV_COLUMN,
    ROW_ORDER_COLUMN,
    STAR_COLUMN,
    OUTCOME_COLUMN,
] + [
    specification[
        "column"
    ]
    for specification in MODELS.values()
]

missing_columns = [
    column
    for column in required_columns
    if column
    not in parquet_file.schema_arrow.names
]

assert not missing_columns, (
    f"Missing same-star columns: "
    f"{missing_columns}"
)

cohort = pd.read_parquet(
    EVALUABLE_PATH,
    columns=required_columns,
).copy()

assert len(
    cohort
) == EXPECTED_ROWS

assert (
    cohort[
        RCV_COLUMN
    ].nunique(
        dropna=False
    )
    == EXPECTED_ROWS
)

row_order = pd.to_numeric(
    cohort[
        ROW_ORDER_COLUMN
    ],
    errors="raise",
).to_numpy()

assert (
    len(
        np.unique(
            row_order
        )
    )
    == EXPECTED_ROWS
)

assert np.all(
    np.diff(
        row_order
    )
    > 0
)

cohort[
    STAR_COLUMN
] = pd.to_numeric(
    cohort[
        STAR_COLUMN
    ],
    errors="raise",
).astype(
    np.int8
)

cohort[
    OUTCOME_COLUMN
] = pd.to_numeric(
    cohort[
        OUTCOME_COLUMN
    ],
    errors="raise",
).astype(
    np.int8
)

assert set(
    cohort[
        STAR_COLUMN
    ].unique()
) == {
    0,
    1,
    2,
    3,
}

assert int(
    cohort[
        OUTCOME_COLUMN
    ].sum()
) == EXPECTED_EVENTS

assert int(
    (
        cohort[
            OUTCOME_COLUMN
        ]
        == 0
    ).sum()
) == EXPECTED_NEGATIVES

for model_key, specification in (
    MODELS.items()
):
    cohort[
        specification[
            "column"
        ]
    ] = pd.to_numeric(
        cohort[
            specification[
                "column"
            ]
        ],
        errors="raise",
    )

    score_values = cohort[
        specification[
            "column"
        ]
    ].to_numpy(
        dtype=float
    )

    assert np.isfinite(
        score_values
    ).all()

    assert np.all(
        (
            score_values >= 0
        )
        & (
            score_values <= 1
        )
    )


# --------------------------------------------------------------------------------------------------
# 12. STRATUM INVENTORY
# --------------------------------------------------------------------------------------------------

inventory_rows = []

for star_level in [
    0,
    1,
    2,
    3,
]:
    stratum = cohort.loc[
        cohort[
            STAR_COLUMN
        ].eq(
            star_level
        )
    ]

    rows = int(
        len(
            stratum
        )
    )

    events = int(
        stratum[
            OUTCOME_COLUMN
        ].sum()
    )

    negatives = int(
        rows
        - events
    )

    prevalence = float(
        events
        / rows
    )

    expected = EXPECTED_STRATA[
        star_level
    ]

    assert rows == expected[
        "rows"
    ]

    assert events == expected[
        "events"
    ]

    assert negatives == expected[
        "negatives"
    ]

    inventory_rows.append(
        {
            "stars": star_level,
            "rows": rows,
            "events": events,
            "negatives": negatives,
            "prevalence": prevalence,
            "status": expected[
                "status"
            ],
            "discrimination_estimable": bool(
                events > 0
                and negatives > 0
            ),
            "extremely_sparse": bool(
                star_level == 3
            ),
        }
    )

stratum_inventory = pd.DataFrame(
    inventory_rows
)

assert len(
    stratum_inventory
) == 4


# --------------------------------------------------------------------------------------------------
# 13. POINT ESTIMATES AND GROUPED-ENGINE VALIDATION
# --------------------------------------------------------------------------------------------------

point_rows = []
group_specifications = {}

for star_level in [
    0,
    1,
    2,
    3,
]:
    stratum = (
        cohort.loc[
            cohort[
                STAR_COLUMN
            ].eq(
                star_level
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    outcome = stratum[
        OUTCOME_COLUMN
    ].to_numpy(
        dtype=np.int8
    )

    rows = len(
        stratum
    )

    events = int(
        outcome.sum()
    )

    negatives = int(
        rows
        - events
    )

    prevalence = float(
        outcome.mean()
    )

    group_specifications[
        star_level
    ] = {}

    for model_key, specification in (
        MODELS.items()
    ):
        score = stratum[
            specification[
                "column"
            ]
        ].to_numpy(
            dtype=float
        )

        group_specification = (
            prepare_score_groups(
                outcome,
                score,
            )
        )

        group_specifications[
            star_level
        ][
            model_key
        ] = group_specification

        if (
            events > 0
            and negatives > 0
        ):
            auprc = float(
                average_precision_score(
                    outcome,
                    score,
                )
            )

            auroc = float(
                roc_auc_score(
                    outcome,
                    score,
                )
            )

            unit_counts = np.ones(
                (
                    1,
                    rows,
                ),
                dtype=np.int64,
            )

            grouped_result = (
                grouped_bootstrap_metrics(
                    unit_counts,
                    group_specification,
                )
            )

            assert abs(
                grouped_result[
                    "auprc"
                ][
                    0
                ]
                - auprc
            ) <= 1e-12

            assert abs(
                grouped_result[
                    "auroc"
                ][
                    0
                ]
                - auroc
            ) <= 1e-12
        else:
            auprc = np.nan
            auroc = np.nan

        event_mean_score = float(
            score[
                outcome == 1
            ].mean()
        )

        negative_mean_score = (
            float(
                score[
                    outcome == 0
                ].mean()
            )
            if negatives > 0
            else np.nan
        )

        point_rows.append(
            {
                "stars": star_level,
                "model_key": model_key,
                "model": specification[
                    "display"
                ],
                "score_column": specification[
                    "column"
                ],
                "rows": rows,
                "events": events,
                "negatives": negatives,
                "prevalence": prevalence,
                "auprc": auprc,
                "auprc_lift_over_prevalence": (
                    float(
                        auprc
                        / prevalence
                    )
                    if np.isfinite(
                        auprc
                    )
                    and prevalence > 0
                    else np.nan
                ),
                "auroc": auroc,
                "mean_score_events": event_mean_score,
                "mean_score_negatives": negative_mean_score,
                "mean_score_event_minus_negative": (
                    event_mean_score
                    - negative_mean_score
                    if np.isfinite(
                        negative_mean_score
                    )
                    else np.nan
                ),
                "minimum_score": float(
                    score.min()
                ),
                "maximum_score": float(
                    score.max()
                ),
                "unique_score_values": int(
                    np.unique(
                        score
                    ).size
                ),
                "discrimination_estimable": bool(
                    events > 0
                    and negatives > 0
                ),
                "extremely_sparse": bool(
                    star_level == 3
                ),
            }
        )

point_estimates = pd.DataFrame(
    point_rows
)

assert len(
    point_estimates
) == 12


# --------------------------------------------------------------------------------------------------
# 14. VERIFY HISTORICAL POINT ESTIMATES
# --------------------------------------------------------------------------------------------------

historical_model = pd.DataFrame(
    HISTORICAL_MODEL_RESULTS
)

historical_model[
    "model"
] = historical_model[
    "model_key"
].map(
    {
        key: value[
            "display"
        ]
        for key, value in (
            MODELS.items()
        )
    }
)

point_verification = (
    historical_model.merge(
        point_estimates[
            [
                "stars",
                "model_key",
                "auprc",
                "auroc",
                "prevalence",
            ]
        ],
        on=[
            "stars",
            "model_key",
        ],
        how="inner",
        validate="one_to_one",
        suffixes=(
            "_historical",
            "_observed",
        ),
    )
)

assert len(
    point_verification
) == 9

point_verification[
    "auprc_point_difference"
] = (
    point_verification[
        "auprc_observed"
    ]
    - point_verification[
        "auprc_historical"
    ]
)

point_verification[
    "auroc_point_difference"
] = (
    point_verification[
        "auroc_observed"
    ]
    - point_verification[
        "auroc_historical"
    ]
)

point_verification[
    "auprc_point_reproduced"
] = (
    point_verification[
        "auprc_point_difference"
    ].abs()
    <= 2.0e-6
)

point_verification[
    "auroc_point_reproduced"
] = (
    point_verification[
        "auroc_point_difference"
    ].abs()
    <= 2.0e-6
)

assert point_verification[
    "auprc_point_reproduced"
].all()

assert point_verification[
    "auroc_point_reproduced"
].all()


# --------------------------------------------------------------------------------------------------
# 15. RUN EXACT MULTINOMIAL REPRESENTATION OF ORDINARY ROW BOOTSTRAP
# --------------------------------------------------------------------------------------------------

rng = np.random.default_rng(
    ROOT_SEED
)

bootstrap_frames = []

bootstrap_start = time.perf_counter()

for star_level in [
    1,
    2,
    3,
]:
    stratum = (
        cohort.loc[
            cohort[
                STAR_COLUMN
            ].eq(
                star_level
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    outcome = stratum[
        OUTCOME_COLUMN
    ].to_numpy(
        dtype=np.int8
    )

    rows = len(
        stratum
    )

    probabilities = np.full(
        rows,
        1.0 / rows,
        dtype=float,
    )

    probabilities[
        -1
    ] = (
        1.0
        - probabilities[
            :-1
        ].sum()
    )

    assert np.isclose(
        probabilities.sum(),
        1.0,
        atol=1e-15,
        rtol=0,
    )

    replicate_number = 1

    for batch_start in range(
        0,
        BOOTSTRAP_ATTEMPTS,
        BOOTSTRAP_BATCH_SIZE,
    ):
        batch_size = min(
            BOOTSTRAP_BATCH_SIZE,
            (
                BOOTSTRAP_ATTEMPTS
                - batch_start
            ),
        )

        bootstrap_counts = rng.multinomial(
            rows,
            probabilities,
            size=batch_size,
        )

        bootstrap_events = (
            bootstrap_counts
            @ outcome
        )

        bootstrap_negatives = (
            rows
            - bootstrap_events
        )

        bootstrap_prevalence = (
            bootstrap_events
            / rows
        )

        valid = (
            bootstrap_events > 0
        ) & (
            bootstrap_negatives > 0
        )

        batch_data = {
            "stars": np.full(
                batch_size,
                star_level,
                dtype=np.int8,
            ),
            "replicate": np.arange(
                replicate_number,
                (
                    replicate_number
                    + batch_size
                ),
                dtype=np.int32,
            ),
            "valid_two_class_replicate": valid,
            "bootstrap_events": bootstrap_events,
            "bootstrap_negatives": bootstrap_negatives,
            "bootstrap_prevalence": bootstrap_prevalence,
        }

        model_valid_masks = []

        for model_key in MODELS:
            metric_result = (
                grouped_bootstrap_metrics(
                    bootstrap_counts,
                    group_specifications[
                        star_level
                    ][
                        model_key
                    ],
                )
            )

            model_valid_masks.append(
                metric_result[
                    "valid"
                ]
            )

            batch_data[
                f"auprc_{model_key}"
            ] = metric_result[
                "auprc"
            ]

            batch_data[
                f"auroc_{model_key}"
            ] = metric_result[
                "auroc"
            ]

        for model_valid in (
            model_valid_masks
        ):
            assert np.array_equal(
                model_valid,
                valid,
            )

        bootstrap_frames.append(
            pd.DataFrame(
                batch_data
            )
        )

        replicate_number += batch_size

    assert (
        replicate_number
        == BOOTSTRAP_ATTEMPTS + 1
    )

bootstrap_elapsed_seconds = float(
    time.perf_counter()
    - bootstrap_start
)

replicates = (
    pd.concat(
        bootstrap_frames,
        ignore_index=True,
    )
    .sort_values(
        [
            "stars",
            "replicate",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

assert len(
    replicates
) == (
    3
    * BOOTSTRAP_ATTEMPTS
)

for star_level in [
    1,
    2,
    3,
]:
    star_replicates = (
        replicates.loc[
            replicates[
                "stars"
            ].eq(
                star_level
            )
        ]
    )

    assert len(
        star_replicates
    ) == BOOTSTRAP_ATTEMPTS

    assert star_replicates[
        "replicate"
    ].tolist() == list(
        range(
            1,
            BOOTSTRAP_ATTEMPTS + 1,
        )
    )

metric_columns = [
    f"{metric}_{model_key}"
    for metric in [
        "auprc",
        "auroc",
    ]
    for model_key in MODELS
]

valid_metric_values = (
    replicates.loc[
        replicates[
            "valid_two_class_replicate"
        ].astype(
            bool
        ),
        metric_columns,
    ]
)

assert valid_metric_values.notna().all().all()

assert np.isfinite(
    valid_metric_values.to_numpy(
        dtype=float
    )
).all()


# --------------------------------------------------------------------------------------------------
# 16. MODEL BOOTSTRAP INTERVALS
# --------------------------------------------------------------------------------------------------

model_interval_rows = []

for star_level in [
    1,
    2,
    3,
]:
    star_replicates = (
        replicates.loc[
            replicates[
                "stars"
            ].eq(
                star_level
            )
        ]
    )

    valid_mask = (
        star_replicates[
            "valid_two_class_replicate"
        ].astype(
            bool
        )
    )

    valid_count = int(
        valid_mask.sum()
    )

    invalid_count = int(
        (
            ~valid_mask
        ).sum()
    )

    prevalence = float(
        stratum_inventory.loc[
            stratum_inventory[
                "stars"
            ].eq(
                star_level
            ),
            "prevalence",
        ].iloc[
            0
        ]
    )

    for model_key, specification in (
        MODELS.items()
    ):
        point_row = (
            point_estimates.loc[
                point_estimates[
                    "stars"
                ].eq(
                    star_level
                )
                & point_estimates[
                    "model_key"
                ].eq(
                    model_key
                )
            ]
            .iloc[
                0
            ]
        )

        for metric in [
            "AUPRC",
            "AUROC",
        ]:
            values = (
                star_replicates.loc[
                    valid_mask,
                    f"{metric.lower()}_{model_key}",
                ].to_numpy(
                    dtype=float
                )
            )

            ci_low, ci_high = (
                percentile_interval(
                    values
                )
            )

            if metric == "AUPRC":
                null_differences = (
                    values
                    - star_replicates.loc[
                        valid_mask,
                        "bootstrap_prevalence",
                    ].to_numpy(
                        dtype=float
                    )
                )

                point_value = float(
                    point_row[
                        "auprc"
                    ]
                )

                fixed_null = prevalence
            else:
                null_differences = (
                    values
                    - 0.50
                )

                point_value = float(
                    point_row[
                        "auroc"
                    ]
                )

                fixed_null = 0.50

            null_ci_low, null_ci_high = (
                percentile_interval(
                    null_differences
                )
            )

            null_sign_result = (
                finite_two_sided_sign_probability(
                    null_differences
                )
            )

            model_interval_rows.append(
                {
                    "stars": star_level,
                    "model_key": model_key,
                    "model": specification[
                        "display"
                    ],
                    "metric": metric,
                    "rows": int(
                        point_row[
                            "rows"
                        ]
                    ),
                    "events": int(
                        point_row[
                            "events"
                        ]
                    ),
                    "negatives": int(
                        point_row[
                            "negatives"
                        ]
                    ),
                    "prevalence": prevalence,
                    "point_estimate": point_value,
                    "bootstrap_mean": float(
                        values.mean()
                    ),
                    "bootstrap_standard_error": float(
                        values.std(
                            ddof=1
                        )
                    ),
                    "percentile_95_ci_low": ci_low,
                    "percentile_95_ci_high": ci_high,
                    "fixed_null_reference": fixed_null,
                    "point_minus_fixed_null": float(
                        point_value
                        - fixed_null
                    ),
                    "bootstrap_null_difference_ci_low": null_ci_low,
                    "bootstrap_null_difference_ci_high": null_ci_high,
                    "bootstrap_null_sign_probability": null_sign_result[
                        "probability"
                    ],
                    "null_interval_conclusion": (
                        model_null_conclusion(
                            metric,
                            ci_low,
                            ci_high,
                            prevalence,
                        )
                    ),
                    "attempted_replicates": BOOTSTRAP_ATTEMPTS,
                    "valid_replicates": valid_count,
                    "invalid_one_class_replicates": invalid_count,
                    "extremely_sparse": bool(
                        star_level == 3
                    ),
                }
            )

model_intervals = pd.DataFrame(
    model_interval_rows
)

assert len(
    model_intervals
) == 18


# --------------------------------------------------------------------------------------------------
# 17. PAIRED FULL-GES-MINUS-COMPARATOR INFERENCE
# --------------------------------------------------------------------------------------------------

paired_rows = []

for star_level in [
    1,
    2,
    3,
]:
    star_replicates = (
        replicates.loc[
            replicates[
                "stars"
            ].eq(
                star_level
            )
        ]
    )

    valid_mask = (
        star_replicates[
            "valid_two_class_replicate"
        ].astype(
            bool
        )
    )

    valid_count = int(
        valid_mask.sum()
    )

    invalid_count = int(
        (
            ~valid_mask
        ).sum()
    )

    for metric in [
        "AUPRC",
        "AUROC",
    ]:
        full_values = (
            star_replicates.loc[
                valid_mask,
                f"{metric.lower()}_full_ges",
            ].to_numpy(
                dtype=float
            )
        )

        full_point = float(
            point_estimates.loc[
                point_estimates[
                    "stars"
                ].eq(
                    star_level
                )
                & point_estimates[
                    "model_key"
                ].eq(
                    "full_ges"
                ),
                metric.lower(),
            ].iloc[
                0
            ]
        )

        for (
            comparator_key,
            comparator_display,
        ) in COMPARATORS.items():
            comparator_values = (
                star_replicates.loc[
                    valid_mask,
                    f"{metric.lower()}_{comparator_key}",
                ].to_numpy(
                    dtype=float
                )
            )

            comparator_point = float(
                point_estimates.loc[
                    point_estimates[
                        "stars"
                    ].eq(
                        star_level
                    )
                    & point_estimates[
                        "model_key"
                    ].eq(
                        comparator_key
                    ),
                    metric.lower(),
                ].iloc[
                    0
                ]
            )

            differences = (
                full_values
                - comparator_values
            )

            ci_low, ci_high = (
                percentile_interval(
                    differences
                )
            )

            sign_result = (
                finite_two_sided_sign_probability(
                    differences
                )
            )

            paired_rows.append(
                {
                    "stars": star_level,
                    "metric": metric,
                    "comparator_key": comparator_key,
                    "comparator": comparator_display,
                    "comparison": (
                        "Full GES minus "
                        + comparator_display
                    ),
                    "full_ges_point_estimate": full_point,
                    "comparator_point_estimate": comparator_point,
                    "point_difference": float(
                        full_point
                        - comparator_point
                    ),
                    "bootstrap_mean_difference": float(
                        differences.mean()
                    ),
                    "bootstrap_standard_error": float(
                        differences.std(
                            ddof=1
                        )
                    ),
                    "percentile_95_ci_low": ci_low,
                    "percentile_95_ci_high": ci_high,
                    "fraction_difference_gt_zero": float(
                        np.mean(
                            differences > 0
                        )
                    ),
                    "fraction_difference_lt_zero": float(
                        np.mean(
                            differences < 0
                        )
                    ),
                    "zero_differences": sign_result[
                        "zero"
                    ],
                    "two_sided_bootstrap_sign_probability": sign_result[
                        "probability"
                    ],
                    "interval_conclusion": interval_conclusion(
                        ci_low,
                        ci_high,
                    ),
                    "attempted_replicates": BOOTSTRAP_ATTEMPTS,
                    "valid_replicates": valid_count,
                    "invalid_one_class_replicates": invalid_count,
                    "extremely_sparse": bool(
                        star_level == 3
                    ),
                }
            )

paired_inference = pd.DataFrame(
    paired_rows
)

assert len(
    paired_inference
) == 12


# --------------------------------------------------------------------------------------------------
# 18. HOLM CORRECTION: SIX AUPRC AND SIX AUROC COMPARISONS
# --------------------------------------------------------------------------------------------------

paired_inference[
    "holm_family"
] = ""

paired_inference[
    "holm_family_size"
] = 0

paired_inference[
    "holm_adjusted_bootstrap_sign_probability"
] = np.nan

paired_inference[
    "holm_rank"
] = 0

for metric in [
    "AUPRC",
    "AUROC",
]:
    metric_mask = (
        paired_inference[
            "metric"
        ].eq(
            metric
        )
    )

    raw_probabilities = (
        paired_inference.loc[
            metric_mask,
            "two_sided_bootstrap_sign_probability",
        ].to_numpy(
            dtype=float
        )
    )

    adjusted_probabilities = (
        holm_adjust(
            raw_probabilities
        )
    )

    probability_order = np.argsort(
        raw_probabilities,
        kind="mergesort",
    )

    ranks = np.empty(
        len(
            probability_order
        ),
        dtype=int,
    )

    ranks[
        probability_order
    ] = np.arange(
        1,
        len(
            probability_order
        )
        + 1,
    )

    paired_inference.loc[
        metric_mask,
        "holm_family",
    ] = (
        f"same_star_{metric.lower()}"
    )

    paired_inference.loc[
        metric_mask,
        "holm_family_size",
    ] = 6

    paired_inference.loc[
        metric_mask,
        "holm_adjusted_bootstrap_sign_probability",
    ] = adjusted_probabilities

    paired_inference.loc[
        metric_mask,
        "holm_rank",
    ] = ranks

paired_inference[
    "holm_reject_at_0_05"
] = (
    paired_inference[
        "holm_adjusted_bootstrap_sign_probability"
    ]
    <= 0.05
)

assert paired_inference[
    "holm_family_size"
].eq(
    6
).all()

multiplicity_table = (
    paired_inference[
        [
            "metric",
            "stars",
            "comparator_key",
            "comparator",
            "holm_family",
            "holm_family_size",
            "holm_rank",
            "two_sided_bootstrap_sign_probability",
            "holm_adjusted_bootstrap_sign_probability",
            "holm_reject_at_0_05",
            "interval_conclusion",
            "extremely_sparse",
        ]
    ]
    .sort_values(
        [
            "metric",
            "holm_rank",
            "stars",
            "comparator_key",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

assert len(
    multiplicity_table
) == 12


# --------------------------------------------------------------------------------------------------
# 19. SPARSE-STRATUM AUDIT
# --------------------------------------------------------------------------------------------------

star_3_valid = int(
    replicates.loc[
        replicates[
            "stars"
        ].eq(
            3
        ),
        "valid_two_class_replicate",
    ].astype(
        bool
    ).sum()
)

star_3_invalid = int(
    BOOTSTRAP_ATTEMPTS
    - star_3_valid
)

sparse_audit = pd.DataFrame(
    [
        {
            "stars": 0,
            "rows": 36,
            "events": 36,
            "negatives": 0,
            "attempted_bootstrap_replicates": 0,
            "valid_bootstrap_replicates": 0,
            "invalid_one_class_replicates": 0,
            "status": "descriptive_only",
            "reason": (
                "All 36 records are events; discrimination "
                "metrics require both classes."
            ),
            "interpretation_boundary": (
                "Exact score values are retained descriptively, "
                "but AUPRC/AUROC superiority cannot be inferred."
            ),
        },
        {
            "stars": 3,
            "rows": 8_153,
            "events": 2,
            "negatives": 8_151,
            "attempted_bootstrap_replicates": BOOTSTRAP_ATTEMPTS,
            "valid_bootstrap_replicates": star_3_valid,
            "invalid_one_class_replicates": star_3_invalid,
            "status": "estimable_but_extremely_sparse",
            "reason": (
                "Only two events are present; ordinary bootstrap "
                "samples may contain no positive events."
            ),
            "interpretation_boundary": (
                "Intervals and paired comparisons are retained as "
                "sparse-event observations and not as stable evidence."
            ),
        },
    ]
)

assert len(
    sparse_audit
) == 2

assert star_3_invalid > 0


# --------------------------------------------------------------------------------------------------
# 20. HISTORICAL TABLES
# --------------------------------------------------------------------------------------------------

historical_model[
    "historical_value_precision"
] = (
    "rounded_values_recorded_in_technical_report"
)

historical_model[
    "historical_row_level_replicate_artifact_available"
] = False

historical_model_long_rows = []

for _, row in (
    historical_model.iterrows()
):
    prevalence = float(
        stratum_inventory.loc[
            stratum_inventory[
                "stars"
            ].eq(
                int(
                    row[
                        "stars"
                    ]
                )
            ),
            "prevalence",
        ].iloc[
            0
        ]
    )

    for metric in [
        "AUPRC",
        "AUROC",
    ]:
        lower_metric = metric.lower()

        historical_model_long_rows.append(
            {
                "stars": int(
                    row[
                        "stars"
                    ]
                ),
                "model_key": row[
                    "model_key"
                ],
                "model": row[
                    "model"
                ],
                "metric": metric,
                "historical_point_estimate": float(
                    row[
                        lower_metric
                    ]
                ),
                "historical_percentile_95_ci_low": float(
                    row[
                        f"{lower_metric}_ci_low"
                    ]
                ),
                "historical_percentile_95_ci_high": float(
                    row[
                        f"{lower_metric}_ci_high"
                    ]
                ),
                "historical_null_conclusion": (
                    model_null_conclusion(
                        metric,
                        row[
                            f"{lower_metric}_ci_low"
                        ],
                        row[
                            f"{lower_metric}_ci_high"
                        ],
                        prevalence,
                    )
                ),
                "historical_value_precision": (
                    "rounded_values_recorded_in_technical_report"
                ),
                "historical_row_level_replicate_artifact_available": False,
            }
        )

historical_model_long = pd.DataFrame(
    historical_model_long_rows
)

assert len(
    historical_model_long
) == 18

historical_paired = pd.DataFrame(
    HISTORICAL_PAIRED_RESULTS
)

historical_paired[
    "comparator"
] = historical_paired[
    "comparator_key"
].map(
    COMPARATORS
)

historical_paired[
    "comparison"
] = (
    "Full GES minus "
    + historical_paired[
        "comparator"
    ]
)

historical_paired[
    "historical_interval_conclusion"
] = historical_paired.apply(
    lambda row: interval_conclusion(
        row[
            "ci_low"
        ],
        row[
            "ci_high"
        ],
    ),
    axis=1,
)

historical_paired[
    "historical_holm_significant"
] = (
    historical_paired[
        "holm_adjusted_probability"
    ]
    <= 0.05
)

historical_paired[
    "historical_value_precision"
] = (
    "rounded_values_recorded_in_technical_report"
)

historical_paired[
    "historical_row_level_replicate_artifact_available"
] = False

assert len(
    historical_paired
) == 12


# --------------------------------------------------------------------------------------------------
# 21. HISTORICAL VERSUS REPRODUCED CONCORDANCE
# --------------------------------------------------------------------------------------------------

model_concordance = (
    historical_model_long.merge(
        model_intervals[
            [
                "stars",
                "model_key",
                "model",
                "metric",
                "point_estimate",
                "percentile_95_ci_low",
                "percentile_95_ci_high",
                "null_interval_conclusion",
                "valid_replicates",
                "invalid_one_class_replicates",
            ]
        ],
        on=[
            "stars",
            "model_key",
            "model",
            "metric",
        ],
        how="inner",
        validate="one_to_one",
    )
)

assert len(
    model_concordance
) == 18

model_concordance[
    "point_difference"
] = (
    model_concordance[
        "point_estimate"
    ]
    - model_concordance[
        "historical_point_estimate"
    ]
)

model_concordance[
    "point_reproduced"
] = (
    model_concordance[
        "point_difference"
    ].abs()
    <= 2.0e-6
)

model_concordance[
    "null_conclusion_concordant"
] = (
    model_concordance[
        "historical_null_conclusion"
    ]
    == model_concordance[
        "null_interval_conclusion"
    ]
)

model_concordance[
    "scientific_conclusion_concordant"
] = (
    model_concordance[
        "point_reproduced"
    ]
    & model_concordance[
        "null_conclusion_concordant"
    ]
)

paired_concordance = (
    historical_paired.merge(
        paired_inference[
            [
                "stars",
                "metric",
                "comparator_key",
                "comparator",
                "comparison",
                "point_difference",
                "percentile_95_ci_low",
                "percentile_95_ci_high",
                "interval_conclusion",
                "holm_adjusted_bootstrap_sign_probability",
                "holm_reject_at_0_05",
                "valid_replicates",
                "invalid_one_class_replicates",
            ]
        ],
        on=[
            "stars",
            "metric",
            "comparator_key",
            "comparator",
            "comparison",
        ],
        how="inner",
        validate="one_to_one",
        suffixes=(
            "_historical",
            "_reproduced",
        ),
    )
)

assert len(
    paired_concordance
) == 12

paired_concordance[
    "point_difference_delta"
] = (
    paired_concordance[
        "point_difference_reproduced"
    ]
    - paired_concordance[
        "point_difference_historical"
    ]
)

paired_concordance[
    "point_reproduced"
] = (
    paired_concordance[
        "point_difference_delta"
    ].abs()
    <= 2.0e-6
)

paired_concordance[
    "interval_conclusion_concordant"
] = (
    paired_concordance[
        "historical_interval_conclusion"
    ]
    == paired_concordance[
        "interval_conclusion"
    ]
)

paired_concordance[
    "holm_conclusion_concordant"
] = (
    paired_concordance[
        "historical_holm_significant"
    ]
    == paired_concordance[
        "holm_reject_at_0_05"
    ]
)

paired_concordance[
    "scientific_conclusion_concordant"
] = (
    paired_concordance[
        "point_reproduced"
    ]
    & paired_concordance[
        "interval_conclusion_concordant"
    ]
    & paired_concordance[
        "holm_conclusion_concordant"
    ]
)

assert model_concordance[
    "point_reproduced"
].all()

assert paired_concordance[
    "point_reproduced"
].all()

assert model_concordance[
    "null_conclusion_concordant"
].all()

assert paired_concordance[
    "interval_conclusion_concordant"
].all()

assert paired_concordance[
    "holm_conclusion_concordant"
].all()

assert model_concordance[
    "scientific_conclusion_concordant"
].all()

assert paired_concordance[
    "scientific_conclusion_concordant"
].all()

model_concordance_export = (
    model_concordance.copy()
)

model_concordance_export[
    "result_type"
] = "model_interval"

paired_concordance_export = (
    paired_concordance.copy()
)

paired_concordance_export[
    "result_type"
] = "paired_comparison"

concordance = pd.concat(
    [
        model_concordance_export,
        paired_concordance_export,
    ],
    ignore_index=True,
    sort=False,
)

assert len(
    concordance
) == 30


# --------------------------------------------------------------------------------------------------
# 22. PRESPECIFIED INTERPRETATION CHECKS
# --------------------------------------------------------------------------------------------------

def get_paired_row(
    stars,
    metric,
    comparator_key,
):
    result = paired_inference.loc[
        paired_inference[
            "stars"
        ].eq(
            stars
        )
        & paired_inference[
            "metric"
        ].eq(
            metric
        )
        & paired_inference[
            "comparator_key"
        ].eq(
            comparator_key
        )
    ]

    assert len(
        result
    ) == 1

    return result.iloc[
        0
    ]


assert get_paired_row(
    1,
    "AUPRC",
    "no_star_ges",
)[
    "interval_conclusion"
] == "full_ges_lower"

assert get_paired_row(
    1,
    "AUROC",
    "no_star_ges",
)[
    "interval_conclusion"
] == "full_ges_lower"

assert get_paired_row(
    1,
    "AUPRC",
    "combined_metadata",
)[
    "interval_conclusion"
] == "full_ges_lower"

assert get_paired_row(
    1,
    "AUROC",
    "combined_metadata",
)[
    "interval_conclusion"
] == "full_ges_lower"

assert get_paired_row(
    2,
    "AUPRC",
    "no_star_ges",
)[
    "interval_conclusion"
] == "interval_includes_zero"

assert bool(
    get_paired_row(
        2,
        "AUPRC",
        "no_star_ges",
    )[
        "holm_reject_at_0_05"
    ]
) is False

assert get_paired_row(
    2,
    "AUROC",
    "no_star_ges",
)[
    "interval_conclusion"
] == "full_ges_higher"

assert get_paired_row(
    2,
    "AUPRC",
    "combined_metadata",
)[
    "interval_conclusion"
] == "full_ges_lower"

assert get_paired_row(
    2,
    "AUROC",
    "combined_metadata",
)[
    "interval_conclusion"
] == "full_ges_lower"


# --------------------------------------------------------------------------------------------------
# 23. WRITE TABLES
# --------------------------------------------------------------------------------------------------

write_csv(
    stratum_inventory,
    STRATUM_INVENTORY_PATH,
)

write_csv(
    point_estimates,
    POINT_ESTIMATE_PATH,
)

write_parquet(
    replicates,
    REPLICATE_PATH,
)

write_csv(
    model_intervals,
    MODEL_INTERVAL_PATH,
)

write_csv(
    paired_inference,
    PAIRED_INFERENCE_PATH,
)

write_csv(
    multiplicity_table,
    MULTIPLICITY_PATH,
)

write_csv(
    sparse_audit,
    SPARSE_AUDIT_PATH,
)

write_csv(
    historical_model_long,
    HISTORICAL_MODEL_PATH,
)

write_csv(
    historical_paired,
    HISTORICAL_PAIRED_PATH,
)

write_csv(
    concordance,
    CONCORDANCE_PATH,
)


# --------------------------------------------------------------------------------------------------
# 24. CREATE SIDECARS
# --------------------------------------------------------------------------------------------------

stratum_inventory_sidecar = (
    write_sidecar(
        STRATUM_INVENTORY_PATH
    )
)

point_estimate_sidecar = (
    write_sidecar(
        POINT_ESTIMATE_PATH
    )
)

replicate_sidecar = write_sidecar(
    REPLICATE_PATH
)

model_interval_sidecar = (
    write_sidecar(
        MODEL_INTERVAL_PATH
    )
)

paired_inference_sidecar = (
    write_sidecar(
        PAIRED_INFERENCE_PATH
    )
)

multiplicity_sidecar = (
    write_sidecar(
        MULTIPLICITY_PATH
    )
)

sparse_audit_sidecar = (
    write_sidecar(
        SPARSE_AUDIT_PATH
    )
)

historical_model_sidecar = (
    write_sidecar(
        HISTORICAL_MODEL_PATH
    )
)

historical_paired_sidecar = (
    write_sidecar(
        HISTORICAL_PAIRED_PATH
    )
)

concordance_sidecar = (
    write_sidecar(
        CONCORDANCE_PATH
    )
)


# --------------------------------------------------------------------------------------------------
# 25. FRESH READBACK
# --------------------------------------------------------------------------------------------------

fresh_inventory = pd.read_csv(
    STRATUM_INVENTORY_PATH
)

fresh_points = pd.read_csv(
    POINT_ESTIMATE_PATH
)

fresh_replicates = pd.read_parquet(
    REPLICATE_PATH
)

fresh_model_intervals = pd.read_csv(
    MODEL_INTERVAL_PATH
)

fresh_paired = pd.read_csv(
    PAIRED_INFERENCE_PATH
)

fresh_multiplicity = pd.read_csv(
    MULTIPLICITY_PATH
)

fresh_sparse = pd.read_csv(
    SPARSE_AUDIT_PATH
)

fresh_historical_model = pd.read_csv(
    HISTORICAL_MODEL_PATH
)

fresh_historical_paired = pd.read_csv(
    HISTORICAL_PAIRED_PATH
)

fresh_concordance = pd.read_csv(
    CONCORDANCE_PATH
)

assert len(
    fresh_inventory
) == 4

assert len(
    fresh_points
) == 12

assert len(
    fresh_replicates
) == 6_000

assert len(
    fresh_model_intervals
) == 18

assert len(
    fresh_paired
) == 12

assert len(
    fresh_multiplicity
) == 12

assert len(
    fresh_sparse
) == 2

assert len(
    fresh_historical_model
) == 18

assert len(
    fresh_historical_paired
) == 12

assert len(
    fresh_concordance
) == 30

assert fresh_concordance[
    "scientific_conclusion_concordant"
].astype(
    bool
).all()

sidecar_pairs = [
    (
        STRATUM_INVENTORY_PATH,
        stratum_inventory_sidecar,
    ),
    (
        POINT_ESTIMATE_PATH,
        point_estimate_sidecar,
    ),
    (
        REPLICATE_PATH,
        replicate_sidecar,
    ),
    (
        MODEL_INTERVAL_PATH,
        model_interval_sidecar,
    ),
    (
        PAIRED_INFERENCE_PATH,
        paired_inference_sidecar,
    ),
    (
        MULTIPLICITY_PATH,
        multiplicity_sidecar,
    ),
    (
        SPARSE_AUDIT_PATH,
        sparse_audit_sidecar,
    ),
    (
        HISTORICAL_MODEL_PATH,
        historical_model_sidecar,
    ),
    (
        HISTORICAL_PAIRED_PATH,
        historical_paired_sidecar,
    ),
    (
        CONCORDANCE_PATH,
        concordance_sidecar,
    ),
]

for (
    result_path,
    sidecar_path,
) in sidecar_pairs:
    assert (
        read_sidecar_hash(
            sidecar_path
        )
        == sha256_file(
            result_path
        )
    )


# --------------------------------------------------------------------------------------------------
# 26. QC
# --------------------------------------------------------------------------------------------------

valid_counts = (
    replicates.groupby(
        "stars"
    )[
        "valid_two_class_replicate"
    ]
    .sum()
    .astype(
        int
    )
    .to_dict()
)

invalid_counts = {
    star_level: int(
        BOOTSTRAP_ATTEMPTS
        - valid_counts[
            star_level
        ]
    )
    for star_level in [
        1,
        2,
        3,
    ]
}

checks = OrderedDict(
    [
        (
            "stage6b_evaluable_hash_verified",
            (
                observed_evaluable_hash
                == EXPECTED_EVALUABLE_SHA256
            ),
        ),
        (
            "stage6b_evaluable_sidecar_verified",
            True,
        ),
        (
            "stage6b_dimensions_verified",
            (
                parquet_file.metadata.num_rows
                == EXPECTED_ROWS
                and parquet_file.metadata.num_columns
                == EXPECTED_COLUMNS
            ),
        ),
        (
            "prior_6c_4d0_manifest_verified",
            True,
        ),
        (
            "cohort_keys_and_row_order_verified",
            True,
        ),
        (
            "outcome_accounting_verified",
            True,
        ),
        (
            "four_review_star_strata_verified",
            (
                len(
                    stratum_inventory
                )
                == 4
            ),
        ),
        (
            "zero_star_one_class_status_preserved",
            bool(
                (
                    stratum_inventory.loc[
                        stratum_inventory[
                            "stars"
                        ].eq(
                            0
                        ),
                        "negatives",
                    ].iloc[
                        0
                    ]
                    == 0
                )
            ),
        ),
        (
            "three_star_extreme_sparsity_preserved",
            bool(
                (
                    stratum_inventory.loc[
                        stratum_inventory[
                            "stars"
                        ].eq(
                            3
                        ),
                        "events",
                    ].iloc[
                        0
                    ]
                    == 2
                )
            ),
        ),
        (
            "twelve_point_estimate_rows_created",
            (
                len(
                    point_estimates
                )
                == 12
            ),
        ),
        (
            "grouped_metric_engine_matches_sklearn",
            True,
        ),
        (
            "historical_point_estimates_reproduced",
            bool(
                point_verification[
                    "auprc_point_reproduced"
                ].all()
                and point_verification[
                    "auroc_point_reproduced"
                ].all()
            ),
        ),
        (
            "six_thousand_bootstrap_attempt_rows_created",
            (
                len(
                    replicates
                )
                == 6_000
            ),
        ),
        (
            "stars_one_and_two_have_all_valid_replicates",
            bool(
                valid_counts[
                    1
                ]
                == BOOTSTRAP_ATTEMPTS
                and valid_counts[
                    2
                ]
                == BOOTSTRAP_ATTEMPTS
            ),
        ),
        (
            "star_three_one_class_attempts_preserved",
            bool(
                invalid_counts[
                    3
                ]
                > 0
            ),
        ),
        (
            "eighteen_model_interval_rows_created",
            (
                len(
                    model_intervals
                )
                == 18
            ),
        ),
        (
            "twelve_paired_inference_rows_created",
            (
                len(
                    paired_inference
                )
                == 12
            ),
        ),
        (
            "holm_applied_to_two_six_test_families",
            bool(
                paired_inference[
                    "holm_family_size"
                ].eq(
                    6
                ).all()
            ),
        ),
        (
            "star_two_auprc_no_star_comparison_remains_null",
            bool(
                get_paired_row(
                    2,
                    "AUPRC",
                    "no_star_ges",
                )[
                    "interval_conclusion"
                ]
                == "interval_includes_zero"
            ),
        ),
        (
            "same_star_mixed_negative_findings_preserved",
            True,
        ),
        (
            "all_model_interval_conclusions_concordant",
            bool(
                model_concordance[
                    "scientific_conclusion_concordant"
                ].all()
            ),
        ),
        (
            "all_paired_conclusions_concordant",
            bool(
                paired_concordance[
                    "scientific_conclusion_concordant"
                ].all()
            ),
        ),
        (
            "ten_scientific_tables_written",
            all(
                path.exists()
                for path in [
                    STRATUM_INVENTORY_PATH,
                    POINT_ESTIMATE_PATH,
                    REPLICATE_PATH,
                    MODEL_INTERVAL_PATH,
                    PAIRED_INFERENCE_PATH,
                    MULTIPLICITY_PATH,
                    SPARSE_AUDIT_PATH,
                    HISTORICAL_MODEL_PATH,
                    HISTORICAL_PAIRED_PATH,
                    CONCORDANCE_PATH,
                ]
            ),
        ),
        (
            "ten_scientific_tables_read_back",
            True,
        ),
        (
            "ten_scientific_table_sidecars_verified",
            True,
        ),
        (
            "historical_results_preserved_separately",
            True,
        ),
        (
            "independent_materialization_preserved_separately",
            True,
        ),
        (
            "no_frozen_scientific_input_modified",
            True,
        ),
        (
            "experiment_2_not_started",
            True,
        ),
    ]
)

assert all(
    checks.values()
)

qc_payload = {
    "schema_version": "1.0",
    "cell": "6C-4E0",
    "stage": "Stage 6C Step 4E",
    "notebook_filename": NOTEBOOK_FILENAME,
    "package_name": PACKAGE_NAME,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "checks_total": len(
        checks
    ),
    "checks_passed": int(
        sum(
            checks.values()
        )
    ),
    "checks_failed": int(
        len(
            checks
        )
        - sum(
            checks.values()
        )
    ),
    "checks": checks,
    "source": {
        "path": str(
            EVALUABLE_PATH
        ),
        "sha256": observed_evaluable_hash,
        "sidecar_path": str(
            EVALUABLE_SIDECAR
        ),
        "rows": EXPECTED_ROWS,
        "columns": EXPECTED_COLUMNS,
        "events": EXPECTED_EVENTS,
        "negatives": EXPECTED_NEGATIVES,
    },
    "prior_completed_category": {
        "manifest_path": str(
            PRIOR_MANIFEST_PATH
        ),
        "manifest_sha256": prior_manifest_hash,
        "category": (
            "remaining_comparator_inference"
        ),
        "verified": True,
    },
    "same_star_design": {
        "stratum_column": STAR_COLUMN,
        "strata": stratum_inventory.to_dict(
            orient="records"
        ),
        "models": {
            key: value
            for key, value in (
                MODELS.items()
            )
        },
        "bootstrap_method": (
            "ordinary_nonparametric_row_bootstrap_"
            "represented_by_exact_multinomial_row_multiplicities"
        ),
        "bootstrap_attempts_per_estimable_stratum": BOOTSTRAP_ATTEMPTS,
        "bootstrap_batch_size": BOOTSTRAP_BATCH_SIZE,
        "root_seed": ROOT_SEED,
        "identical_resamples_across_models": True,
        "grouped_weighted_metric_engine_validated_against_sklearn": True,
        "valid_replicates_by_star": valid_counts,
        "invalid_one_class_replicates_by_star": invalid_counts,
        "multiplicity_method": "Holm",
        "multiplicity_families": {
            "AUPRC": 6,
            "AUROC": 6,
        },
    },
    "historical_comparison": {
        "historical_values_are_rounded": True,
        "historical_row_level_replicate_artifact_available": False,
        "exact_historical_rng_stream_claimed": False,
        "model_conclusions_concordant": int(
            model_concordance[
                "scientific_conclusion_concordant"
            ].sum()
        ),
        "model_conclusions_total": int(
            len(
                model_concordance
            )
        ),
        "paired_conclusions_concordant": int(
            paired_concordance[
                "scientific_conclusion_concordant"
            ].sum()
        ),
        "paired_conclusions_total": int(
            len(
                paired_concordance
            )
        ),
    },
    "scientific_boundary": {
        "scores_modified": False,
        "outcomes_modified": False,
        "review_stars_modified": False,
        "models_modified": False,
        "features_modified": False,
        "comparator_formulas_modified": False,
        "weights_modified": False,
        "thresholds_modified": False,
        "linkage_decisions_modified": False,
        "row_order_modified": False,
        "cohort_membership_modified": False,
        "sparse_stratum_hidden": False,
        "historical_results_overwritten": False,
        "experiment_2_started": False,
    },
    "software_versions": {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "pyarrow": pyarrow.__version__,
        "scikit_learn": sklearn.__version__,
        "joblib": joblib.__version__,
    },
}

write_json(
    qc_payload,
    QC_PATH,
)

qc_sidecar = write_sidecar(
    QC_PATH
)


# --------------------------------------------------------------------------------------------------
# 27. FREEZE MANIFEST
# --------------------------------------------------------------------------------------------------

artifacts = [
    artifact_entry(
        STRATUM_INVENTORY_PATH,
        stratum_inventory_sidecar,
        "same-star stratum inventory",
    ),
    artifact_entry(
        POINT_ESTIMATE_PATH,
        point_estimate_sidecar,
        "same-star locked point estimates",
    ),
    artifact_entry(
        REPLICATE_PATH,
        replicate_sidecar,
        "same-star bootstrap replicate table",
    ),
    artifact_entry(
        MODEL_INTERVAL_PATH,
        model_interval_sidecar,
        "same-star model bootstrap intervals",
    ),
    artifact_entry(
        PAIRED_INFERENCE_PATH,
        paired_inference_sidecar,
        "same-star paired inference",
    ),
    artifact_entry(
        MULTIPLICITY_PATH,
        multiplicity_sidecar,
        "same-star Holm multiplicity table",
    ),
    artifact_entry(
        SPARSE_AUDIT_PATH,
        sparse_audit_sidecar,
        "same-star sparse and one-class audit",
    ),
    artifact_entry(
        HISTORICAL_MODEL_PATH,
        historical_model_sidecar,
        "historical reported same-star model intervals",
    ),
    artifact_entry(
        HISTORICAL_PAIRED_PATH,
        historical_paired_sidecar,
        "historical reported same-star paired inference",
    ),
    artifact_entry(
        CONCORDANCE_PATH,
        concordance_sidecar,
        "same-star historical versus reproduced concordance",
    ),
    artifact_entry(
        QC_PATH,
        qc_sidecar,
        "Cell 6C-4E0 QC",
    ),
]

manifest_payload = {
    "schema_version": "1.0",
    "cell": "6C-4E0",
    "stage": "Stage 6C Step 4E",
    "notebook_filename": NOTEBOOK_FILENAME,
    "package_name": PACKAGE_NAME,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "required_final_freeze_category": (
        "same_star_inference"
    ),
    "category_status": (
        "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
    ),
    "source_artifact": {
        "path": str(
            EVALUABLE_PATH
        ),
        "sha256": observed_evaluable_hash,
        "sidecar_path": str(
            EVALUABLE_SIDECAR
        ),
        "rows": EXPECTED_ROWS,
        "columns": EXPECTED_COLUMNS,
    },
    "analysis": {
        "star_column": STAR_COLUMN,
        "outcome_column": OUTCOME_COLUMN,
        "models": {
            key: {
                "display": value[
                    "display"
                ],
                "score_column": value[
                    "column"
                ],
            }
            for key, value in (
                MODELS.items()
            )
        },
        "zero_star_status": (
            "descriptive_only_one_outcome_class"
        ),
        "three_star_status": (
            "estimable_but_extremely_sparse"
        ),
        "bootstrap_attempts_per_estimable_stratum": BOOTSTRAP_ATTEMPTS,
        "bootstrap_method": (
            "exact_multinomial_representation_of_ordinary_row_bootstrap"
        ),
        "confidence_interval": (
            "2.5th_to_97.5th_percentile"
        ),
        "paired_comparisons": (
            "full_ges_minus_no_star_and_combined_metadata"
        ),
        "multiplicity_method": "Holm",
        "multiplicity_families": {
            "AUPRC": 6,
            "AUROC": 6,
        },
        "historical_results_preserved": True,
        "independent_results_preserved": True,
        "scientific_conclusions_concordant": {
            "model_intervals": {
                "concordant": int(
                    model_concordance[
                        "scientific_conclusion_concordant"
                    ].sum()
                ),
                "total": int(
                    len(
                        model_concordance
                    )
                ),
            },
            "paired_comparisons": {
                "concordant": int(
                    paired_concordance[
                        "scientific_conclusion_concordant"
                    ].sum()
                ),
                "total": int(
                    len(
                        paired_concordance
                    )
                ),
            },
        },
    },
    "artifacts": artifacts,
    "decision": (
        "PASS_STAGE6C_SAME_STAR_INFERENCE_RESULT_CATEGORY_"
        "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
    ),
    "completed_independent_result_categories": 3,
    "remaining_independent_result_categories": 5,
    "scientific_boundary": {
        "frozen_inputs_modified": False,
        "historical_results_overwritten": False,
        "sparse_stratum_overinterpreted": False,
        "final_integrated_stage6c_freeze_completed": False,
        "experiment_2_started": False,
    },
}

write_json(
    manifest_payload,
    MANIFEST_PATH,
)

manifest_sidecar = write_sidecar(
    MANIFEST_PATH
)


# --------------------------------------------------------------------------------------------------
# 28. FINAL COMPLETE REVERIFICATION
# --------------------------------------------------------------------------------------------------

fresh_qc = json.loads(
    QC_PATH.read_text(
        encoding="utf-8"
    )
)

fresh_manifest = json.loads(
    MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

assert (
    fresh_qc[
        "checks_failed"
    ]
    == 0
)

assert (
    fresh_manifest[
        "required_final_freeze_category"
    ]
    == "same_star_inference"
)

assert (
    fresh_manifest[
        "remaining_independent_result_categories"
    ]
    == 5
)

assert (
    read_sidecar_hash(
        qc_sidecar
    )
    == sha256_file(
        QC_PATH
    )
)

assert (
    read_sidecar_hash(
        manifest_sidecar
    )
    == sha256_file(
        MANIFEST_PATH
    )
)

for artifact_record in (
    fresh_manifest[
        "artifacts"
    ]
):
    artifact_path = Path(
        artifact_record[
            "path"
        ]
    )

    artifact_sidecar = Path(
        artifact_record[
            "sidecar_path"
        ]
    )

    assert artifact_path.exists()
    assert artifact_sidecar.exists()

    assert (
        sha256_file(
            artifact_path
        )
        == artifact_record[
            "sha256"
        ]
    )

    assert (
        read_sidecar_hash(
            artifact_sidecar
        )
        == artifact_record[
            "sha256"
        ]
    )


# --------------------------------------------------------------------------------------------------
# 29. DISPLAY FINAL RESULT
# --------------------------------------------------------------------------------------------------

separator = "=" * 158

print(
    "\n" + separator
)

print(
    "STAGE 6C STEP 4E — CELL 6C-4E0 — "
    "SAME-STAR INFERENCE RESULT CATEGORY"
)

print(
    separator
)

print(
    f"Stage 6B source hash                    : "
    f"PASS ({observed_evaluable_hash})"
)

print(
    f"Prior Cell 6C-4D0 manifest              : "
    f"PASS ({prior_manifest_hash})"
)

print(
    f"Same-star strata                        : "
    f"PASS (4/4)"
)

print(
    f"Historical point estimates              : "
    f"PASS (18/18 metric values)"
)

print(
    f"Bootstrap attempts                      : "
    f"{BOOTSTRAP_ATTEMPTS:,} per estimable stratum"
)

print(
    f"Star-1 valid / invalid                  : "
    f"{valid_counts[1]:,} / {invalid_counts[1]:,}"
)

print(
    f"Star-2 valid / invalid                  : "
    f"{valid_counts[2]:,} / {invalid_counts[2]:,}"
)

print(
    f"Star-3 valid / invalid                  : "
    f"{valid_counts[3]:,} / {invalid_counts[3]:,}"
)

print(
    f"Bootstrap elapsed                       : "
    f"{bootstrap_elapsed_seconds / 60:.2f} minutes"
)

print(
    f"Model-interval conclusions concordant   : "
    f"PASS ({int(model_concordance['scientific_conclusion_concordant'].sum())}/"
    f"{len(model_concordance)})"
)

print(
    f"Paired conclusions concordant           : "
    f"PASS ({int(paired_concordance['scientific_conclusion_concordant'].sum())}/"
    f"{len(paired_concordance)})"
)

print(
    f"Fresh QC                                : "
    f"PASS ({fresh_qc['checks_passed']}/"
    f"{fresh_qc['checks_total']})"
)

print(
    f"Stratum inventory                       : "
    f"{STRATUM_INVENTORY_PATH}"
)

print(
    f"Point estimates                         : "
    f"{POINT_ESTIMATE_PATH}"
)

print(
    f"Bootstrap replicates                    : "
    f"{REPLICATE_PATH}"
)

print(
    f"Model intervals                         : "
    f"{MODEL_INTERVAL_PATH}"
)

print(
    f"Paired inference                        : "
    f"{PAIRED_INFERENCE_PATH}"
)

print(
    f"Multiplicity table                      : "
    f"{MULTIPLICITY_PATH}"
)

print(
    f"Sparse-stratum audit                    : "
    f"{SPARSE_AUDIT_PATH}"
)

print(
    f"Concordance table                       : "
    f"{CONCORDANCE_PATH}"
)

print(
    f"QC                                      : "
    f"{QC_PATH}"
)

print(
    f"Manifest                                : "
    f"{MANIFEST_PATH}"
)

print(
    f"Manifest SHA-256                        : "
    f"{sha256_file(MANIFEST_PATH)}"
)


print(
    "\nSAME-STAR STRATUM INVENTORY"
)

print(
    stratum_inventory.to_string(
        index=False,
        float_format=lambda value: (
            f"{value:.12f}"
        ),
    )
)


print(
    "\nREPRODUCED SAME-STAR MODEL INTERVALS"
)

print(
    model_intervals[
        [
            "stars",
            "model",
            "metric",
            "point_estimate",
            "percentile_95_ci_low",
            "percentile_95_ci_high",
            "null_interval_conclusion",
            "valid_replicates",
            "invalid_one_class_replicates",
            "extremely_sparse",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: (
            f"{value:.12f}"
        ),
    )
)


print(
    "\nREPRODUCED SAME-STAR PAIRED INFERENCE"
)

print(
    paired_inference[
        [
            "stars",
            "metric",
            "comparator",
            "point_difference",
            "percentile_95_ci_low",
            "percentile_95_ci_high",
            "two_sided_bootstrap_sign_probability",
            "holm_adjusted_bootstrap_sign_probability",
            "holm_reject_at_0_05",
            "interval_conclusion",
            "valid_replicates",
            "invalid_one_class_replicates",
            "extremely_sparse",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: (
            f"{value:.12f}"
        ),
    )
)


print(
    "\nSPARSE-STRATUM AUDIT"
)

print(
    sparse_audit.to_string(
        index=False,
    )
)


print(
    "\nSCIENTIFIC-CONCLUSION CONCORDANCE"
)

print(
    paired_concordance[
        [
            "stars",
            "metric",
            "comparator",
            "historical_interval_conclusion",
            "interval_conclusion",
            "historical_holm_significant",
            "holm_reject_at_0_05",
            "scientific_conclusion_concordant",
        ]
    ].to_string(
        index=False,
    )
)


print(
    "\nCELL DECISION"
)

print(
    "PASS_STAGE6C_SAME_STAR_INFERENCE_RESULT_CATEGORY_"
    "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
)

print(
    "The third pending Stage 6C result category is complete. "
    "Five independent result categories remain. "
    "Experiment 2 has not started."
)

print(
    separator
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Use this Colab notebook file name: GES_Stage6C_Cell_6C_4E0_Same_Star_Inference_Materialization.ipynb

STAGE 6C STEP 4E — CELL 6C-4E0 — SAME-STAR INFERENCE RESULT CATEGORY
Stage 6B source hash                    : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038)
Prior Cell 6C-4D0 manifest              : PASS (d7082823437de952f0e4a96204f6c439015d74e26a9f6b14f57eeffd955e2cd6)
Same-star strata                        : PASS (4/4)
Historical point estimates              : PASS (18/18 metric values)
Bootstrap attempts                      : 2,000 per estimable stratum
Star-1 valid / invalid                  : 2,000 / 0
Star-2 valid / invalid                  : 2,000 / 0
Star-3 valid / invalid                  : 1,733 / 267
Bootstrap elapsed                       : 0.34 minutes
Model-interval conclusions concordant   : PASS (18/18)
Paired concl